In [ ]:
### Frequencia média DHW > 4 e > 8 per site + Mapas (versão com escala de cores corrigida)
### Períodos entre anos (Julho a Junho)
###GIF high

import os
import glob
import re
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import geopandas as gpd
from shapely.geometry import box, LineString
from shapely.ops import unary_union, polygonize
import matplotlib.lines as mlines
import matplotlib.cm as cm
from matplotlib.colors import LinearSegmentedColormap  # Import necessário para criar colormap personalizado

# ============================================================
# Configurações iniciais
# ============================================================
dhw_dir = r'H:\remote sensing\CRW_DHW_FULL'

lat_min, lat_max = -20.5, -14.5
lon_min, lon_max = -40.5, -35.5

#lat_min, lat_max = -18.7, -16.5 
#lon_min, lon_max = -36.9, -36.6

bathymetry_file = r'C:\Users\rbfra\OneDrive\GIS shapes\gebco_2024_ASO.nc'
output_dir = (r'C:\Users\rbfra\OneDrive\########PUBLICACOES\############Menezes et al. Mus his distribution and abundance Abrolhos\########NEW RESULTS\output_DHW_frequency_Jul-Jun_GIF') 
os.makedirs(output_dir, exist_ok=True)

protected_areas_shp = r'H:\remote sensing\Mascaras\uc_fed_agosto_2016_site_shp\uc_fed_agosto_2016_site.shp'
coast_shp = (r'C:\Users\rbfra\OneDrive\GIS shapes\batimetria_new'
             r'\LINHA_DE_COSTA_IMAGEM_GEOCOVER_SIRGAS_2000.shp')
islands_shp = (r'C:\Users\rbfra\OneDrive\GIS shapes\batimetria_new'
               r'\ILHAS_IMAGEM_GEOCOVER_SIRGAS_2000.shp')
reefs_shp = r'H:\remote sensing\Mascaras\Recifes_Banco_dos_Abrolhos\Recifes_Banco_dos_Abrolhos.shp'

# Anos permitidos (anos de início do período Jul-Jun)
allowed_years = set(range(1985, 2026))  # inclui de 1985 até 2008

# ============================================================
# Funções auxiliares
# ============================================================
def get_lat_slice(ds, lat_min, lat_max):
    lat_vals = ds['lat'].values
    return slice(lat_min, lat_max) if lat_vals[0] < lat_vals[-1] else slice(lat_max, lat_min)

def load_bathymetry(bathy_file, lat_min, lat_max, lon_min, lon_max):
    ds = xr.open_dataset(bathy_file)
    lat_slice = get_lat_slice(ds, lat_min, lat_max)
    bathy = ds['elevation'].sel(lon=slice(lon_min, lon_max), lat=lat_slice)
    mask = (bathy >= -200).astype(float)
    return bathy, mask

def adjust_longitudes(da):
    if (da.lon > 180).any():
        da = da.assign_coords(lon=(((da.lon + 180) % 360) - 180)).sortby('lon')
    return da

def extract_date_from_filename(fname):
    match = re.search(r'(\d{8})', os.path.basename(fname))
    return pd.to_datetime(match.group(1), format='%Y%m%d') if match else None

def load_shapefile(path, name="shp"):
    gdf = gpd.read_file(path)
    if gdf.crs != "EPSG:4326":
        gdf = gdf.to_crs("EPSG:4326")
    return gdf

def load_coast_and_islands(coast_file, islands_file, lat_min, lat_max, lon_min, lon_max):
    bbox = box(lon_min, lat_min, lon_max, lat_max)
    bbox_gdf = gpd.GeoDataFrame({'geometry': [bbox]}, crs="EPSG:4326")
    coast = gpd.read_file(coast_file).to_crs("EPSG:4326")
    islands = gpd.read_file(islands_file).to_crs("EPSG:4326")
    islands_clip = gpd.overlay(islands, bbox_gdf, how='intersection')
    coast_union = coast.geometry.unary_union
    bbox_line = LineString(list(bbox.exterior.coords))
    polygons = list(polygonize(unary_union([coast_union, bbox_line])))
    ocean_poly, max_area = None, 0
    for poly in polygons:
        inter = poly.intersection(bbox)
        if not inter.is_empty and inter.area > max_area:
            ocean_poly, max_area = inter, inter.area
    if ocean_poly is None:
        ocean_poly = bbox
    land_poly = bbox.difference(ocean_poly)
    if not islands_clip.empty:
        land_poly = unary_union([land_poly, islands_clip.unary_union])
    land = gpd.GeoDataFrame({'geometry': [land_poly]}, crs="EPSG:4326")
    return gpd.overlay(land, bbox_gdf, how='intersection')

# ============================================================
# Carregar dados estáticos (batimetria, shapefiles)
# ============================================================
bathy, marine_mask = load_bathymetry(bathymetry_file, lat_min, lat_max, lon_min, lon_max)
water_mask = (bathy < 0).astype(float)
marine_mask = ((bathy >= -200) & (bathy < 0)).astype(float)

gdf_protected = load_shapefile(protected_areas_shp, "Área Protegida")
gdf_coast = load_shapefile(coast_shp, "Linha de Costa")
gdf_islands = load_shapefile(islands_shp, "Ilhas")
gdf_recifes = load_shapefile(reefs_shp, "Recifes")
gdf_land = load_coast_and_islands(coast_shp, islands_shp, lat_min, lat_max, lon_min, lon_max)
gdf_islands_bound = gpd.read_file(islands_shp).to_crs("EPSG:4326")

# ============================================================
# Processamento dos arquivos DHW
# ============================================================
dhw_files = sorted(glob.glob(os.path.join(dhw_dir, '*.nc')))
print(f"Arquivos DHW encontrados: {len(dhw_files)}")

dhw_bool_4, dhw_bool_8 = [], []
for f in dhw_files:
    dt = extract_date_from_filename(f)
    # ### ALTERAÇÃO ###
    # A verificação de ano continua a mesma, pois estamos interessados nos dados desses anos,
    # a agregação será feita depois.
    if dt is None or dt.year not in range(min(allowed_years), max(allowed_years) + 2): # Carrega um ano a mais para completar o último período
        continue

    try:
        ds = xr.open_dataset(f)
    except Exception as e:
        print(f"[AVISO] Arquivo corrompido identificado: {f}. Erro: {e}.")
        print("Este arquivo será removido e não entrará nas análises.")
        os.remove(f)
        continue

    dhw_raw = ds['degree_heating_week']

    if 'time' in dhw_raw.dims:
        dhw_raw = dhw_raw.isel(time=0, drop=True)

    dhw = dhw_raw.sel(
        lat=get_lat_slice(ds, lat_min, lat_max),
        lon=slice(lon_min, lon_max)
    )
    dhw = adjust_longitudes(dhw)

    mask_interp = marine_mask.interp(lat=dhw.lat, lon=dhw.lon, method='nearest') >= 0.5
    dhw_masked = dhw.where(mask_interp)

    dhw_bool_4.append((dhw_masked > 4).astype(int).expand_dims(time=[dt]))
    dhw_bool_8.append((dhw_masked > 8).astype(int).expand_dims(time=[dt]))

# Empilhar e preencher faltantes
stack_4 = xr.concat(dhw_bool_4, dim='time').sortby('time').fillna(0)
stack_8 = xr.concat(dhw_bool_8, dim='time').sortby('time').fillna(0)

# ### ALTERAÇÃO ###
# Agrupar por "ano hidrológico" (Julho a Junho) usando resample.
# 'AS-JUL' significa "Annual Start, July"
print("Agrupando dados por períodos de Julho a Junho...")
hydro_yearly_sum_4 = stack_4.resample(time='AS-JUL').sum(dim='time')
hydro_yearly_sum_8 = stack_8.resample(time='AS-JUL').sum(dim='time')

# Contar o número de dias em cada período para um cálculo de percentual preciso
hydro_yearly_count = stack_4.resample(time='AS-JUL').count(dim='time')

hydro_yearly_percent_4 = (hydro_yearly_sum_4 / hydro_yearly_count) * 100
hydro_yearly_percent_8 = (hydro_yearly_sum_8 / hydro_yearly_count) * 100

# =========================================================================
# ### ETAPA DE CORREÇÃO: ENCONTRAR O VMAX GLOBAL PARA CONSISTÊNCIA ###
# =========================================================================
# Encontra o valor máximo absoluto de frequência em todos os anos para cada limiar.
global_vmax_4 = np.nanmax(hydro_yearly_percent_4.values)
global_vmax_8 = np.nanmax(hydro_yearly_percent_8.values)

# Arredonda o vmax para o próximo múltiplo de 5 para uma legenda mais limpa
vmax_plot_4 = np.ceil(global_vmax_4 / 5) * 5 if global_vmax_4 > 0 else 1
vmax_plot_8 = np.ceil(global_vmax_8 / 5) * 5 if global_vmax_8 > 0 else 1

print("\n--- Análise da Escala de Cores Global ---")
print(f"Frequência máxima absoluta para DHW > 4: {global_vmax_4:.2f}%")
print(f"Frequência máxima absoluta para DHW > 8: {global_vmax_8:.2f}%")
print(f"==> Usando VMAX = {vmax_plot_4} para todos os mapas de DHW > 4")
print(f"==> Usando VMAX = {vmax_plot_8} para todos os mapas de DHW > 8")
print("-----------------------------------------")


# ============================================================
# Função de plotagem MODIFICADA para aceitar VMAX
# ============================================================
def plot_dhw_map(dhw_da, threshold, year_label, out_dir, vmax=100): # Adiciona vmax como argumento
    fig, ax = plt.subplots(figsize=(8, 6), subplot_kw={'projection': ccrs.PlateCarree()})
    ax.set_extent([lon_min, lon_max, lat_min, lat_max])
    ax.set_facecolor('white')

    mask_interp = marine_mask.interp(lat=dhw_da.lat, lon=dhw_da.lon, method='nearest')
    dhw_masked = dhw_da.where(mask_interp >= 0.5)

    cmap = LinearSegmentedColormap.from_list('custom_warm', ['white', 'yellow', 'orange', 'red'])
    
    # Usa o vmax passado como argumento, garantindo que o valor mínimo seja pelo menos 1 para evitar erro
    effective_vmax = max(vmax, 1)

    img = ax.pcolormesh(
        dhw_da.lon,
        dhw_da.lat,
        dhw_masked,
        cmap=cmap,
        transform=ccrs.PlateCarree(),
        vmin=0,
        vmax=effective_vmax, # USA O VMAX AJUSTADO
        zorder=1
    )

    cbar = plt.colorbar(img, ax=ax, pad=0.05)
    cbar.set_label(f"Frequência de dias com DHW > {threshold} (%)")
    
    # ... (o resto da função de plotagem é igual)
    protected_marine = gpd.overlay(gdf_protected, gdf_land, how='difference')
    if not protected_marine.empty:
        protected_marine.boundary.plot(ax=ax, edgecolor='darkblue', linewidth=1.5, transform=ccrs.PlateCarree(), zorder=5)
    gdf_land.plot(ax=ax, facecolor='white', edgecolor='black', linewidth=1, transform=ccrs.PlateCarree(), zorder=10)
    ax.add_geometries(gdf_coast.geometry, crs=ccrs.PlateCarree(), facecolor='none', edgecolor='black', linewidth=1.5, zorder=15)
    bathy_ocean = bathy.where(bathy < 0)
    ax.contour(bathy_ocean.lon, bathy_ocean.lat, bathy_ocean, levels=[-200], colors='black', linewidths=1, transform=ccrs.PlateCarree(), zorder=12)
    gdf_islands_bound.boundary.plot(ax=ax, edgecolor='black', linewidth=1, transform=ccrs.PlateCarree(), zorder=25)
    gdf_recifes.boundary.plot(ax=ax, edgecolor='purple', linewidth=0.5, transform=ccrs.PlateCarree(), zorder=30)
    protected_handle = mlines.Line2D([], [], color='darkblue', linewidth=1.5, label='Área Protegida (marinha)')
    reef_handle = mlines.Line2D([], [], color='purple', linewidth=0.5, label='Recifes')
    ax.legend(handles=[protected_handle, reef_handle], loc='upper right', fontsize=8)
    ax.set_title(f"Frequência de dias com DHW > {threshold} ({year_label})")
    fname = os.path.join(out_dir, f"dhw_frequency_gt{threshold}_{year_label}.png")
    plt.savefig(fname, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Mapa salvo: {fname}")

# ============================================================
# Geração dos mapas anuais (período Jul-Jun) COM ESCALA CONSISTENTE
# ============================================================
print("\nGerando mapas para os períodos de Julho a Junho...")

for t in hydro_yearly_percent_4.time.values:
    start_year = pd.to_datetime(t).year
    if start_year not in allowed_years:
        continue
    end_year = start_year + 1
    year_label = f"{start_year}-{end_year}"
    
    data_slice_4 = hydro_yearly_percent_4.sel(time=t)
    data_slice_8 = hydro_yearly_percent_8.sel(time=t)
    
    # Chama a função de plotagem passando o VMAX GLOBAL calculado
    plot_dhw_map(data_slice_4, 4, year_label, output_dir, vmax=vmax_plot_4)
    plot_dhw_map(data_slice_8, 8, year_label, output_dir, vmax=vmax_plot_8)

# ============================================================
# (Opcional) Mapas agregados para todo o período
# ============================================================
start_period = pd.to_datetime(hydro_yearly_percent_4.time.values[0]).year
end_period = pd.to_datetime(hydro_yearly_percent_4.time.values[-1]).year + 1
total_period_label = f"{start_period}-{end_period}"

mean_freq_4 = hydro_yearly_percent_4.mean(dim='time')
mean_freq_8 = hydro_yearly_percent_8.mean(dim='time')

# Para o mapa de média, também usamos a escala global para consistência
plot_dhw_map(mean_freq_4, 4, f"Média {total_period_label}", output_dir, vmax=vmax_plot_4)
plot_dhw_map(mean_freq_8, 8, f"Média {total_period_label}", output_dir, vmax=vmax_plot_8)

print(f"\nProcessamento concluído com sucesso – mapas para períodos de {total_period_label} gerados.")


# ============================================================
# ### Geração dos GIFs animados
# ============================================================
# Esta seção requer a biblioteca Pillow. Se não a tiver, instale com: pip install Pillow
from PIL import Image
import glob

def create_dhw_gif(output_directory, threshold, year_range, duration_per_frame_ms=500):
    """
    Encontra todos os mapas PNG para um limiar de DHW, os ordena e cria um GIF animado.

    Args:
        output_directory (str): Pasta onde os arquivos PNG estão e onde o GIF será salvo.
        threshold (int): O limiar de DHW (4 ou 8) para procurar nos nomes dos arquivos.
        year_range (set): O conjunto de anos permitidos para garantir a ordem correta.
        duration_per_frame_ms (int): Duração de cada quadro no GIF, em milissegundos.
    """
    print(f"\n--- Criando GIF para DHW > {threshold} ---")
    
    # 1. Encontrar todos os arquivos de imagem para o limiar especificado
    search_pattern = os.path.join(output_directory, f"dhw_frequency_gt{threshold}_*.png")
    image_files = sorted(glob.glob(search_pattern))

    # Filtra para garantir que apenas os anos processados sejam incluídos, em ordem
    # Isso é uma segurança extra para garantir a ordem cronológica correta
    valid_files_for_gif = []
    for year in sorted(list(year_range)):
        expected_filename = os.path.join(output_directory, f"dhw_frequency_gt{threshold}_{year}-{year+1}.png")
        if expected_filename in image_files:
            valid_files_for_gif.append(expected_filename)

    if not valid_files_for_gif:
        print(f"Nenhuma imagem encontrada para o limiar {threshold}. GIF não será criado.")
        return

    print(f"Encontradas {len(valid_files_for_gif)} imagens para a animação.")

    # 2. Ler as imagens e armazená-las como frames
    frames = []
    for filename in valid_files_for_gif:
        try:
            frames.append(Image.open(filename))
        except Exception as e:
            print(f"  [Aviso] Não foi possível abrir o arquivo {filename}. Pulando. Erro: {e}")

    if len(frames) < 2:
        print("São necessárias pelo menos 2 imagens para criar um GIF. Abortando.")
        return

    # 3. Salvar os frames como um GIF animado
    # O primeiro frame é usado para iniciar o arquivo, e o resto é anexado.
    first_frame = frames[0]
    output_gif_path = os.path.join(output_directory, f"animacao_dhw_gt{threshold}_{min(year_range)}-{max(year_range)+1}.gif")
    
    first_frame.save(
        output_gif_path,
        format='GIF',
        append_images=frames[1:],  # Anexa os frames restantes
        save_all=True,
        duration=duration_per_frame_ms,  # Duração de cada frame em ms (500ms = 0.5s)
        loop=0  # 0 significa que o GIF irá repetir infinitamente
    )
    
    print(f">>> GIF salvo com sucesso em: {output_gif_path}")

# --- Chamadas da função para criar os GIFs ---
# Lembre-se de passar o mesmo 'allowed_years' que você usou na análise
create_dhw_gif(output_dir, 4, allowed_years, duration_per_frame_ms=500)
create_dhw_gif(output_dir, 8, allowed_years, duration_per_frame_ms=500)

In [ ]:
### Frequencia média DHW > 4 e > 8 per site + Mapas (versão com escala de cores corrigida)
### Períodos entre anos (Julho a Junho)
###GIF low

import os
import glob
import re
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import geopandas as gpd
from shapely.geometry import box, LineString
from shapely.ops import unary_union, polygonize
import matplotlib.lines as mlines
import matplotlib.cm as cm
from matplotlib.colors import LinearSegmentedColormap  # Import necessário para criar colormap personalizado

# ============================================================
# Configurações iniciais
# ============================================================
dhw_dir = r'D:\remote sensing\CRW_DHW_FULL'

lat_min, lat_max = -20.5, -14.5
lon_min, lon_max = -40.5, -35.5

#lat_min, lat_max = -18.7, -16.5 
#lon_min, lon_max = -36.9, -36.6

bathymetry_file = r'C:\Users\rbfra\OneDrive\GIS shapes\gebco_2024_ASO.nc'
output_dir = (r'C:\Users\rbfra\OneDrive\########PUBLICACOES\############Menezes et al. Mus his distribution and abundance Abrolhos\########NEW RESULTS\output_DHW_frequency_Jul-Jun_GIF_low') 
os.makedirs(output_dir, exist_ok=True)

protected_areas_shp = r'D:\remote sensing\Mascaras\uc_fed_agosto_2016_site_shp\uc_fed_agosto_2016_site.shp'
coast_shp = (r'C:\Users\rbfra\OneDrive\GIS shapes\batimetria_new'
             r'\LINHA_DE_COSTA_IMAGEM_GEOCOVER_SIRGAS_2000.shp')
islands_shp = (r'C:\Users\rbfra\OneDrive\GIS shapes\batimetria_new'
               r'\ILHAS_IMAGEM_GEOCOVER_SIRGAS_2000.shp')
reefs_shp = r'D:\remote sensing\Mascaras\Recifes_Banco_dos_Abrolhos\Recifes_Banco_dos_Abrolhos.shp'

# Anos permitidos (anos de início do período Jul-Jun)
allowed_years = set(range(1985, 2026))  # inclui de 1985 até 2008

# ============================================================
# Funções auxiliares
# ============================================================
def get_lat_slice(ds, lat_min, lat_max):
    lat_vals = ds['lat'].values
    return slice(lat_min, lat_max) if lat_vals[0] < lat_vals[-1] else slice(lat_max, lat_min)

def load_bathymetry(bathy_file, lat_min, lat_max, lon_min, lon_max):
    ds = xr.open_dataset(bathy_file)
    lat_slice = get_lat_slice(ds, lat_min, lat_max)
    bathy = ds['elevation'].sel(lon=slice(lon_min, lon_max), lat=lat_slice)
    mask = (bathy >= -200).astype(float)
    return bathy, mask

def adjust_longitudes(da):
    if (da.lon > 180).any():
        da = da.assign_coords(lon=(((da.lon + 180) % 360) - 180)).sortby('lon')
    return da

def extract_date_from_filename(fname):
    match = re.search(r'(\d{8})', os.path.basename(fname))
    return pd.to_datetime(match.group(1), format='%Y%m%d') if match else None

def load_shapefile(path, name="shp"):
    gdf = gpd.read_file(path)
    if gdf.crs != "EPSG:4326":
        gdf = gdf.to_crs("EPSG:4326")
    return gdf

def load_coast_and_islands(coast_file, islands_file, lat_min, lat_max, lon_min, lon_max):
    bbox = box(lon_min, lat_min, lon_max, lat_max)
    bbox_gdf = gpd.GeoDataFrame({'geometry': [bbox]}, crs="EPSG:4326")
    coast = gpd.read_file(coast_file).to_crs("EPSG:4326")
    islands = gpd.read_file(islands_file).to_crs("EPSG:4326")
    islands_clip = gpd.overlay(islands, bbox_gdf, how='intersection')
    coast_union = coast.geometry.unary_union
    bbox_line = LineString(list(bbox.exterior.coords))
    polygons = list(polygonize(unary_union([coast_union, bbox_line])))
    ocean_poly, max_area = None, 0
    for poly in polygons:
        inter = poly.intersection(bbox)
        if not inter.is_empty and inter.area > max_area:
            ocean_poly, max_area = inter, inter.area
    if ocean_poly is None:
        ocean_poly = bbox
    land_poly = bbox.difference(ocean_poly)
    if not islands_clip.empty:
        land_poly = unary_union([land_poly, islands_clip.unary_union])
    land = gpd.GeoDataFrame({'geometry': [land_poly]}, crs="EPSG:4326")
    return gpd.overlay(land, bbox_gdf, how='intersection')

# ============================================================
# Carregar dados estáticos (batimetria, shapefiles)
# ============================================================
bathy, marine_mask = load_bathymetry(bathymetry_file, lat_min, lat_max, lon_min, lon_max)
water_mask = (bathy < 0).astype(float)
marine_mask = ((bathy >= -200) & (bathy < 0)).astype(float)

gdf_protected = load_shapefile(protected_areas_shp, "Área Protegida")
gdf_coast = load_shapefile(coast_shp, "Linha de Costa")
gdf_islands = load_shapefile(islands_shp, "Ilhas")
gdf_recifes = load_shapefile(reefs_shp, "Recifes")
gdf_land = load_coast_and_islands(coast_shp, islands_shp, lat_min, lat_max, lon_min, lon_max)
gdf_islands_bound = gpd.read_file(islands_shp).to_crs("EPSG:4326")

# ============================================================
# Processamento dos arquivos DHW
# ============================================================
dhw_files = sorted(glob.glob(os.path.join(dhw_dir, '*.nc')))
print(f"Arquivos DHW encontrados: {len(dhw_files)}")

dhw_bool_4, dhw_bool_8 = [], []
for f in dhw_files:
    dt = extract_date_from_filename(f)
    # ### ALTERAÇÃO ###
    # A verificação de ano continua a mesma, pois estamos interessados nos dados desses anos,
    # a agregação será feita depois.
    if dt is None or dt.year not in range(min(allowed_years), max(allowed_years) + 2): # Carrega um ano a mais para completar o último período
        continue

    try:
        ds = xr.open_dataset(f)
    except Exception as e:
        print(f"[AVISO] Arquivo corrompido identificado: {f}. Erro: {e}.")
        print("Este arquivo será removido e não entrará nas análises.")
        os.remove(f)
        continue

    dhw_raw = ds['degree_heating_week']

    if 'time' in dhw_raw.dims:
        dhw_raw = dhw_raw.isel(time=0, drop=True)

    dhw = dhw_raw.sel(
        lat=get_lat_slice(ds, lat_min, lat_max),
        lon=slice(lon_min, lon_max)
    )
    dhw = adjust_longitudes(dhw)

    mask_interp = marine_mask.interp(lat=dhw.lat, lon=dhw.lon, method='nearest') >= 0.5
    dhw_masked = dhw.where(mask_interp)

    dhw_bool_4.append((dhw_masked > 4).astype(int).expand_dims(time=[dt]))
    dhw_bool_8.append((dhw_masked > 8).astype(int).expand_dims(time=[dt]))

# Empilhar e preencher faltantes
stack_4 = xr.concat(dhw_bool_4, dim='time').sortby('time').fillna(0)
stack_8 = xr.concat(dhw_bool_8, dim='time').sortby('time').fillna(0)

# ### ALTERAÇÃO ###
# Agrupar por "ano hidrológico" (Julho a Junho) usando resample.
# 'AS-JUL' significa "Annual Start, July"
print("Agrupando dados por períodos de Julho a Junho...")
hydro_yearly_sum_4 = stack_4.resample(time='AS-JUL').sum(dim='time')
hydro_yearly_sum_8 = stack_8.resample(time='AS-JUL').sum(dim='time')

# Contar o número de dias em cada período para um cálculo de percentual preciso
hydro_yearly_count = stack_4.resample(time='AS-JUL').count(dim='time')

hydro_yearly_percent_4 = (hydro_yearly_sum_4 / hydro_yearly_count) * 100
hydro_yearly_percent_8 = (hydro_yearly_sum_8 / hydro_yearly_count) * 100

# =========================================================================
# ### ETAPA DE CORREÇÃO: ENCONTRAR O VMAX GLOBAL PARA CONSISTÊNCIA ###
# =========================================================================
# Encontra o valor máximo absoluto de frequência em todos os anos para cada limiar.
global_vmax_4 = np.nanmax(hydro_yearly_percent_4.values)
global_vmax_8 = np.nanmax(hydro_yearly_percent_8.values)

# Arredonda o vmax para o próximo múltiplo de 5 para uma legenda mais limpa
vmax_plot_4 = np.ceil(global_vmax_4 / 5) * 5 if global_vmax_4 > 0 else 1
vmax_plot_8 = np.ceil(global_vmax_8 / 5) * 5 if global_vmax_8 > 0 else 1

print("\n--- Análise da Escala de Cores Global ---")
print(f"Frequência máxima absoluta para DHW > 4: {global_vmax_4:.2f}%")
print(f"Frequência máxima absoluta para DHW > 8: {global_vmax_8:.2f}%")
print(f"==> Usando VMAX = {vmax_plot_4} para todos os mapas de DHW > 4")
print(f"==> Usando VMAX = {vmax_plot_8} para todos os mapas de DHW > 8")
print("-----------------------------------------")


# ============================================================
# Função de plotagem MODIFICADA para aceitar VMAX
# ============================================================
def plot_dhw_map(dhw_da, threshold, year_label, out_dir, vmax=100): # Adiciona vmax como argumento
    fig, ax = plt.subplots(figsize=(8, 6), subplot_kw={'projection': ccrs.PlateCarree()})
    ax.set_extent([lon_min, lon_max, lat_min, lat_max])
    ax.set_facecolor('white')

    mask_interp = marine_mask.interp(lat=dhw_da.lat, lon=dhw_da.lon, method='nearest')
    dhw_masked = dhw_da.where(mask_interp >= 0.5)

    cmap = LinearSegmentedColormap.from_list('custom_warm', ['white', 'yellow', 'orange', 'red'])
    
    # Usa o vmax passado como argumento, garantindo que o valor mínimo seja pelo menos 1 para evitar erro
    effective_vmax = max(vmax, 1)

    img = ax.pcolormesh(
        dhw_da.lon,
        dhw_da.lat,
        dhw_masked,
        cmap=cmap,
        transform=ccrs.PlateCarree(),
        vmin=0,
        vmax=effective_vmax, # USA O VMAX AJUSTADO
        zorder=1
    )

    cbar = plt.colorbar(img, ax=ax, pad=0.05)
    cbar.set_label(f"Frequência de dias com DHW > {threshold} (%)")
    
    # ... (o resto da função de plotagem é igual)
    protected_marine = gpd.overlay(gdf_protected, gdf_land, how='difference')
    if not protected_marine.empty:
        protected_marine.boundary.plot(ax=ax, edgecolor='darkblue', linewidth=1.5, transform=ccrs.PlateCarree(), zorder=5)
    gdf_land.plot(ax=ax, facecolor='white', edgecolor='black', linewidth=1, transform=ccrs.PlateCarree(), zorder=10)
    ax.add_geometries(gdf_coast.geometry, crs=ccrs.PlateCarree(), facecolor='none', edgecolor='black', linewidth=1.5, zorder=15)
    bathy_ocean = bathy.where(bathy < 0)
    ax.contour(bathy_ocean.lon, bathy_ocean.lat, bathy_ocean, levels=[-200], colors='black', linewidths=1, transform=ccrs.PlateCarree(), zorder=12)
    gdf_islands_bound.boundary.plot(ax=ax, edgecolor='black', linewidth=1, transform=ccrs.PlateCarree(), zorder=25)
    gdf_recifes.boundary.plot(ax=ax, edgecolor='purple', linewidth=0.5, transform=ccrs.PlateCarree(), zorder=30)
    protected_handle = mlines.Line2D([], [], color='darkblue', linewidth=1.5, label='Área Protegida (marinha)')
    reef_handle = mlines.Line2D([], [], color='purple', linewidth=0.5, label='Recifes')
    ax.legend(handles=[protected_handle, reef_handle], loc='upper right', fontsize=8)
    ax.set_title(f"Frequência de dias com DHW > {threshold} ({year_label})")
    fname = os.path.join(out_dir, f"dhw_frequency_gt{threshold}_{year_label}.png")
    plt.savefig(fname, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Mapa salvo: {fname}")

# ============================================================
# Geração dos mapas anuais (período Jul-Jun) COM ESCALA CONSISTENTE
# ============================================================
print("\nGerando mapas para os períodos de Julho a Junho...")

for t in hydro_yearly_percent_4.time.values:
    start_year = pd.to_datetime(t).year
    if start_year not in allowed_years:
        continue
    end_year = start_year + 1
    year_label = f"{start_year}-{end_year}"
    
    data_slice_4 = hydro_yearly_percent_4.sel(time=t)
    data_slice_8 = hydro_yearly_percent_8.sel(time=t)
    
    # Chama a função de plotagem passando o VMAX GLOBAL calculado
    plot_dhw_map(data_slice_4, 4, year_label, output_dir, vmax=vmax_plot_4)
    plot_dhw_map(data_slice_8, 8, year_label, output_dir, vmax=vmax_plot_8)

# ============================================================
# (Opcional) Mapas agregados para todo o período
# ============================================================
start_period = pd.to_datetime(hydro_yearly_percent_4.time.values[0]).year
end_period = pd.to_datetime(hydro_yearly_percent_4.time.values[-1]).year + 1
total_period_label = f"{start_period}-{end_period}"

mean_freq_4 = hydro_yearly_percent_4.mean(dim='time')
mean_freq_8 = hydro_yearly_percent_8.mean(dim='time')

# Para o mapa de média, também usamos a escala global para consistência
plot_dhw_map(mean_freq_4, 4, f"Média {total_period_label}", output_dir, vmax=vmax_plot_4)
plot_dhw_map(mean_freq_8, 8, f"Média {total_period_label}", output_dir, vmax=vmax_plot_8)

print(f"\nProcessamento concluído com sucesso – mapas para períodos de {total_period_label} gerados.")


# ============================================================
# ### Geração dos GIFs animados (VERSÃO OTIMIZADA E CORRIGIDA)
# ============================================================
# Esta seção requer a biblioteca Pillow. Se não a tiver, instale com: pip install Pillow
from PIL import Image
import glob

def create_dhw_gif_optimized(output_directory, threshold, year_range, duration_per_frame_ms=500, max_width=800):
    """
    Cria um GIF animado OTIMIZADO. CORRIGIDO para lidar com imagens com transparência (RGBA).
    """
    print(f"\n--- Criando GIF OTIMIZADO para DHW > {threshold} ---")
    
    # 1. Encontrar e ordenar os arquivos de imagem
    search_pattern = os.path.join(output_directory, f"dhw_frequency_gt{threshold}_*.png")
    image_files = []
    for year in sorted(list(year_range)):
        expected_filename = os.path.join(output_directory, f"dhw_frequency_gt{threshold}_{year}-{year+1}.png")
        if os.path.exists(expected_filename):
            image_files.append(expected_filename)

    if len(image_files) < 2:
        print(f"São necessárias pelo menos 2 imagens para o limiar {threshold}. GIF não será criado.")
        return

    print(f"Encontradas {len(image_files)} imagens. Processando e otimizando...")

    # 2. Processar e otimizar cada quadro
    frames = []
    for filename in image_files:
        try:
            with Image.open(filename) as img:
                # ==========================================================
                # ### INÍCIO DA CORREÇÃO ###
                # Se a imagem tiver transparência (RGBA), converta para RGB
                # "achatando-a" sobre um fundo branco.
                # ==========================================================
                if img.mode == 'RGBA':
                    # Cria um fundo branco no modo RGB
                    background = Image.new('RGB', img.size, (255, 255, 255))
                    # Cola a imagem RGBA sobre o fundo, usando o canal alfa como máscara
                    background.paste(img, mask=img.split()[3])
                    img_to_process = background
                else:
                    # Se já for RGB, usa a imagem diretamente
                    img_to_process = img.convert('RGB')
                # ==========================================================
                # ### FIM DA CORREÇÃO ###
                # ==========================================================

                # O resto do processo de otimização continua igual, mas usando a imagem corrigida
                width, height = img_to_process.size
                new_height = int(max_width * (height / width))
                img_resized = img_to_process.resize((max_width, new_height), Image.Resampling.LANCZOS)
                
                # Agora a quantização funcionará, pois a imagem é RGB
                img_quantized = img_resized.quantize(colors=256, method=Image.Quantize.MEDIANCUT)
                
                frames.append(img_quantized)
        except Exception as e:
            print(f"  [Aviso] Não foi possível processar o arquivo {filename}. Pulando. Erro: {e}")

    if not frames:
        print("Nenhum quadro pôde ser processado. Abortando.")
        return

    # 3. Salvar os frames como um GIF animado com otimização
    first_frame = frames[0]
    output_gif_path = os.path.join(output_directory, f"animacao_dhw_gt{threshold}_{min(year_range)}-{max(year_range)+1}_optimized.gif")
    
    first_frame.save(
        output_gif_path,
        format='GIF',
        append_images=frames[1:],
        save_all=True,
        duration=duration_per_frame_ms,
        loop=0,
        optimize=True,
        disposal=2
    )
    
    file_size_mb = os.path.getsize(output_gif_path) / (1024 * 1024)
    print(f">>> GIF otimizado salvo com sucesso em: {output_gif_path}")
    print(f"    Tamanho final do arquivo: {file_size_mb:.2f} MB")


# --- Chamadas da função para criar os GIFs OTIMIZADOS ---
# Use esta nova função no seu script
create_dhw_gif_optimized(output_dir, 4, allowed_years, duration_per_frame_ms=280, max_width=600)
create_dhw_gif_optimized(output_dir, 8, allowed_years, duration_per_frame_ms=280, max_width=600)

In [ ]:
### Frequencia média DHW > 4 e > 8 per site + Mapas 
### Anual (JAN-DEZ APROXIMADO 365 dias)
 
import os
import glob
import re
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import geopandas as gpd
from shapely.geometry import box, LineString
from shapely.ops import unary_union, polygonize
import matplotlib.lines as mlines
import matplotlib.cm as cm
from matplotlib.colors import LinearSegmentedColormap  # Import necessário para criar colormap personalizado

# ============================================================
# Configurações iniciais
# ============================================================
dhw_dir = r'H:\remote sensing\CRW_DHW_FULL'

lat_min, lat_max = -20.5, -14.5
lon_min, lon_max = -40.5, -35.5

#lat_min, lat_max = -18.7, -16.5 
#lon_min, lon_max = -36.9, -36.6

bathymetry_file = r'C:\Users\rbfra\OneDrive\GIS shapes\gebco_2024_ASO.nc'
output_dir = (r'C:\Users\rbfra\OneDrive\########PUBLICACOES\############Menezes et al. Mus his distribution and abundance Abrolhos\########NEW RESULTS\output_DHW_frequency_yearly_V1') 
os.makedirs(output_dir, exist_ok=True)

protected_areas_shp = r'H:\remote sensing\Mascaras\uc_fed_agosto_2016_site_shp\uc_fed_agosto_2016_site.shp'
coast_shp = (r'C:\Users\rbfra\OneDrive\GIS shapes\batimetria_new'
             r'\LINHA_DE_COSTA_IMAGEM_GEOCOVER_SIRGAS_2000.shp')
islands_shp = (r'C:\Users\rbfra\OneDrive\GIS shapes\batimetria_new'
               r'\ILHAS_IMAGEM_GEOCOVER_SIRGAS_2000.shp')
reefs_shp = r'H:\remote sensing\Mascaras\Recifes_Banco_dos_Abrolhos\Recifes_Banco_dos_Abrolhos.shp'

# Anos permitidos
allowed_years = set(range(1985, 2009))  # inclui de 1985 até 2008

# ============================================================
# Funções auxiliares
# ============================================================
def get_lat_slice(ds, lat_min, lat_max):
    lat_vals = ds['lat'].values
    return slice(lat_min, lat_max) if lat_vals[0] < lat_vals[-1] else slice(lat_max, lat_min)

def load_bathymetry(bathy_file, lat_min, lat_max, lon_min, lon_max):
    ds = xr.open_dataset(bathy_file)
    lat_slice = get_lat_slice(ds, lat_min, lat_max)
    bathy = ds['elevation'].sel(lon=slice(lon_min, lon_max), lat=lat_slice)
    mask = (bathy >= -200).astype(float)
    return bathy, mask

def adjust_longitudes(da):
    if (da.lon > 180).any():
        da = da.assign_coords(lon=(((da.lon + 180) % 360) - 180)).sortby('lon')
    return da

def extract_date_from_filename(fname):
    match = re.search(r'(\d{8})', os.path.basename(fname))
    return pd.to_datetime(match.group(1), format='%Y%m%d') if match else None

def load_shapefile(path, name="shp"):
    gdf = gpd.read_file(path)
    if gdf.crs != "EPSG:4326":
        gdf = gdf.to_crs("EPSG:4326")
    return gdf

def load_coast_and_islands(coast_file, islands_file, lat_min, lat_max, lon_min, lon_max):
    bbox = box(lon_min, lat_min, lon_max, lat_max)
    bbox_gdf = gpd.GeoDataFrame({'geometry': [bbox]}, crs="EPSG:4326")
    coast = gpd.read_file(coast_file).to_crs("EPSG:4326")
    islands = gpd.read_file(islands_file).to_crs("EPSG:4326")
    islands_clip = gpd.overlay(islands, bbox_gdf, how='intersection')
    coast_union = coast.geometry.unary_union
    bbox_line = LineString(list(bbox.exterior.coords))
    polygons = list(polygonize(unary_union([coast_union, bbox_line])))
    ocean_poly, max_area = None, 0
    for poly in polygons:
        inter = poly.intersection(bbox)
        if not inter.is_empty and inter.area > max_area:
            ocean_poly, max_area = inter, inter.area
    if ocean_poly is None:
        ocean_poly = bbox
    land_poly = bbox.difference(ocean_poly)
    if not islands_clip.empty:
        land_poly = unary_union([land_poly, islands_clip.unary_union])
    land = gpd.GeoDataFrame({'geometry': [land_poly]}, crs="EPSG:4326")
    return gpd.overlay(land, bbox_gdf, how='intersection')

# ============================================================
# Carregar dados estáticos (batimetria, shapefiles)
# ============================================================
bathy, marine_mask = load_bathymetry(bathymetry_file, lat_min, lat_max, lon_min, lon_max)
water_mask = (bathy < 0).astype(float)
marine_mask = ((bathy >= -200) & (bathy < 0)).astype(float)

gdf_protected = load_shapefile(protected_areas_shp, "Área Protegida")
gdf_coast = load_shapefile(coast_shp, "Linha de Costa")
gdf_islands = load_shapefile(islands_shp, "Ilhas")
gdf_recifes = load_shapefile(reefs_shp, "Recifes")
gdf_land = load_coast_and_islands(coast_shp, islands_shp, lat_min, lat_max, lon_min, lon_max)
gdf_islands_bound = gpd.read_file(islands_shp).to_crs("EPSG:4326")

# ============================================================
# Processamento dos arquivos DHW
# ============================================================
dhw_files = sorted(glob.glob(os.path.join(dhw_dir, '*.nc')))
print(f"Arquivos DHW encontrados: {len(dhw_files)}")

dhw_bool_4, dhw_bool_8 = [], []
for f in dhw_files:
    dt = extract_date_from_filename(f)
    if dt is None or dt.year not in allowed_years:
        continue

    try:
        ds = xr.open_dataset(f)
    except Exception as e:
        print(f"[AVISO] Arquivo corrompido identificado: {f}. Erro: {e}.")
        print("Este arquivo será removido e não entrará nas análises.")
        os.remove(f)
        continue

    dhw_raw = ds['degree_heating_week']

    # Remover dimensão 'time' se já existir
    if 'time' in dhw_raw.dims:
        dhw_raw = dhw_raw.isel(time=0, drop=True)

    dhw = dhw_raw.sel(
        lat=get_lat_slice(ds, lat_min, lat_max),
        lon=slice(lon_min, lon_max)
    )
    dhw = adjust_longitudes(dhw)

    mask_interp = marine_mask.interp(lat=dhw.lat, lon=dhw.lon, method='nearest') >= 0.5
    dhw_masked = dhw.where(mask_interp)

    dhw_bool_4.append((dhw_masked > 4).astype(int).expand_dims(time=[dt]))
    dhw_bool_8.append((dhw_masked > 8).astype(int).expand_dims(time=[dt]))

# Empilhar e preencher faltantes
stack_4 = xr.concat(dhw_bool_4, dim='time').sortby('time').fillna(0)
stack_8 = xr.concat(dhw_bool_8, dim='time').sortby('time').fillna(0)

# Somar por ano
yearly_4 = stack_4.groupby('time.year').sum(dim='time')
yearly_8 = stack_8.groupby('time.year').sum(dim='time')

# Converter para percentual (assumindo 365 dias/ano para simplicidade)
yearly_4 = (yearly_4 / 365) * 100
yearly_8 = (yearly_8 / 365) * 100

# ============================================================
# Função de plotagem corrigida
# ============================================================
def plot_dhw_map(dhw_da, threshold, year, out_dir):
    fig, ax = plt.subplots(figsize=(8, 6), subplot_kw={'projection': ccrs.PlateCarree()})
    ax.set_extent([lon_min, lon_max, lat_min, lat_max])
    ax.set_facecolor('white')

    # Aplicar máscara batimétrica
    mask_interp = marine_mask.interp(lat=dhw_da.lat, lon=dhw_da.lon, method='nearest')
    dhw_masked = dhw_da.where(mask_interp >= 0.5)

    # Criar colormap personalizado: de branco para cores quentes
    cmap = LinearSegmentedColormap.from_list('custom_warm', ['white', 'yellow', 'orange', 'red'])
    img = ax.pcolormesh(
        dhw_da.lon,
        dhw_da.lat,
        dhw_masked,
        cmap=cmap,
        transform=ccrs.PlateCarree(),
        vmin=0,
        vmax=100,
        zorder=1
    )

    cbar = plt.colorbar(img, ax=ax, pad=0.05)
    cbar.set_label(f"Frequência de dias com DHW > {threshold} (%)")

    # Sobreposições
    protected_marine = gpd.overlay(gdf_protected, gdf_land, how='difference')
    if not protected_marine.empty:
        protected_marine.boundary.plot(
            ax=ax,
            edgecolor='darkblue',
            linewidth=1.5,
            transform=ccrs.PlateCarree(),
            zorder=5
        )

    gdf_land.plot(
        ax=ax,
        facecolor='white',
        edgecolor='black',
        linewidth=1,
        transform=ccrs.PlateCarree(),
        zorder=10
    )

    ax.add_geometries(
        gdf_coast.geometry,
        crs=ccrs.PlateCarree(),
        facecolor='none',
        edgecolor='black',
        linewidth=1.5,
        zorder=15
    )

    # Linha de -200m
    bathy_ocean = bathy.where(bathy < 0)
    ax.contour(
        bathy_ocean.lon,
        bathy_ocean.lat,
        bathy_ocean,
        levels=[-200],
        colors='black',
        linewidths=1,
        transform=ccrs.PlateCarree(),
        zorder=12
    )

    gdf_islands_bound.boundary.plot(
        ax=ax,
        edgecolor='black',
        linewidth=1,
        transform=ccrs.PlateCarree(),
        zorder=25
    )

    gdf_recifes.boundary.plot(
        ax=ax,
        edgecolor='purple',
        linewidth=0.5,
        transform=ccrs.PlateCarree(),
        zorder=30
    )

    protected_handle = mlines.Line2D(
        [], [],
        color='darkblue',
        linewidth=1.5,
        label='Área Protegida (marinha)'
    )
    reef_handle = mlines.Line2D(
        [], [],
        color='purple',
        linewidth=0.5,
        label='Recifes'
    )
    ax.legend(
        handles=[protected_handle, reef_handle],
        loc='upper right',
        fontsize=8
    )

    ax.set_title(f"Frequência de dias com DHW > {threshold} ({year})")
    fname = os.path.join(out_dir, f"dhw_frequency_gt{threshold}_{year}.png")
    plt.savefig(fname, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Mapa salvo: {fname}")

# ============================================================
# Geração dos mapas anuais
# ============================================================
for yr in yearly_4.year.values:
    plot_dhw_map(yearly_4.sel(year=yr), 4, int(yr), output_dir)
    plot_dhw_map(yearly_8.sel(year=yr), 8, int(yr), output_dir)

# ============================================================
# (Opcional) Mapas agregados para todo o período
# ============================================================
total_4 = yearly_4.sum(dim='year')
total_8 = yearly_8.sum(dim='year')
plot_dhw_map(total_4, 4, '1985-2008', output_dir)
plot_dhw_map(total_8, 8, '1985-2008', output_dir)

print("\nProcessamento concluído com sucesso – mapas anuais gerados.")


In [ ]:
### PCA GLOBAL com bubbleplot
### DLI Bentônico (Daily Light Integral) 
### Versão com PCA global apenas, agregada por SÍTIO
### CORRIGIDO COM SELETOR ÚNICO DE ANÁLISE ('analysis_selector')

# ==============================
# Imports
# ==============================
import os
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
os.environ['OMP_NUM_THREADS'] = '1'
import glob
import re
import logging
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import box, LineString
from shapely.ops import unary_union, polygonize
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from dask.distributed import Client
import dask
import psutil
import gc

# ==============================
# Configurações Iniciais e Caminhos
# ==============================
logging.basicConfig(
    filename='file_open_errors_global_pca_sites.log', # Log específico
    filemode='w',
    level=logging.ERROR,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
output_dir = r"C:\Users\rbfra\OneDrive\########PUBLICACOES\############Menezes et al. Mus his distribution and abundance Abrolhos\########NEW RESULTS\output__PCA_GLOBAL_full_SITES_CV_ALL"
os.makedirs(output_dir, exist_ok=True)
dask.config.set({
    'array.slicing.split_large_chunks': True,
    'distributed.worker.memory.target': 0.6,
    'distributed.worker.memory.spill': 0.7,
    'distributed.worker.memory.pause': 0.9,
    'distributed.worker.memory.terminate': 0.95,
    'scheduler': 'single-threaded',
    'temporary_directory': os.path.join(output_dir, 'dask_temp')
})
os.makedirs(os.path.join(output_dir, 'dask_temp'), exist_ok=True)

# ==============================
# Configurações Adicionais e Seletor de Análise
# ==============================

### <-- MUDANÇA APLICADA AQUI ###
# --- SELETOR ÚNICO DE ANÁLISE ---
# Altere esta variável para definir TODA a análise (cálculo, PCA, gráficos).
# O script calculará APENAS a métrica de CV correspondente a esta escolha.
# Opções: um número (ex: 2, 7, 32) ou a string 'all'.
analysis_selector = 'all'  # <-- ✨ ESTE É SEU ÚNICO INTERRUPTOR ✨
# analysis_selector = 2    # Exemplo: para rodar tudo para a janela de 2 dias

# As variáveis 'window_sizes' e 'CV_WINDOW_FOR_GLOBAL_PCA' foram removidas.
print(f"============================================================")
print(f"==> ANÁLISE CONFIGURADA PARA A OPÇÃO DE CV: '{analysis_selector}' <==")
print(f"============================================================")

sst_dir   = r"H:\remote sensing\CRW_SST_FULL"
modis_dir = r"H:\remote sensing\MODIS_DATA_FULL"
chl_dir   = r"H:\remote sensing\MODIS_DATA_FULL"
dhw_dir   = r"H:\remote sensing\CRW_DHW_FULL"

sst_period   = (1985, 2008)
dhw_period   = (1985, 2008)
light_period = (2002, 2008)
chl_period   = (2002, 2008)

#sst_period   = (2004, 2005)
#dhw_period   = (2004, 2005)
#light_period = (2004, 2005)
#chl_period   = (2004, 2005)

bathymetry_file      = r"C:\Users\rbfra\OneDrive\GIS shapes\gebco_2024_ASO.nc"
protected_areas_shp  = r"H:\remote sensing\Mascaras\uc_fed_agosto_2016_site_shp\uc_fed_agosto_2016_site.shp"
coast_islands_dir    = r"C:\Users\rbfra\OneDrive\GIS shapes\batimetria_new"
reefs_shp            = r"H:\remote sensing\Mascaras\Recifes_Banco_dos_Abrolhos\Recifes_Banco_dos_Abrolhos.shp"
sites_csv_file = r"C:\Users\rbfra\OneDrive\########CEBIMAR\####PROJETOS\#####Coral trade offs\sites_list_full_clean.csv"

coast_shp   = os.path.join(coast_islands_dir, "LINHA_DE_COSTA_IMAGEM_GEOCOVER_SIRGAS_2000.shp")
islands_shp = os.path.join(coast_islands_dir, "ILHAS_IMAGEM_GEOCOVER_SIRGAS_2000.shp")

lat_min, lat_max = -20.5, -14.5
lon_min, lon_max = -40.5, -35.5

sst_pattern   = 'coraltemp_v3.1_*.nc'
dhw_pattern   = '*.nc'
kd490_pattern = 'AQUA_MODIS.*.L3m.DAY.KD.Kd_490.4km.nc'
par_pattern   = 'AQUA_MODIS.*.L3m.DAY.PAR.par.4km.nc'
chl_pattern   = 'AQUA_MODIS.*.L3m.DAY.CHL.chlor_a.4km.nc'

KDPAR_GATTUSO_A, KDPAR_GATTUSO_B, KDPAR_GATTUSO_C = 0.0665, 0.874, 0.00121
KDPAR_KD490_MIN_THRESHOLD = 0.001
MIN_DEPTH_METERS, MAX_DEPTH_METERS = 1, 200

# ==============================
# Carregamento da lista de sítios
# ==============================
try:
    df_sites_full = pd.read_csv(sites_csv_file, sep=';')
    df_sites_full.columns = df_sites_full.columns.str.strip()
    for col in df_sites_full.select_dtypes(['object']).columns:
        df_sites_full[col] = df_sites_full[col].str.strip()

    print("Agregando dados por SÍTIO, removendo distinção de habitat/profundidade.")
    df_sites_info = df_sites_full.drop_duplicates(subset=['Site_name'], keep='first').reset_index(drop=True)
    
    sites = list(zip(df_sites_info['Latitude'], df_sites_info['Longitude']))
    reef_names = list(df_sites_info['Reef_name'])
    site_names = list(df_sites_info['Site_name'])
    
    print(f"DataFrame original com {len(df_sites_full)} linhas.")
    print(f"DataFrame agregado com {len(df_sites_info)} sítios únicos.")
    print("DataFrame de sítios únicos carregado e processado com sucesso.")

except Exception as e:
    logging.critical(f"FALHA CRÍTICA AO CARREGAR O ARQUIVO DE SÍTIOS: {e}")
    raise

# ==============================
# Funções Auxiliares (sem alterações)
# ==============================
def print_memory_usage(label=""): pass
def get_lat_slice(ds, lat_min, lat_max):
    lat_vals = ds['lat'].values
    return slice(lat_min, lat_max) if lat_vals[0] < lat_vals[-1] else slice(lat_max, lat_min)
def verify_file_integrity(filepath):
    # (código da função)
    checks = {'exists': os.path.exists(filepath), 'not_empty': False, 'readable': False, 'valid_netcdf': False}
    if checks['exists']:
        try:
            checks['not_empty'] = os.path.getsize(filepath) > 0
            with open(filepath, 'rb') as f: _ = f.read(100)
            checks['readable'] = True
            with xr.open_dataset(filepath) as ds: _ = ds.attrs
            checks['valid_netcdf'] = True
        except Exception as e:
            logging.warning(f"Erro na verificação de {filepath}: {str(e)}")
    return checks
def load_satellite_data(pattern, base_dir, period, var_name_options, chunks={'time': 30, 'lat': 100, 'lon': 100}):
    # (código da função)
    full_search_path = os.path.join(base_dir, pattern)
    print(f"\nBuscando em: {full_search_path} para o período {period}")
    all_files = sorted(glob.glob(full_search_path))
    if not all_files:
        logging.error(f"Nenhum arquivo encontrado para '{pattern}' em '{base_dir}'")
        return None
    candidate_paths = []
    date_pattern = re.compile(r'(\d{8})')
    for f_path in all_files:
        match = date_pattern.search(os.path.basename(f_path))
        if match and period[0] <= int(match.group(1)[:4]) <= period[1]:
            candidate_paths.append(f_path)
    if not candidate_paths:
        logging.warning(f"Nenhum arquivo encontrado para '{pattern}' no período {period}")
        return None
    valid_paths = [f for f in candidate_paths if verify_file_integrity(f)['valid_netcdf']]
    if not valid_paths:
        logging.error(f"Nenhum arquivo NetCDF válido encontrado para '{pattern}'")
        return None
    print(f"Carregando {len(valid_paths)} arquivos válidos...")
    def preprocess_with_time(ds):
        var_name = next((v for v in var_name_options if v in ds.data_vars), None)
        if var_name is None: return xr.Dataset()
        ds_subset = ds[[var_name]].astype('float32')
        if 'time' not in ds_subset.coords:
            filename = os.path.basename(ds.encoding.get("source", ""))
            match = re.search(r'(\d{8})', filename)
            if match:
                dt = pd.to_datetime(match.group(1), format='%Y%m%d')
                return ds_subset.expand_dims(time=[dt])
        return ds_subset
    try:
        ds_raw = xr.open_mfdataset(valid_paths, preprocess=preprocess_with_time, combine='by_coords', parallel=True, chunks=chunks, engine='netcdf4')
        lat_slice = get_lat_slice(ds_raw, lat_min, lat_max)
        ds_final = ds_raw.sel(lat=lat_slice, lon=slice(lon_min, lon_max))
        return ds_final.sortby('time')
    except Exception as e:
        logging.critical(f"Falha ao concatenar arquivos para '{pattern}': {e}")
        return None
def calculate_fixed_window_cv_optimized(data, window_size):
    # (código da função)
    epsilon = 1e-9
    min_periods = max(2, int(window_size * 0.25))
    with dask.config.set(scheduler='single-threaded'):
        rolling_mean = data.rolling(time=window_size, min_periods=min_periods, center=True).mean()
        rolling_std = data.rolling(time=window_size, min_periods=min_periods, center=True).std()
        return ((rolling_std / (rolling_mean + epsilon)) * 100).compute()
def calculate_kdpar_gattuso(kd490_da):
    # (código da função)
    kd490_safe = xr.where((kd490_da.isnull()) | (kd490_da <= KDPAR_KD490_MIN_THRESHOLD), np.nan, kd490_da)
    epsilon = 1e-9
    kdpar = KDPAR_GATTUSO_A + KDPAR_GATTUSO_B * kd490_safe - KDPAR_GATTUSO_C / (kd490_safe + epsilon)
    return xr.where((kdpar.isnull()) | (kdpar <= 0), np.nan, kdpar).rename("kdpar")
def calculate_benthic_dli(dli_surface_da, kdpar_da, depth_z_da):
    # (código da função)
    depth_z_positive = xr.where(depth_z_da < MIN_DEPTH_METERS, MIN_DEPTH_METERS, depth_z_da)
    attenuation = np.exp(-kdpar_da * depth_z_positive)
    return (dli_surface_da * attenuation).rename("benthic_dli")
def load_and_calculate_dli_series(kd490_pattern, par_pattern, period, base_dir, reference_da, depth_da):
    # (código da função)
    TEMP_DIR = os.path.join(output_dir, "temp_dli_files")
    os.makedirs(TEMP_DIR, exist_ok=True)
    CHUNKS = {'lat': 100, 'lon': 100}
    def get_date_map(files, period):
        date_map = {}
        for f in files:
            try:
                match = re.search(r'(\d{8})', os.path.basename(f))
                if match and period[0] <= int(match.group(1)[:4]) <= period[1] and verify_file_integrity(f)['valid_netcdf']:
                    date_map[pd.to_datetime(match.group(1), format='%Y%m%d')] = f
            except Exception as e:
                logging.error(f"Erro ao processar {f}: {str(e)}")
        return date_map
    kd490_map = get_date_map(sorted(glob.glob(os.path.join(base_dir, kd490_pattern))), period)
    par_map = get_date_map(sorted(glob.glob(os.path.join(base_dir, par_pattern))), period)
    reference_grid = reference_da.isel(time=0, drop=True).chunk(CHUNKS).load()
    depth_aligned = depth_da.interp_like(reference_grid, method='nearest').chunk(CHUNKS).load()
    common_dates = sorted(set(kd490_map.keys()) & set(par_map.keys()))
    if not common_dates: raise ValueError("Nenhuma data comum entre Kd490 e PAR encontrada")
    processed_files = []
    for i, date in enumerate(common_dates):
        try:
            with xr.open_dataset(kd490_map[date], chunks=CHUNKS) as ds_kd, \
                 xr.open_dataset(par_map[date], chunks=CHUNKS) as ds_par:
                kd490_interp = ds_kd['Kd_490'].interp_like(reference_grid, method='nearest')
                par_interp = ds_par['par'].interp_like(reference_grid, method='nearest')
                benthic_dli = calculate_benthic_dli(par_interp, calculate_kdpar_gattuso(kd490_interp), depth_aligned).expand_dims(time=[date])
                temp_file = os.path.join(TEMP_DIR, f"dli_temp_{date.strftime('%Y%m%d')}.nc")
                benthic_dli.to_netcdf(temp_file)
                processed_files.append(temp_file)
        except Exception as e:
            logging.error(f"Falha no processamento para {date}: {str(e)}")
            continue
    if not processed_files: raise ValueError("Nenhum arquivo DLI válido foi gerado")
    dli_final = xr.concat([xr.open_dataset(f)['benthic_dli'] for f in processed_files], dim='time').sortby('time')
    return dli_final.chunk({'time': 120, 'lat': 200, 'lon': 200})
def load_bathymetry(bathymetry_file, lat_min, lat_max, lon_min, lon_max):
    # (código da função)
    bathy_ds = xr.open_dataset(bathymetry_file)
    lat_slice = get_lat_slice(bathy_ds, lat_min, lat_max)
    bathy = bathy_ds['elevation'].sel(lat=lat_slice, lon=slice(lon_min, lon_max))
    mask_shallow = (((bathy >= -MAX_DEPTH_METERS) & (bathy < 0))).astype(float)
    return bathy, mask_shallow
def extract_site_values(da, sites):
    # (código da função)
    if da is None: return [np.nan] * len(sites)
    if isinstance(da, xr.Dataset): da = da[list(da.data_vars)[0]]
    lats = xr.DataArray([s[0] for s in sites], dims="site"); lons = xr.DataArray([s[1] for s in sites], dims="site")
    return da.sel(lat=lats, lon=lons, method='nearest').values
def load_coast_and_islands(coast_file, islands_file, lat_min, lat_max, lon_min, lon_max):
    # (código da função)
    bounding_box = box(lon_min, lat_min, lon_max, lat_max)
    bbox_gdf = gpd.GeoDataFrame({'geometry': [bounding_box]}, crs="EPSG:4326")
    try:
        gdf_coast_line = gpd.read_file(coast_file).to_crs("EPSG:4326")
        gdf_islands = gpd.read_file(islands_file).to_crs("EPSG:4326")
        gdf_islands_clipped = gpd.overlay(gdf_islands, bbox_gdf, how='intersection')
        coast_union = gdf_coast_line.geometry.unary_union
        bbox_line = LineString(list(bounding_box.exterior.coords))
        combined_lines = unary_union([coast_union, bbox_line])
        polygons = list(polygonize(combined_lines))
        ocean_polygon = max([p for p in polygons if p.intersects(bounding_box)], key=lambda p: p.area, default=bounding_box)
        land_polygon = bounding_box.difference(ocean_polygon)
        if not gdf_islands_clipped.empty: land_polygon = unary_union([land_polygon, gdf_islands_clipped.unary_union])
        return gpd.GeoDataFrame({'geometry': [land_polygon]}, crs="EPSG:4326")
    except Exception as e:
        logging.warning(f"Não foi possível ler shapefiles de costa/ilhas: {e}"); return None
def plot_spatial_map(da, title, output_filename, cmap='viridis', cbar_label=''):
    # (código da função)
    if da is None: print(f"Dados nulos para o mapa '{title}'."); return
    if isinstance(da, xr.Dataset): da = da[list(da.data_vars)[0]]
    if mask_shallow is not None:
        mask_aligned = mask_shallow.interp_like(da, method='nearest')
        da = da.where(mask_aligned >= 0.5)
    fig, ax = plt.subplots(figsize=(8, 6)); ax.set_xlim(lon_min, lon_max); ax.set_ylim(lat_min, lat_max)
    ax.set_facecolor('white')
    cmap_obj = plt.get_cmap(cmap); cmap_obj.set_bad('white', 1.)
    data_to_plot = da.load()
    valid_data = data_to_plot.values[np.isfinite(data_to_plot.values)]
    vmin, vmax = (valid_data.min(), valid_data.max()) if valid_data.size > 0 else (None, None)
    img = ax.pcolormesh(data_to_plot.lon, data_to_plot.lat, data_to_plot, cmap=cmap_obj, shading='auto', vmin=vmin, vmax=vmax)
    cbar = plt.colorbar(img, ax=ax, orientation='vertical', pad=0.02)
    if cbar_label: cbar.set_label(cbar_label)
    ax.set_title(title); ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
    gdf_land = load_coast_and_islands(coast_shp, islands_shp, lat_min, lat_max, lon_min, lon_max)
    if gdf_land is not None: gdf_land.boundary.plot(ax=ax, edgecolor='black', linewidth=1, zorder=3)
    if bathy_data is not None: ax.contour(bathy_data.lon, bathy_data.lat, bathy_data, levels=[-200], colors='dimgray', linewidths=1, zorder=2)
    if os.path.exists(reefs_shp): gpd.read_file(reefs_shp).to_crs("EPSG:4326").boundary.plot(ax=ax, edgecolor='purple', linewidth=0.7, zorder=4)
    if os.path.exists(protected_areas_shp) and 'gdf_land' in locals() and gdf_land is not None:
        protected_marine = gpd.overlay(gpd.read_file(protected_areas_shp).to_crs("EPSG:4326"), gdf_land, how='difference')
        if not protected_marine.empty: protected_marine.boundary.plot(ax=ax, edgecolor='darkblue', linewidth=1.5, zorder=5)
    plt.tight_layout(); plt.savefig(output_filename, dpi=300, facecolor='white'); plt.close(fig)
    print(f"Mapa '{title}' salvo em {output_filename}.")
def scale_marker_sizes(values, scale_factor=300, min_size=30):
    # (código da função)
    vals = np.asarray(values, dtype=float); valid = np.isfinite(vals)
    if valid.sum() == 0: return np.full(vals.shape, min_size)
    vmin, vmax = vals[valid].min(), vals[valid].max()
    if vmax == vmin: return np.full(vals.shape, min_size + scale_factor/2)
    scaled = (vals - vmin) / (vmax - vmin); scaled[~valid] = 0
    return scaled * scale_factor + min_size

# ==============================
# Processamento Principal
# ==============================
print("\n--- 1. Carregando e Preparando Dados Base ---")
bathy_data, mask_shallow = load_bathymetry(bathymetry_file, lat_min, lat_max, lon_min, lon_max)
if bathy_data is None: raise ValueError("FALHA CRÍTICA: Não foi possível carregar batimetria.")
sst_all_ds = load_satellite_data(sst_pattern, sst_dir, sst_period, ['analysed_sst', 'sea_surface_temperature', 'sst'])
if sst_all_ds is None: raise ValueError("FALHA CRÍTICA: Nenhum dado de SST encontrado.")
dhw_all_ds = load_satellite_data(dhw_pattern, dhw_dir, dhw_period, ['degree_heating_week'])
sst_all = sst_all_ds[list(sst_all_ds.data_vars)[0]].astype('float32').chunk({'time': 30, 'lat': 100, 'lon': 100})
dhw_all = dhw_all_ds[list(dhw_all_ds.data_vars)[0]].astype('float32').chunk({'time': 'auto'}) if dhw_all_ds is not None else None
del sst_all_ds, dhw_all_ds
gc.collect()

print("\n--- 2. Carregando e Preparando Dados Auxiliares ---")
chl_all_ds = load_satellite_data(chl_pattern, chl_dir, chl_period, ['chlor_a'])
if chl_all_ds is not None:
    chl_all = chl_all_ds.interp_like(sst_all, method='nearest')[list(chl_all_ds.data_vars)[0]].astype('float32').chunk({'time': 'auto'})
    del chl_all_ds
else:
    chl_all = None
gc.collect()
print("\nCalculando mapa de DLI com batimetria GEBCO...")
depth_z_meters = -bathy_data.where((bathy_data >= -MAX_DEPTH_METERS) & (bathy_data <= -MIN_DEPTH_METERS))
dli_all = load_and_calculate_dli_series(kd490_pattern, par_pattern, light_period, modis_dir, sst_all, depth_z_meters)
if dli_all is None:
    print("AVISO: Mapa de DLI para visualização não pôde ser calculado.")

print("\n--- 3. Definindo e Computando Métricas Espaciais ---")
computed_results = {}
dask.config.set({'array.slicing.split_large_chunks': True, 'scheduler': 'single-threaded'})
print("Calculando mean_SST...")
computed_results['mean_SST'] = sst_all.mean('time', skipna=True).compute()

### <-- MUDANÇA APLICADA AQUI ###
# Calcula o CV de SST com base no seletor, em vez de um loop
if analysis_selector == 'all':
    print("Calculando cv_all para SST...")
    computed_results['cv_all'] = ((sst_all.std('time', skipna=True) / (sst_all.mean('time', skipna=True) + 1e-9)) * 100).compute()
else:
    window = int(analysis_selector)
    print(f"Calculando cv_{window} para SST...")
    cv = calculate_fixed_window_cv_optimized(sst_all, window)
    computed_results[f'cv_{window}'] = cv.mean('time', skipna=True).compute()
    del cv

if dhw_all is not None:
    print("Calculando métricas DHW...")
    computed_results['prop_DHW_gt4'] = (dhw_all > 4).mean('time', skipna=True).compute() * 100
    computed_results['prop_DHW_gt8'] = (dhw_all > 8).mean('time', skipna=True).compute() * 100
if chl_all is not None:
    print("Calculando mean_CHL...")
    computed_results['mean_CHL'] = chl_all.mean('time', skipna=True).compute()
if dli_all is not None:
    print("\nCalculando métricas do mapa de DLI...")
    computed_results['mean_DLI_map'] = dli_all.mean('time', skipna=True).compute()
    
    ### <-- MUDANÇA APLICADA AQUI ###
    # Calcula o CV de DLI com base no seletor, em vez de um loop
    if analysis_selector == 'all':
        print("Calculando dli_cv_all...")
        computed_results['dli_cv_all'] = ((dli_all.std('time', skipna=True) / (dli_all.mean('time', skipna=True) + 1e-9)) * 100).compute()
    else:
        window = int(analysis_selector)
        print(f"Calculando dli_cv_{window}...")
        cv_dli = calculate_fixed_window_cv_optimized(dli_all, window)
        computed_results[f'dli_cv_{window}'] = cv_dli.mean('time', skipna=True).compute()
        del cv_dli
else:
    computed_results['mean_DLI_map'] = None
    # Garante que as chaves de CV de DLI não existentes sejam nulas
    if analysis_selector == 'all':
        computed_results['dli_cv_all'] = None
    else:
        computed_results[f'dli_cv_{int(analysis_selector)}'] = None

del sst_all, dhw_all, chl_all
if 'dli_all' in locals(): del dli_all
gc.collect()

# =============================================================================
# SEÇÃO 4: GERAÇÃO DE SAÍDAS E ANÁLISE PCA GLOBAL
# =============================================================================

print("\n--- 4. Geração de Mapas e Planilhas ---")

### <-- MUDANÇA APLICADA AQUI ###
# Define as variáveis de CV para a análise com base no seletor único
if analysis_selector == 'all':
    selected_cv_var_sst = 'cv_all'
    selected_dli_cv_var = 'dli_cv_all'
else:
    # Garante que a conversão para int não falhe
    try:
        window = int(analysis_selector)
        selected_cv_var_sst = f'cv_{window}'
        selected_dli_cv_var = f'dli_cv_{window}'
    except ValueError:
        raise ValueError(f"analysis_selector ('{analysis_selector}') deve ser 'all' ou um número inteiro.")

print(f"Variável de CV de SST selecionada para a análise: '{selected_cv_var_sst}'")
print(f"Variável de CV de DLI selecionada para a análise: '{selected_dli_cv_var}'")

# A lógica de mapeamento e plotagem agora usa as variáveis selecionadas dinamicamente
map_configs = {
    'mean_SST': ("Média de SST", "SST (°C)", 'viridis'),
    selected_cv_var_sst: (f"CV de SST ({analysis_selector}d)", "CV (%)", 'viridis'),
    'prop_DHW_gt4': ("Proporção DHW>4", "Proporção (%)", 'YlOrRd'),
    'prop_DHW_gt8': ("Proporção DHW>8", "Proporção (%)", 'YlOrRd'),
    'mean_CHL': ("Média de Clorofila", "Chl-a (mg/m³)", 'plasma'),
    'mean_DLI_map': ("Média de DLI Bentônico (Mapa GEBCO)", "DLI (mol/m²/day)", 'magma'),
    selected_dli_cv_var: (f"CV de DLI ({analysis_selector}d, Mapa GEBCO)", "CV (%)", 'magma')
}
for key, (title, cbar, cmap) in map_configs.items():
    if key and key in computed_results and computed_results[key] is not None:
        plot_spatial_map(computed_results[key], title, os.path.join(output_dir, f"mapa_{key}.png"), cmap=cmap, cbar_label=cbar)

# Geração da Planilha Base para a PCA Global
df_sites_base_global = df_sites_info.copy()
global_vars_to_extract = [selected_cv_var_sst, selected_dli_cv_var, 'prop_DHW_gt4', 'mean_SST', 'mean_CHL', 'mean_DLI_map']
for var in global_vars_to_extract:
    if var in computed_results and computed_results[var] is not None:
        df_sites_base_global[var] = extract_site_values(computed_results[var], sites)
    else:
        df_sites_base_global[var] = np.nan
excel_file_global = os.path.join(output_dir, "planilha_base_analise_global_sites.xlsx")
df_sites_base_global.to_excel(excel_file_global, index=False)
print(f"Planilha base para análise GLOBAL por SÍTIO salva em {excel_file_global}")

# --- ANÁLISE PCA GLOBAL (PURAMENTE ESPACIAL) ---
print("\n--- Iniciando Análise PCA GLOBAL ---")
# As variáveis para a PCA são as mesmas que foram extraídas para a planilha
map_vars_for_global_pca = {
    var: computed_results.get(var) for var in global_vars_to_extract
}
global_pca_data_list = [v for v in map_vars_for_global_pca.values() if v is not None]
global_pca_var_names = [k for k, v in map_vars_for_global_pca.items() if v is not None]

print(f"Variáveis utilizadas na PCA GLOBAL: {global_pca_var_names}")

transformed_global_names = list(global_pca_var_names)
for i, name in enumerate(global_pca_var_names):
    if name == 'prop_DHW_gt4': global_pca_data_list[i] = np.sqrt(global_pca_data_list[i] + 0.5); transformed_global_names[i] = "sqrt(DHW>4)"
    elif name == 'mean_CHL': global_pca_data_list[i] = np.log1p(global_pca_data_list[i]); transformed_global_names[i] = "log(CHL+1)"
    elif name == 'mean_DLI_map': global_pca_data_list[i] = np.log1p(global_pca_data_list[i]); transformed_global_names[i] = "log(DLI_map_mean+1)"
mask_valid_global = np.isfinite(global_pca_data_list[0])
for da in global_pca_data_list[1:]: mask_valid_global = mask_valid_global & np.isfinite(da)
indices_valid_global = np.where(mask_valid_global.values.flatten())[0]
flat_vars_valid = [da.values.flatten()[indices_valid_global] for da in global_pca_data_list]
data_matrix_global = np.column_stack(flat_vars_valid)
scaler_global = StandardScaler()
norm_data_global = scaler_global.fit_transform(data_matrix_global)
pca_global = PCA(n_components=2)
scores_global_pixels = pca_global.fit_transform(norm_data_global)
explained_variance_global = pca_global.explained_variance_ratio_ * 100
loadings_df_global = pd.DataFrame(pca_global.components_.T, columns=["PC1", "PC2"], index=transformed_global_names)
loadings_path_global = os.path.join(output_dir, "loadings_PCA_GLOBAL.csv")
loadings_df_global.to_csv(loadings_path_global)
print(f"Loadings da PCA GLOBAL salvos em {loadings_path_global}")
ny, nx = mask_valid_global.shape
pc1_map = np.full(ny * nx, np.nan); pc2_map = np.full(ny * nx, np.nan)
pc1_map[indices_valid_global] = scores_global_pixels[:, 0]
pc2_map[indices_valid_global] = scores_global_pixels[:, 1]
coords = mask_valid_global.coords
pc1_da_global = xr.DataArray(pc1_map.reshape(ny, nx), coords=coords, dims=['lat', 'lon'])
pc2_da_global = xr.DataArray(pc2_map.reshape(ny, nx), coords=coords, dims=['lat', 'lon'])
plot_spatial_map(pc1_da_global, f"PCA Global - PC1 ({explained_variance_global[0]:.1f}% var.)", os.path.join(output_dir, "mapa_PCA_GLOBAL_PC1.png"), cmap='coolwarm', cbar_label='PC1 Score')
plot_spatial_map(pc2_da_global, f"PCA Global - PC2 ({explained_variance_global[1]:.1f}% var.)", os.path.join(output_dir, "mapa_PCA_GLOBAL_PC2.png"), cmap='coolwarm', cbar_label='PC2 Score')

# Extração de scores para os sítios
df_scores_global = df_sites_info.copy()
df_scores_global['PC1_global'] = extract_site_values(pc1_da_global, sites)
df_scores_global['PC2_global'] = extract_site_values(pc2_da_global, sites)
scores_path_global = os.path.join(output_dir, "scores_PCA_GLOBAL_sites.xlsx")
df_scores_global.to_excel(scores_path_global, index=False)
print(f"Scores da PCA GLOBAL para os SÍTIOS salvos em {scores_path_global}")

# =============================================================================
# SEÇÃO 5: GERAÇÃO DOS GRÁFICOS DE ORDENAÇÃO E BUBBLE PLOTS
# =============================================================================
print("\n--- 5. Gerando Gráficos Finais da PCA Global por SÍTIO ---")

# A lógica de plotagem é simplificada e não precisa de alterações
unique_reefs = df_sites_info['Reef_name'].unique()
cmap = plt.get_cmap('tab10')
color_map = {r: cmap(i % 10) for i, r in enumerate(unique_reefs)}

# Gráfico de Ordenação da PCA GLOBAL (por SÍTIO)
plt.figure(figsize=(8, 7))
for _, row in df_scores_global.iterrows():
    plt.scatter(row['PC1_global'], row['PC2_global'], color=color_map[row['Reef_name']], marker='o', s=80, alpha=0.8)

legend_elements_color = [plt.Line2D([0], [0], marker='o', color='w', label=reef, markersize=10, markerfacecolor=color_map[reef]) for reef in unique_reefs]
plt.legend(title="Recife", handles=legend_elements_color, loc='upper left')

plt.xlabel(f"PC1 ({explained_variance_global[0]:.1f}%)")
plt.ylabel(f"PC2 ({explained_variance_global[1]:.1f}%)")
plt.title("Ordenação PCA GLOBAL (por Sítio)")
plt.grid(True, linestyle='--', alpha=0.6)
plt.axhline(0, color='grey', lw=0.5)
plt.axvline(0, color='grey', lw=0.5)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "ord_PCA_GLOBAL_sites.png"), dpi=300)
plt.close()

# Bubble Plots da PCA GLOBAL (por SÍTIO)
bubble_vars_global = {}
# As variáveis para o bubble plot são as mesmas que entraram na PCA
for var in global_pca_var_names:
    bubble_vars_global[var] = extract_site_values(computed_results[var], sites)

for var_name, site_values in bubble_vars_global.items():
    if site_values is None or np.all(np.isnan(site_values)):
        print(f"Pulando bubble plot para '{var_name}', pois não há dados.")
        continue
    sizes = scale_marker_sizes(site_values)
    plt.figure(figsize=(8, 7))
    for i, row in df_scores_global.iterrows():
        plt.scatter(row['PC1_global'], row['PC2_global'], s=sizes[i], color=color_map[row['Reef_name']], marker='o', alpha=0.7, edgecolor='k')
    
    plt.legend(title="Recife", handles=legend_elements_color, loc='upper left')
    
    plt.xlabel(f"PC1 ({explained_variance_global[0]:.1f}%)")
    plt.ylabel(f"PC2 ({explained_variance_global[1]:.1f}%)")
    plt.title(f"PCA Global por Sítio - Tamanho ∝ {var_name}")
    plt.grid(True, linestyle='--')
    plt.axhline(0, color='grey', lw=0.5)
    plt.axvline(0, color='grey', lw=0.5)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"pca_bubble_GLOBAL_sites_{var_name}.png"), dpi=300)
    plt.close()

print("\nScript da PCA Global por SÍTIO finalizado com sucesso!")

In [ ]:
# Roda PERMANOVA, ANOVA e Levene para REEF e HAB
# PCA GLOBAL

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from skbio.stats.distance import DistanceMatrix
from skbio.stats.distance import permanova
from scipy.spatial.distance import pdist, squareform
from scipy.stats import f_oneway, levene

# === Caminhos ===
# Assumindo que seu arquivo de scores também contém as colunas 'HAB', 'Depth_m' e 'mean_DLI_local'
input_file = r"C:\Users\rbfra\OneDrive\########PUBLICACOES\############Menezes et al. Mus his distribution and abundance Abrolhos\########NEW RESULTS\output__PCA_GLOBAL_full_SITES_CV_ALL\scores_PCA_GLOBAL_sites.xlsx"
output_dir = r"C:\Users\rbfra\OneDrive\########PUBLICACOES\############Menezes et al. Mus his distribution and abundance Abrolhos\########NEW RESULTS\PERMANOVA_ANOVA"
os.makedirs(output_dir, exist_ok=True)

# === Carrega e prepara os dados ===
df = pd.read_excel(input_file)
# Lista de colunas essenciais para as análises
required_cols = ['PC1', 'PC2', 'Reef_name', 'Site_name', 'HAB', 'Depth_m', 'mean_DLI_local']
df_valid = df.dropna(subset=required_cols).copy()

print(f"Dados carregados: {len(df_valid)} linhas válidas.")
print("\nContagem de habitats por recife:")
print(df_valid.groupby('Reef_name')['HAB'].value_counts())

# === Matriz de distância Euclidiana (base para todas as PERMANOVAs) ===
data_matrix = df_valid[['PC1', 'PC2']].values
dist_matrix = squareform(pdist(data_matrix, metric='euclidean'))
dm = DistanceMatrix(dist_matrix, ids=df_valid['Site_name'].astype(str).tolist())

# ============================================
# ANÁLISE POR REEF
# ============================================
print("\n--- Analisando por REEF ---")
grouping_reef = df_valid.set_index('Site_name')['Reef_name']

# PERMANOVA para REEF
permanova_reef = permanova(distance_matrix=dm, grouping=grouping_reef, permutations=999)

# ANOVA e Levene para REEF vs PC1/PC2
grouped_pc1_reef = [g['PC1'].values for _, g in df_valid.groupby('Reef_name')]
anova_pc1_reef = f_oneway(*grouped_pc1_reef)
levene_pc1_reef = levene(*grouped_pc1_reef)

grouped_pc2_reef = [g['PC2'].values for _, g in df_valid.groupby('Reef_name')]
anova_pc2_reef = f_oneway(*grouped_pc2_reef)
levene_pc2_reef = levene(*grouped_pc2_reef)

# Boxplots para REEF
fig_reef, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.boxplot(ax=axes[0], x='Reef_name', y='PC1', data=df_valid, palette='viridis')
axes[0].set_title(f'PC1 por REEF\nANOVA p={anova_pc1_reef.pvalue:.4f} | Levene p={levene_pc1_reef.pvalue:.4f}')
axes[0].tick_params(axis='x', rotation=45)
sns.boxplot(ax=axes[1], x='Reef_name', y='PC2', data=df_valid, palette='viridis')
axes[1].set_title(f'PC2 por REEF\nANOVA p={anova_pc2_reef.pvalue:.4f} | Levene p={levene_pc2_reef.pvalue:.4f}')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "boxplots_PCA_vs_REEF.png"), dpi=300)
plt.close()

# ============================================
# ANÁLISE POR HABITAT
# ============================================
print("\n--- Analisando por HAB ---")
grouping_hab = df_valid.set_index('Site_name')['HAB']

# PERMANOVA para HAB
permanova_hab = permanova(distance_matrix=dm, grouping=grouping_hab, permutations=999)

# ANOVA e Levene para HAB vs PC1/PC2
grouped_pc1_hab = [g['PC1'].values for _, g in df_valid.groupby('HAB')]
anova_pc1_hab = f_oneway(*grouped_pc1_hab)
levene_pc1_hab = levene(*grouped_pc1_hab)

grouped_pc2_hab = [g['PC2'].values for _, g in df_valid.groupby('HAB')]
anova_pc2_hab = f_oneway(*grouped_pc2_hab)
levene_pc2_hab = levene(*grouped_pc2_hab)

# Boxplots para HAB
fig_hab, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(ax=axes[0], x='HAB', y='PC1', data=df_valid, palette='pastel')
axes[0].set_title(f'PC1 por HABITAT\nANOVA p={anova_pc1_hab.pvalue:.4f} | Levene p={levene_pc1_hab.pvalue:.4f}')
sns.boxplot(ax=axes[1], x='HAB', y='PC2', data=df_valid, palette='pastel')
axes[1].set_title(f'PC2 por HABITAT\nANOVA p={anova_pc2_hab.pvalue:.4f} | Levene p={levene_pc2_hab.pvalue:.4f}')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "boxplots_PCA_vs_HAB.png"), dpi=300)
plt.close()

# ============================================
# ANÁLISE DIRETA: HABITAT vs Variáveis Físicas
# ============================================
print("\n--- Analisando link direto: HAB vs. Depth e DLI ---")

# ANOVA para Depth vs HAB
grouped_depth_hab = [g['Depth_m'].values for _, g in df_valid.groupby('HAB')]
anova_depth_hab = f_oneway(*grouped_depth_hab)

# ANOVA para DLI vs HAB
grouped_dli_hab = [g['mean_DLI_local'].values for _, g in df_valid.groupby('HAB')]
anova_dli_hab = f_oneway(*grouped_dli_hab)

# Boxplots para Variáveis Físicas vs HAB
fig_direct, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(ax=axes[0], x='HAB', y='Depth_m', data=df_valid, palette='mako')
axes[0].set_title(f'Profundidade por HABITAT\nANOVA p={anova_depth_hab.pvalue:.4f}')
axes[0].set_ylabel('Profundidade (m)')
sns.boxplot(ax=axes[1], x='HAB', y='mean_DLI_local', data=df_valid, palette='rocket')
axes[1].set_title(f'DLI Médio por HABITAT\nANOVA p={anova_dli_hab.pvalue:.4f}')
axes[1].set_ylabel('DLI Médio (mol/m²/dia)')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "boxplots_Fisico_vs_HAB.png"), dpi=300)
plt.close()

# ============================================
# Salvar todos os resultados numéricos
# ============================================
output_txt = os.path.join(output_dir, "resultados_completos_justificativa.txt")
with open(output_txt, 'w') as f:
    f.write("="*50 + "\n")
    f.write("RESULTADOS DA ANÁLISE DE REDUNDÂNCIA DE FATORES LOCAIS\n")
    f.write("="*50 + "\n\n")

    f.write("CAMADA 1: ESTRUTURAÇÃO DO AMBIENTE GERAL (PERMANOVA)\n")
    f.write("-" * 50 + "\n")
    f.write("PERMANOVA baseada em REEF:\n")
    f.write(str(permanova_reef) + "\n\n")
    f.write("PERMANOVA baseada em HAB:\n")
    f.write(str(permanova_hab) + "\n\n")

    f.write("CAMADA 2: REDUNDÂNCIA COM OS EIXOS DA PCA (ANOVA)\n")
    f.write("-" * 50 + "\n")
    f.write("Resultados para REEF:\n")
    f.write(f"  PC1 vs REEF: ANOVA F={anova_pc1_reef.statistic:.3f}, p={anova_pc1_reef.pvalue:.4f} | Levene W={levene_pc1_reef.statistic:.3f}, p={levene_pc1_reef.pvalue:.4f}\n")
    f.write(f"  PC2 vs REEF: ANOVA F={anova_pc2_reef.statistic:.3f}, p={anova_pc2_reef.pvalue:.4f} | Levene W={levene_pc2_reef.statistic:.3f}, p={levene_pc2_reef.pvalue:.4f}\n\n")
    f.write("Resultados para HAB:\n")
    f.write(f"  PC1 vs HAB: ANOVA F={anova_pc1_hab.statistic:.3f}, p={anova_pc1_hab.pvalue:.4f} | Levene W={levene_pc1_hab.statistic:.3f}, p={levene_pc1_hab.pvalue:.4f}\n")
    f.write(f"  PC2 vs HAB: ANOVA F={anova_pc2_hab.statistic:.3f}, p={anova_pc2_hab.pvalue:.4f} | Levene W={levene_pc2_hab.statistic:.3f}, p={levene_pc2_hab.pvalue:.4f}\n\n")

    f.write("CAMADA 3: LINK DIRETO COM VARIÁVEIS FÍSICAS (ANOVA)\n")
    f.write("-" * 50 + "\n")
    f.write(f"  Profundidade (Depth_m) vs HAB: ANOVA F={anova_depth_hab.statistic:.3f}, p={anova_depth_hab.pvalue:.4f}\n")
    f.write(f"  DLI Médio (mean_DLI_local) vs HAB: ANOVA F={anova_dli_hab.statistic:.3f}, p={anova_dli_hab.pvalue:.4f}\n")

print("\nAnálise de justificativa finalizada com sucesso!")
print(f"Resultados e gráficos exportados para:\n{output_dir}")

In [ ]:
### PCA LOCAL com bubbleplot - VERSÃO AVANÇADA 2.3 (Estilo de Arco Aprimorado) ###
# 1. Realiza duas PCAs separadas: Variabilidade (CVs) e Magnitude (Médias + DHW).
# 2. Diferencia arcos com bordas distintas (grossa/preta para inner, fina/cinza para outer).
# 3. Adiciona uma legenda explicando o estilo das bordas dos arcos.
# 4. Gera resumo de segmentos e salva scores de AMBAS as PCAs.
# 5. Funciona perfeitamente com analysis_selector = 'all' ou numérico.
# 6. Figuras PCA compostas

# ==============================
# Imports
# ==============================
import os
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
os.environ['OMP_NUM_THREADS'] = '1'
import glob
import re
import logging
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import dask
import psutil
import gc

# ==============================
# Configurações Iniciais
# ==============================
logging.basicConfig(
    filename='pca_local_errors.log',
    filemode='w',
    level=logging.ERROR,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# ==============================
# Caminhos e parâmetros principais
# ==============================
output_dir = r"C:\Users\rbfra\OneDrive\########CEBIMAR\###Orientacoes e Supervisoes\###Mestrado Mariana\PROJETO FAPESP\R1_plus_v2"
os.makedirs(output_dir, exist_ok=True)

# ==============================
# Configurações do Dask
# ==============================
dask.config.set({
    'array.slicing.split_large_chunks': True,
    'scheduler': 'single-threaded',
    'temporary_directory': os.path.join(output_dir, 'dask_temp')
})
os.makedirs(os.path.join(output_dir, 'dask_temp'), exist_ok=True)

# ==============================
# Parâmetros da Análise
# ==============================
analysis_selector = 'all'

print(f"============================================================")
print(f"==> ANÁLISE CONFIGURADA PARA A OPÇÃO DE CV: '{analysis_selector}' <==")
print(f"============================================================")

# --- Períodos e Caminhos ---
sst_dir   = r"E:\remote sensing\CRW_SST_FULL"
modis_dir = r"E:\remote sensing\MODIS_DATA_FULL"
chl_dir   = r"E:\remote sensing\MODIS_DATA_FULL"
dhw_dir   = r"E:\remote sensing\CRW_DHW_FULL"

sst_period   = (2020, 2024)
dhw_period   = (2020, 2024)
light_period = (2020, 2024)
chl_period   = (2020, 2024)

#sst_period   = (2007, 2008)
#dhw_period   = (2007, 2008)
#light_period = (2007, 2008)
#chl_period   = (2007, 2008)


sites_csv_file = r"C:\Users\rbfra\OneDrive\########CEBIMAR\####PROJETOS\#####Coral trade offs\sites_list_full.csv"
lat_min, lat_max = -20.5, -14.5
lon_min, lon_max = -40.5, -35.5
sst_pattern   = 'coraltemp_v3.1_*.nc'
dhw_pattern   = '*.nc'
kd490_pattern = 'AQUA_MODIS.*.L3m.DAY.KD.Kd_490.4km.nc'
par_pattern   = 'AQUA_MODIS.*.L3m.DAY.PAR.par.4km.nc'
chl_pattern   = 'AQUA_MODIS.*.L3m.DAY.CHL.chlor_a.4km.nc'
KDPAR_GATTUSO_A, KDPAR_GATTUSO_B, KDPAR_GATTUSO_C = 0.0665, 0.874, 0.00121
KDPAR_KD490_MIN_THRESHOLD = 0.001

# ==============================
# Carregamento da lista de sítios
# ==============================
try:
    df_sites_info = pd.read_csv(sites_csv_file, sep=';')
    df_sites_info.columns = [col.strip().replace(' ', '_') for col in df_sites_info.columns]
    for col in df_sites_info.select_dtypes(['object']).columns:
        df_sites_info[col] = df_sites_info[col].str.strip()
    df_sites_info['unique_id'] = df_sites_info['Site_name'] + '_' + df_sites_info['HAB']
    sites = list(zip(df_sites_info['Latitude'], df_sites_info['Longitude']))
    print("DataFrame de sítios carregado com sucesso.")
except Exception as e:
    logging.critical(f"FALHA CRÍTICA AO CARREGAR O ARQUIVO DE SÍTIOS: {e}")
    raise

# ==============================
# Funções Auxiliares Essenciais
# ==============================
# Funções de carregamento de dados e cálculo de métricas permanecem as mesmas da versão anterior.
# Omitidas para brevidade.
def verify_file_integrity(filepath):
    if not os.path.exists(filepath) or os.path.getsize(filepath) == 0: return False
    try:
        with xr.open_dataset(filepath) as ds: _ = ds.attrs
        return True
    except Exception: return False
def load_satellite_data(pattern, base_dir, period, var_name_options, chunks={'time': 30, 'lat': 100, 'lon': 100}):
    full_search_path = os.path.join(base_dir, pattern); print(f"\n--- Carregando dados para: {pattern} em {period} ---")
    all_files = sorted(glob.glob(full_search_path));
    if not all_files: logging.error(f"Nenhum arquivo encontrado para '{pattern}' em {base_dir}"); return None
    candidate_paths = [f for f in all_files if (match := re.search(r'(\d{4})\d{4}', os.path.basename(f))) and period[0] <= int(match.group(1)) <= period[1]]
    if not candidate_paths: logging.warning(f"Nenhum arquivo encontrado para '{pattern}' no período {period}"); return None
    valid_paths = [f for f in candidate_paths if verify_file_integrity(f)]
    if not valid_paths: logging.error(f"Nenhum arquivo VÁLIDO encontrado para '{pattern}' no período {period}"); return None
    print(f"Encontrados {len(valid_paths)} arquivos válidos.")
    def preprocess_with_time(ds):
        var_name = next((v for v in var_name_options if v in ds.data_vars), None)
        if var_name is None: return xr.Dataset()
        ds_subset = ds[[var_name]].astype('float32')
        if 'time' not in ds_subset.coords:
            match = re.search(r'(\d{8})', os.path.basename(ds.encoding.get("source", "")))
            if match: dt = pd.to_datetime(match.group(1), format='%Y%m%d'); return ds_subset.expand_dims(time=[dt])
        return ds_subset
    try:
        ds_raw = xr.open_mfdataset(valid_paths, preprocess=preprocess_with_time, combine='by_coords', parallel=True, chunks=chunks, engine='netcdf4')
        if not ds_raw.data_vars: print(f"AVISO: Nenhum dado carregado para '{pattern}'."); return None
        lat_slice = slice(lat_max, lat_min) if ds_raw['lat'].values[0] > ds_raw['lat'].values[-1] else slice(lat_min, lat_max)
        ds_final = ds_raw.sel(lat=lat_slice, lon=slice(lon_min, lon_max))
        return ds_final.sortby('time')[list(ds_final.data_vars)[0]]
    except Exception as e: logging.critical(f"Falha ao concatenar arquivos para '{pattern}': {e}"); return None
def calculate_kdpar_gattuso(kd490_da):
    kd490_safe = xr.where((kd490_da.isnull()) | (kd490_da <= KDPAR_KD490_MIN_THRESHOLD), np.nan, kd490_da)
    epsilon = 1e-9; kdpar = KDPAR_GATTUSO_A + KDPAR_GATTUSO_B * kd490_safe - KDPAR_GATTUSO_C / (kd490_safe + epsilon)
    kdpar_final = xr.where((kdpar.isnull()) | (kdpar <= 0), np.nan, kdpar); kdpar_final.name = "kdpar"
    return kdpar_final
def extract_and_compute_site_metrics(da, sites_coords, metrics_to_calc):
    if da is None: return {}
    if isinstance(da, xr.Dataset): da = da[list(da.data_vars)[0]]
    lats = xr.DataArray([s[0] for s in sites_coords], dims="site"); lons = xr.DataArray([s[1] for s in sites_coords], dims="site")
    site_timeseries = da.sel(lat=lats, lon=lons, method='nearest').load()
    results = {}
    for metric in metrics_to_calc:
        print(f"  Calculando métrica: {metric}...")
        if metric == 'mean':
            results['mean'] = site_timeseries.mean('time', skipna=True).values
        elif metric == 'cv_all':
            mean_vals = site_timeseries.mean('time', skipna=True); std_vals = site_timeseries.std('time', skipna=True)
            results['cv_all'] = (std_vals / (mean_vals + 1e-9) * 100).values
            results['cv_all_segment_count'] = site_timeseries.notnull().sum('time').values
        elif metric.startswith('cv_'):
            try:
                window = int(metric.split('_')[1]); min_p = max(2, int(window * 0.25))
                rolling_mean = site_timeseries.rolling(time=window, min_periods=min_p, center=True).mean()
                rolling_std = site_timeseries.rolling(time=window, min_periods=min_p, center=True).std()
                cv_ts = (rolling_std / (rolling_mean + 1e-9)) * 100
                results[metric] = cv_ts.mean('time', skipna=True).values
                results[f'{metric}_segment_count'] = cv_ts.notnull().sum('time').values
            except (ValueError, IndexError): print(f"    AVISO: Não foi possível processar a métrica '{metric}'. Pulando."); continue
        elif metric.startswith('prop_gt_'):
            threshold = float(metric.split('_')[-1])
            results[metric] = (site_timeseries > threshold).mean('time', skipna=True).values
    return results
def calculate_site_specific_dli(kd490_da, par_da, sites_info_df, cv_selector):
    if kd490_da is None or par_da is None: return {}
    lats = xr.DataArray(sites_info_df['Latitude'].values, dims="site"); lons = xr.DataArray(sites_info_df['Longitude'].values, dims="site")
    depths = xr.DataArray(sites_info_df['Depth_m'].values, dims="site")
    kd490_points = kd490_da.sel(lat=lats, lon=lons, method='nearest').load()
    par_points = par_da.sel(lat=lats, lon=lons, method='nearest').load()
    kdpar_points = calculate_kdpar_gattuso(kd490_points)
    benthic_dli_timeseries = par_points * np.exp(-kdpar_points * depths)
    results = {}; mean_dli = benthic_dli_timeseries.mean('time', skipna=True); results['mean_DLI_local'] = mean_dli.values
    if cv_selector == 'all':
        std_dli = benthic_dli_timeseries.std('time', skipna=True)
        results['cv_DLI_local'] = (std_dli / (mean_dli + 1e-9) * 100).values
        results['cv_DLI_local_segment_count'] = benthic_dli_timeseries.notnull().sum('time').values
    else:
        window = int(cv_selector); min_p = max(2, int(window * 0.25))
        rolling_mean = benthic_dli_timeseries.rolling(time=window, min_periods=min_p, center=True).mean()
        rolling_std = benthic_dli_timeseries.rolling(time=window, min_periods=min_p, center=True).std()
        cv_ts = (rolling_std / (rolling_mean + 1e-9)) * 100
        results[f'dli_cv_{window}'] = cv_ts.mean('time', skipna=True).values
        results[f'dli_cv_{window}_segment_count'] = cv_ts.notnull().sum('time').values
    return results

# ==============================
# Processamento Principal
# ==============================
print("\n--- 1. Carregando Dados de Satélite ---")
sst_all_ds = load_satellite_data(sst_pattern, sst_dir, sst_period, ['analysed_sst', 'sea_surface_temperature', 'sst'])
dhw_all_ds = load_satellite_data(dhw_pattern, dhw_dir, dhw_period, ['degree_heating_week'])
chl_all_ds = load_satellite_data(chl_pattern, chl_dir, chl_period, ['chlor_a'])
kd490_all_ds = load_satellite_data(kd490_pattern, modis_dir, light_period, ['Kd_490'])
par_all_ds = load_satellite_data(par_pattern, modis_dir, light_period, ['par'])

print("\n--- 2. Calculando Métricas para os Sítios ---")
df_sites_local = df_sites_info.copy()

# DLI
dli_metrics = calculate_site_specific_dli(kd490_all_ds, par_all_ds, df_sites_info, analysis_selector)
for key, values in dli_metrics.items(): df_sites_local[key] = values
del kd490_all_ds, par_all_ds, dli_metrics; gc.collect()

# SST
sst_metrics_to_calculate = ['mean']; sst_metrics_to_calculate.append('cv_all' if str(analysis_selector) == 'all' else f'cv_{analysis_selector}')
sst_metrics = extract_and_compute_site_metrics(sst_all_ds, sites, sst_metrics_to_calculate)
for key, values in sst_metrics.items(): df_sites_local[f'sst_{key}'] = values
del sst_all_ds, sst_metrics; gc.collect()

# DHW
dhw_metrics = extract_and_compute_site_metrics(dhw_all_ds, sites, ['prop_gt_4', 'prop_gt_8'])
if 'prop_gt_4' in dhw_metrics:
    df_sites_local['prop_DHW_gt4'] = dhw_metrics['prop_gt_4']
if 'prop_gt_8' in dhw_metrics:
    df_sites_local['prop_DHW_gt8'] = dhw_metrics['prop_gt_8']
del dhw_all_ds, dhw_metrics; gc.collect()

# CHL
chl_metrics_to_calculate = ['mean']; chl_metrics_to_calculate.append('cv_all' if str(analysis_selector) == 'all' else f'cv_{analysis_selector}')
chl_metrics = extract_and_compute_site_metrics(chl_all_ds, sites, chl_metrics_to_calculate)
for key, values in chl_metrics.items(): df_sites_local[f'chl_{key}'] = values
del chl_all_ds, chl_metrics; gc.collect()

# =============================================================================
# --- SEÇÃO 3: FUNÇÃO GERAL PARA PCA E PLOTAGEM (MODIFICADA) ---
# =============================================================================
def run_pca_and_generate_bubble_plots(df_input, pca_vars_list, output_suffix):
    """
    Executa uma análise PCA, gera os BUBBLE PLOTS individuais e RETORNA
    os dados necessários para a figura composta (scores, loadings, etc.).
    """
    print("\n" + "="*50)
    print(f"INICIANDO PCA: {output_suffix.replace('_', ' ').strip().title()}")
    print("="*50)

    existing_vars = [var for var in pca_vars_list if var in df_input.columns]
    if len(existing_vars) < 2:
        print(f"AVISO: PCA para '{output_suffix}' pulada. Variáveis encontradas: {existing_vars}")
        return df_input, None # Retorna None para indicar falha

    print(f"Variáveis usadas na PCA: {existing_vars}")
    df_subset = df_input.dropna(subset=existing_vars) # Remove linhas com NaNs antes da PCA
    if len(df_subset) < 2:
        print(f"AVISO: PCA para '{output_suffix}' pulada. Pontos de dados insuficientes após remover NaNs.")
        return df_input, None
    
    data_for_pca = df_subset[existing_vars].copy()
    transformed_names = list(data_for_pca.columns)
    
    # Transformações
    for i, col_name in enumerate(data_for_pca.columns):
        if col_name == 'prop_DHW_gt4':
            data_for_pca[col_name] = np.sqrt(data_for_pca[col_name] + 0.5)
            transformed_names[i] = "sqrt(DHW>4)"
        elif col_name in ['chl_mean', 'sst_mean', 'mean_DLI_local']:
            data_for_pca[col_name] = np.log1p(data_for_pca[col_name])
            transformed_names[i] = f"log({col_name}+1)"

    # PCA
    scaler = StandardScaler()
    data_matrix = scaler.fit_transform(data_for_pca)
    pca = PCA(n_components=2)
    scores = pca.fit_transform(data_matrix)
    explained_variance = pca.explained_variance_ratio_ * 100
    
    # Salva loadings
    loadings_df = pd.DataFrame(pca.components_.T, columns=["PC1", "PC2"], index=transformed_names)
    loadings_path = os.path.join(output_dir, f"loadings_PCA{output_suffix}.csv")
    loadings_df.to_csv(loadings_path)
    print(f"Loadings salvos em {loadings_path}")

    # Adiciona scores ao DataFrame original (usando o índice do df_subset)
    df_scores = df_input.copy()
    df_scores.loc[df_subset.index, f'PC1{output_suffix}'] = scores[:, 0]
    df_scores.loc[df_subset.index, f'PC2{output_suffix}'] = scores[:, 1]
    
    # --- Geração dos BUBBLE PLOTS (individualmente) ---
    print(f"\n--- Gerando Bubble Plots para PCA {output_suffix} ---")
    unique_reefs = df_scores['Reef_name'].unique()
    cmap = plt.get_cmap('tab10')
    color_map = {r: cmap(i % 10) for i, r in enumerate(unique_reefs)}
    unique_habitats = df_scores['HAB'].unique()
    habitat_shapes = ['o', 's', '^', 'D', 'v', '<', '>']
    shape_map = {hab: habitat_shapes[i % len(habitat_shapes)] for i, hab in enumerate(unique_habitats)}
    arch_styles = {'inner': {'edgecolor': 'black', 'linewidth': 2.0}, 'outer': {'edgecolor': 'darkgrey', 'linewidth': 0.75}}

    def scale_marker_sizes(values, scale_factor=300, min_size=30):
        # ... (código da função scale_marker_sizes)
        vals = np.asarray(values, dtype=float); valid = np.isfinite(vals)
        if valid.sum() == 0: return np.full(vals.shape, min_size)
        vmin, vmax = vals[valid].min(), vals[valid].max()
        if vmax == vmin: return np.full(vals.shape, min_size + scale_factor / 2)
        scaled = (vals - vmin) / (vmax - vmin); scaled[~valid] = 0
        return scaled * scale_factor + min_size
        
    for var_name in existing_vars:
        site_values = df_scores[var_name].values
        if np.all(np.isnan(site_values)): continue
        sizes = scale_marker_sizes(site_values)
        fig, ax = plt.subplots(figsize=(12, 8))
        for i, row in df_scores.iterrows():
            if pd.notna(row[f'PC1{output_suffix}']):
                style = arch_styles.get(row['Arch'].lower(), {'edgecolor': 'grey', 'linewidth': 0.5})
                ax.scatter(row[f'PC1{output_suffix}'], row[f'PC2{output_suffix}'], s=sizes[i],
                           color=color_map.get(row['Reef_name'], 'grey'), marker=shape_map.get(row['HAB'], 'x'),
                           alpha=0.7, **style)
        # ... (código para adicionar legendas ao bubble plot, omitido para brevidade)
        ax.set_title(f"PCA {output_suffix.replace('_', ' ')} - Tamanho ∝ {var_name}")
        # ... (código para salvar o bubble plot)
        plt.close(fig)
    
    # Prepara o dicionário de resultados para a plotagem composta
    pca_results = {
        'df_scores': df_scores,
        'loadings': loadings_df,
        'explained_variance': explained_variance,
        'pc1_col': f'PC1{output_suffix}',
        'pc2_col': f'PC2{output_suffix}',
        'title_suffix': output_suffix.replace('_', ' ').strip()
    }
    
    return df_scores, pca_results

# =============================================================================
# --- SEÇÃO 4: EXECUÇÃO DAS DUAS ANÁLISES PCA ---
# =============================================================================
df_processed = df_sites_local.copy()

# Define as variáveis de CV com base no seletor
if str(analysis_selector).lower() == 'all':
    cv_sst_col, cv_dli_col, cv_chl_col = 'sst_cv_all', 'cv_DLI_local', 'chl_cv_all'
else:
    cv_sst_col = f'sst_cv_{analysis_selector}'
    cv_dli_col = f'dli_cv_{analysis_selector}'
    cv_chl_col = f'chl_cv_{analysis_selector}'

# Executa PCA de Variabilidade
pca_vars_variability = [cv_sst_col, cv_dli_col, cv_chl_col]
df_processed, variability_results = run_pca_and_generate_bubble_plots(df_processed, pca_vars_variability, "_Variability")

# Executa PCA de Magnitude
pca_vars_magnitude = ['sst_mean', 'mean_DLI_local', 'chl_mean', 'prop_DHW_gt4']
df_processed, magnitude_results = run_pca_and_generate_bubble_plots(df_processed, pca_vars_magnitude, "_Magnitude")

# Salva o arquivo de dados consolidado
final_scores_path = os.path.join(output_dir, "dados_consolidados_com_scores_das_duas_PCAs.xlsx")
df_processed.to_excel(final_scores_path, index=False)
print(f"\nArquivo final com todas as métricas e scores salvo em: {final_scores_path}")

# =============================================================================
# --- SEÇÃO 5: GERAÇÃO DE TODAS AS FIGURAS COMPOSTAS (COM LEGENDAS) ---
# =============================================================================

# --- Função Auxiliar 1: Para a figura de Ordenação + Loadings (COM LEGENDAS) ---
def create_composite_pca_figure(pca_results, output_filename):
    """
    Cria uma figura composta (1 linha, 3 colunas) para os resultados de uma PCA.
    Col 1: Ordenação com legendas | Col 2: Loadings | Col 3: Scree Plot
    """
    if pca_results is None:
        print(f"Não há resultados de PCA para gerar a figura {output_filename}. Pulando.")
        return

    df_scores = pca_results['df_scores']
    loadings = pca_results['loadings']
    explained_variance = pca_results['explained_variance']
    pc1_col, pc2_col = pca_results['pc1_col'], pca_results['pc2_col']
    title_suffix = pca_results['title_suffix']
    
    fig, axes = plt.subplots(1, 3, figsize=(24, 7), gridspec_kw={'width_ratios': [1.2, 1, 0.8]})
    fig.suptitle(f'Resumo da Análise de Componentes Principais - {title_suffix}', fontsize=20, y=1.02)

    # --- Coluna 1: Gráfico de Ordenação ---
    ax1 = axes[0]
    unique_reefs = df_scores['Reef_name'].unique(); cmap = plt.get_cmap('tab10'); color_map = {r: cmap(i % 10) for i, r in enumerate(unique_reefs)}
    unique_habitats = df_scores['HAB'].unique(); habitat_shapes = ['o', 's', '^', 'D', 'v', '<', '>']; shape_map = {hab: habitat_shapes[i % len(habitat_shapes)] for i, hab in enumerate(unique_habitats)}
    arch_styles = {'inner': {'edgecolor': 'black', 'linewidth': 2.0}, 'outer': {'edgecolor': 'darkgrey', 'linewidth': 0.75}}

    for _, row in df_scores.iterrows():
        if pd.notna(row[pc1_col]):
            style = arch_styles.get(row['Arch'].lower(), {'edgecolor': 'grey', 'linewidth': 0.5})
            ax1.scatter(row[pc1_col], row[pc2_col], color=color_map.get(row['Reef_name']), marker=shape_map.get(row['HAB']), s=100, alpha=0.8, **style)

    ax1.set_xlabel(f"PC1 ({explained_variance[0]:.1f}%)", fontsize=14)
    ax1.set_ylabel(f"PC2 ({explained_variance[1]:.1f}%)", fontsize=14)
    ax1.set_title("Ordenação dos Sítios", fontsize=16)
    ax1.grid(True, linestyle='--', alpha=0.6)
    ax1.axhline(0, color='grey', lw=0.5); ax1.axvline(0, color='grey', lw=0.5)

    # <--- INÍCIO DA ADIÇÃO DAS LEGENDAS --->
    # Cria os elementos para cada legenda
    legend_elements_color = [plt.Line2D([0], [0], marker='o', color='w', label=reef, markersize=10, markerfacecolor=color_map[reef]) for reef in unique_reefs]
    legend_elements_shape = [plt.Line2D([0], [0], marker=shape_map[hab], color='grey', label=hab, linestyle='None', markersize=10) for hab in unique_habitats]
    legend_elements_arch = [
        plt.Line2D([0], [0], marker='o', color='w', label='Inner Arc', markersize=10, markeredgecolor=arch_styles['inner']['edgecolor'], markeredgewidth=arch_styles['inner']['linewidth']),
        plt.Line2D([0], [0], marker='o', color='w', label='Outer Arc', markersize=10, markeredgecolor=arch_styles['outer']['edgecolor'], markeredgewidth=arch_styles['outer']['linewidth'])
    ]
    # Adiciona as legendas à FIGURA, não ao eixo, para posicionamento global
    leg1 = fig.legend(title="Recife", handles=legend_elements_color, loc='center left', bbox_to_anchor=(0.91, 0.75))
    leg2 = fig.legend(title="Habitat", handles=legend_elements_shape, loc='center left', bbox_to_anchor=(0.91, 0.5))
    leg3 = fig.legend(title="Arco", handles=legend_elements_arch, loc='center left', bbox_to_anchor=(0.91, 0.25))
    # <--- FIM DA ADIÇÃO DAS LEGENDAS --->

    # --- Coluna 2: Gráfico de Loadings ---
    ax2 = axes[1]
    # ... (código do gráfico de loadings, sem alterações)
    ax2.axhline(0, color='grey', lw=0.5); ax2.axvline(0, color='grey', lw=0.5)
    for i, var in enumerate(loadings.index):
        ax2.arrow(0, 0, loadings['PC1'][i]*2, loadings['PC2'][i]*2, head_width=0.05, head_length=0.1, fc='red', ec='red')
        ax2.text(loadings['PC1'][i]*2.2, loadings['PC2'][i]*2.2, var, color='black', ha='center', va='center', fontsize=12)
    ax2.set_xlim(-2.5, 2.5); ax2.set_ylim(-2.5, 2.5)
    ax2.set_xlabel("Contribuição para PC1", fontsize=14)
    ax2.set_ylabel("Contribuição para PC2", fontsize=14)
    ax2.set_title("Loadings das Variáveis", fontsize=16)
    ax2.set_aspect('equal', adjustable='box')

    # --- Coluna 3: Scree Plot ---
    ax3 = axes[2]
    # ... (código do scree plot, sem alterações)
    components = ['PC1', 'PC2']
    ax3.bar(components, explained_variance, color='skyblue', edgecolor='black')
    ax3.set_ylabel("Variância Explicada (%)", fontsize=14)
    ax3.set_title("Importância dos Componentes", fontsize=16)
    ax3.set_ylim(0, 100)
    for i, v in enumerate(explained_variance):
        ax3.text(i, v + 2, f"{v:.1f}%", ha='center', color='black', fontsize=12)

    # Ajusta o layout para criar espaço para as legendas à direita
    fig.subplots_adjust(right=0.9)
    plt.savefig(output_filename, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"Figura composta de resumo salva em: {output_filename}")


# --- Função Auxiliar 2: Para a figura de Bubble Plots (COM LEGENDAS) ---
def create_composite_bubble_plot_figure(pca_results, pca_vars, output_filename, nrows, ncols):
    """
    Cria uma figura composta com os bubble plots das variáveis da PCA em um grid customizado.
    """
    if pca_results is None:
        print(f"Não há resultados de PCA para gerar a figura {output_filename}. Pulando.")
        return

    df_scores = pca_results['df_scores']
    explained_variance = pca_results['explained_variance']
    pc1_col, pc2_col = pca_results['pc1_col'], pca_results['pc2_col']
    title_suffix = pca_results['title_suffix']
    
    # Ajusta o tamanho da figura para ter espaço extra para a legenda
    fig_width = (6 * ncols) + 3 # 6 por subplot + 3 de espaço para a legenda
    fig_height = 6 * nrows
    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_width, fig_height), sharex=True, sharey=True)
    
    if nrows == 1 and ncols == 1: axes = np.array([[axes]])
    elif nrows == 1 or ncols == 1: axes = np.array([axes]).flatten().reshape(nrows, ncols)

    fig.suptitle(f'Bubble Plots - PCA de {title_suffix}', fontsize=20, y=0.98)

    # Definições de estilo e legenda
    unique_reefs = df_scores['Reef_name'].unique(); cmap = plt.get_cmap('tab10'); color_map = {r: cmap(i % 10) for i, r in enumerate(unique_reefs)}
    unique_habitats = df_scores['HAB'].unique(); habitat_shapes = ['o', 's', '^', 'D', 'v', '<', '>']; shape_map = {hab: habitat_shapes[i % len(habitat_shapes)] for i, hab in enumerate(unique_habitats)}
    arch_styles = {'inner': {'edgecolor': 'black', 'linewidth': 2.0}, 'outer': {'edgecolor': 'darkgrey', 'linewidth': 0.75}}

    def scale_marker_sizes(values, scale_factor=300, min_size=30):
        # ... (código da função scale_marker_sizes, sem alterações)
        vals = np.asarray(values, dtype=float); valid = np.isfinite(vals)
        if valid.sum() == 0: return np.full(vals.shape, min_size)
        vmin, vmax = vals[valid].min(), vals[valid].max()
        if vmax == vmin: return np.full(vals.shape, min_size + scale_factor / 2)
        scaled = (vals - vmin) / (vmax - vmin); scaled[~valid] = 0
        return scaled * scale_factor + min_size
        
    for i, var_name in enumerate(pca_vars):
        row_idx, col_idx = divmod(i, ncols)
        ax = axes[row_idx, col_idx]
        
        site_values = df_scores[var_name].values
        sizes = scale_marker_sizes(site_values)
        for _, row in df_scores.iterrows():
            if pd.notna(row[pc1_col]):
                style = arch_styles.get(row['Arch'].lower(), {'edgecolor': 'grey', 'linewidth': 0.5})
                ax.scatter(row[pc1_col], row[pc2_col], s=sizes[_],
                           color=color_map.get(row['Reef_name']), marker=shape_map.get(row['HAB']),
                           alpha=0.7, **style)

        ax.set_title(f"Tamanho ∝ {var_name}", fontsize=16)
        ax.grid(True, linestyle='--', alpha=0.6)
        ax.axhline(0, color='grey', lw=0.5); ax.axvline(0, color='grey', lw=0.5)

    for j in range(len(pca_vars), nrows * ncols):
        row_idx, col_idx = divmod(j, ncols)
        axes[row_idx, col_idx].set_visible(False)

    # Rótulos dos eixos principais
    fig.text(0.5, 0.04, f"PC1 ({explained_variance[0]:.1f}%)", ha='center', va='center', fontsize=14)
    fig.text(0.04, 0.5, f"PC2 ({explained_variance[1]:.1f}%)", ha='center', va='center', rotation='vertical', fontsize=14)
    
    # <--- ADIÇÃO DAS LEGENDAS (mesma lógica da outra função) --->
    legend_elements_color = [plt.Line2D([0], [0], marker='o', color='w', label=reef, markersize=10, markerfacecolor=color_map[reef]) for reef in unique_reefs]
    legend_elements_shape = [plt.Line2D([0], [0], marker=shape_map[hab], color='grey', label=hab, linestyle='None', markersize=10) for hab in unique_habitats]
    legend_elements_arch = [
        plt.Line2D([0], [0], marker='o', color='w', label='Inner Arc', markersize=10, markeredgecolor=arch_styles['inner']['edgecolor'], markeredgewidth=arch_styles['inner']['linewidth']),
        plt.Line2D([0], [0], marker='o', color='w', label='Outer Arc', markersize=10, markeredgecolor=arch_styles['outer']['edgecolor'], markeredgewidth=arch_styles['outer']['linewidth'])
    ]
    # Calcula a posição da legenda com base no número de colunas
    legend_x_pos = 1 - (1.5 / fig_width)
    leg1 = fig.legend(title="Recife", handles=legend_elements_color, loc='center left', bbox_to_anchor=(legend_x_pos, 0.75))
    leg2 = fig.legend(title="Habitat", handles=legend_elements_shape, loc='center left', bbox_to_anchor=(legend_x_pos, 0.5))
    leg3 = fig.legend(title="Arco", handles=legend_elements_arch, loc='center left', bbox_to_anchor=(legend_x_pos, 0.25))

    # Ajusta o layout para criar espaço para a legenda
    fig.subplots_adjust(right=1 - (3.0 / fig_width))
    plt.savefig(output_filename, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"Figura composta de bubble plots salva em: {output_filename}")


# --- Gerar TODAS as figuras finais ---
print("\n--- Gerando Figuras Compostas Finais ---")

# Figuras de Resumo (Ordenação + Loadings)
create_composite_pca_figure(magnitude_results, os.path.join(output_dir, "figura_composta_PCA_Magnitude.png"))
create_composite_pca_figure(variability_results, os.path.join(output_dir, "figura_composta_PCA_Variability.png"))

# Figuras de Bubble Plots - com layout customizado
create_composite_bubble_plot_figure(
    pca_results=magnitude_results, 
    pca_vars=pca_vars_magnitude, 
    output_filename=os.path.join(output_dir, "figura_composta_bubbles_Magnitude.png"),
    nrows=2, 
    ncols=2
)
create_composite_bubble_plot_figure(
    pca_results=variability_results, 
    pca_vars=pca_vars_variability, 
    output_filename=os.path.join(output_dir, "figura_composta_bubbles_Variability.png"),
    nrows=3, 
    ncols=1
)


# --- SEÇÃO 6: Salvar Resumo de Segmentos ---
print("\n--- Gerando arquivo de resumo de segmentos (janelas válidas) ---")
# ... (código desta seção permanece o mesmo)
segment_cols = [col for col in df_processed.columns if 'segment_count' in col]
summary_cols = ['Site_name', 'HAB', 'Arch'] + segment_cols
if segment_cols:
    df_segments_summary = df_processed[summary_cols]
    summary_path = os.path.join(output_dir, "resumo_segmentos_por_site.xlsx")
    df_segments_summary.to_excel(summary_path, index=False)
    print(f"Resumo de segmentos por site salvo em: {summary_path}")
else:
    print("Nenhuma coluna de contagem de segmentos encontrada para criar o resumo.")

print("\n\nProcesso completo finalizado com sucesso!")

In [ ]:
# Roda PERMANOVA, ANOVA e Levene para REEF e HAB
# PCA LOCAL

# =============================================================================
# ANÁLISE ESTATÍSTICA (PERMANOVA/ANOVA) DOS SCORES DA PCA LOCAL
# Versão 2.0 - Adaptado para ler a saída da PCA Local e analisar
#              separadamente a PCA de Magnitude e a de Variabilidade.
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from skbio.stats.distance import DistanceMatrix
from skbio.stats.distance import permanova
from scipy.spatial.distance import pdist, squareform
from scipy.stats import f_oneway, levene

# === CAMINHOS E CONFIGURAÇÕES ===
# Aponta para o arquivo de saída padrão do script PCA LOCAL
input_file = r"C:\Users\rbfra\OneDrive\########PUBLICACOES\############Menezes et al. Mus his distribution and abundance Abrolhos\########NEW RESULTS\#####output_local_PCA_CV_2_FINAL\dados_consolidados_com_scores_das_duas_PCAs.xlsx"

# Diretório principal para os resultados desta análise
base_output_dir = r"C:\Users\rbfra\OneDrive\########PUBLICACOES\############Menezes et al. Mus his distribution and abundance Abrolhos\########NEW RESULTS\PERMANOVA_ANOVA_LOCAL_PCA_CV_2"
os.makedirs(base_output_dir, exist_ok=True)


# === FUNÇÃO PARA RODAR A ANÁLISE COMPLETA ===
def run_full_analysis(df, pc1_col, pc2_col, analysis_name, output_dir):
    """
    Executa PERMANOVA, ANOVA, Levene e gera gráficos para um conjunto de scores de PCA.
    
    Args:
        df (pd.DataFrame): DataFrame com os dados.
        pc1_col (str): Nome da coluna do PC1.
        pc2_col (str): Nome da coluna do PC2.
        analysis_name (str): Nome da análise (ex: "Magnitude", "Variability").
        output_dir (str): Diretório para salvar os resultados.
    """
    os.makedirs(output_dir, exist_ok=True)
    print("\n" + "="*60)
    print(f"  INICIANDO ANÁLISE PARA: PCA {analysis_name.upper()}")
    print("="*60)

    # --- 1. Preparação dos Dados e Matriz de Distância ---
    # Colunas essenciais para esta análise específica
    required_cols = [pc1_col, pc2_col, 'Reef_name', 'Site_name', 'HAB']
    df_valid = df.dropna(subset=required_cols).copy()
    
    if len(df_valid) < 3:
        print(f"AVISO: Dados insuficientes ({len(df_valid)} linhas) para a análise {analysis_name}. Pulando.")
        return

    print(f"Dados válidos para {analysis_name}: {len(df_valid)} linhas.")
    
    data_matrix = df_valid[[pc1_col, pc2_col]].values
    dist_matrix = squareform(pdist(data_matrix, metric='euclidean'))
    # Garante que os IDs sejam únicos para a matriz de distância
    unique_ids = df_valid.reset_index().apply(lambda row: f"{row['Site_name']}_{row['HAB']}_{row['index']}", axis=1).tolist()
    dm = DistanceMatrix(dist_matrix, ids=unique_ids)

    # --- 2. Análise por REEF ---
    print(f"\n--- Analisando por REEF ({analysis_name}) ---")
    grouping_reef = df_valid.reset_index().set_index(pd.Index(unique_ids))['Reef_name']
    permanova_reef = permanova(distance_matrix=dm, grouping=grouping_reef, permutations=999)

    grouped_pc1_reef = [g[pc1_col].values for _, g in df_valid.groupby('Reef_name')]
    anova_pc1_reef = f_oneway(*grouped_pc1_reef)
    levene_pc1_reef = levene(*grouped_pc1_reef)

    grouped_pc2_reef = [g[pc2_col].values for _, g in df_valid.groupby('Reef_name')]
    anova_pc2_reef = f_oneway(*grouped_pc2_reef)
    levene_pc2_reef = levene(*grouped_pc2_reef)

    fig_reef, axes = plt.subplots(1, 2, figsize=(14, 6))
    sns.boxplot(ax=axes[0], x='Reef_name', y=pc1_col, data=df_valid, palette='viridis')
    axes[0].set_title(f'{pc1_col} por REEF\nANOVA p={anova_pc1_reef.pvalue:.4f} | Levene p={levene_pc1_reef.pvalue:.4f}')
    axes[0].tick_params(axis='x', rotation=45)
    sns.boxplot(ax=axes[1], x='Reef_name', y=pc2_col, data=df_valid, palette='viridis')
    axes[1].set_title(f'{pc2_col} por REEF\nANOVA p={anova_pc2_reef.pvalue:.4f} | Levene p={levene_pc2_reef.pvalue:.4f}')
    axes[1].tick_params(axis='x', rotation=45)
    fig_reef.suptitle(f"Análise por Recife - PCA de {analysis_name}", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(os.path.join(output_dir, f"boxplots_PCA_{analysis_name}_vs_REEF.png"), dpi=300)
    plt.close()

    # --- 3. Análise por HABITAT ---
    print(f"\n--- Analisando por HAB ({analysis_name}) ---")
    grouping_hab = df_valid.reset_index().set_index(pd.Index(unique_ids))['HAB']
    permanova_hab = permanova(distance_matrix=dm, grouping=grouping_hab, permutations=999)
    
    grouped_pc1_hab = [g[pc1_col].values for _, g in df_valid.groupby('HAB')]
    anova_pc1_hab = f_oneway(*grouped_pc1_hab)
    levene_pc1_hab = levene(*grouped_pc1_hab)

    grouped_pc2_hab = [g[pc2_col].values for _, g in df_valid.groupby('HAB')]
    anova_pc2_hab = f_oneway(*grouped_pc2_hab)
    levene_pc2_hab = levene(*grouped_pc2_hab)
    
    fig_hab, axes = plt.subplots(1, 2, figsize=(12, 5))
    sns.boxplot(ax=axes[0], x='HAB', y=pc1_col, data=df_valid, palette='pastel')
    axes[0].set_title(f'{pc1_col} por HABITAT\nANOVA p={anova_pc1_hab.pvalue:.4f} | Levene p={levene_pc1_hab.pvalue:.4f}')
    sns.boxplot(ax=axes[1], x='HAB', y=pc2_col, data=df_valid, palette='pastel')
    axes[1].set_title(f'{pc2_col} por HABITAT\nANOVA p={anova_pc2_hab.pvalue:.4f} | Levene p={levene_pc2_hab.pvalue:.4f}')
    fig_hab.suptitle(f"Análise por Habitat - PCA de {analysis_name}", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(os.path.join(output_dir, f"boxplots_PCA_{analysis_name}_vs_HAB.png"), dpi=300)
    plt.close()

    # --- 4. Salvar Resultados Numéricos ---
    output_txt = os.path.join(output_dir, f"resultados_estatisticos_{analysis_name}.txt")
    with open(output_txt, 'w') as f:
        f.write("="*50 + "\n")
        f.write(f"RESULTADOS ESTATÍSTICOS - PCA de {analysis_name.upper()}\n")
        f.write("="*50 + "\n\n")

        f.write("PERMANOVA - ESTRUTURAÇÃO MULTIVARIADA\n")
        f.write("-" * 50 + "\n")
        f.write("PERMANOVA baseada em REEF:\n")
        f.write(str(permanova_reef) + "\n\n")
        f.write("PERMANOVA baseada em HAB:\n")
        f.write(str(permanova_hab) + "\n\n")

        f.write("ANOVA/LEVENE - ESTRUTURAÇÃO UNIVARIADA POR EIXO\n")
        f.write("-" * 50 + "\n")
        f.write("Resultados para REEF:\n")
        f.write(f"  {pc1_col} vs REEF: ANOVA F={anova_pc1_reef.statistic:.3f}, p={anova_pc1_reef.pvalue:.4f} | Levene W={levene_pc1_reef.statistic:.3f}, p={levene_pc1_reef.pvalue:.4f}\n")
        f.write(f"  {pc2_col} vs REEF: ANOVA F={anova_pc2_reef.statistic:.3f}, p={anova_pc2_reef.pvalue:.4f} | Levene W={levene_pc2_reef.statistic:.3f}, p={levene_pc2_reef.pvalue:.4f}\n\n")
        f.write("Resultados para HAB:\n")
        f.write(f"  {pc1_col} vs HAB: ANOVA F={anova_pc1_hab.statistic:.3f}, p={anova_pc1_hab.pvalue:.4f} | Levene W={levene_pc1_hab.statistic:.3f}, p={levene_pc1_hab.pvalue:.4f}\n")
        f.write(f"  {pc2_col} vs HAB: ANOVA F={anova_pc2_hab.statistic:.3f}, p={anova_pc2_hab.pvalue:.4f} | Levene W={levene_pc2_hab.statistic:.3f}, p={levene_pc2_hab.pvalue:.4f}\n\n")
    
    print(f"Análise para PCA de {analysis_name} concluída. Resultados em: {output_dir}")


# === SCRIPT PRINCIPAL ===

# --- Carrega e prepara os dados UMA VEZ ---
try:
    df_full = pd.read_excel(input_file)
    print(f"Arquivo '{os.path.basename(input_file)}' carregado com sucesso.")
except FileNotFoundError:
    print(f"ERRO: O arquivo de entrada não foi encontrado em: {input_file}")
    exit()

# --- ANÁLISE 1: PCA DE MAGNITUDE ---
# Define as colunas de score para a PCA de Magnitude
pc1_mag_col = 'PC1_Magnitude'
pc2_mag_col = 'PC2_Magnitude'
output_mag_dir = os.path.join(base_output_dir, "analise_magnitude")

if pc1_mag_col in df_full.columns and pc2_mag_col in df_full.columns:
    run_full_analysis(df_full, pc1_mag_col, pc2_mag_col, "Magnitude", output_mag_dir)
else:
    print(f"AVISO: Colunas para PCA de Magnitude ('{pc1_mag_col}', '{pc2_mag_col}') não encontradas. Análise pulada.")


# <--- Continuação do código ---

# --- ANÁLISE 2: PCA DE VARIABILIDADE ---
# Define as colunas de score para a PCA de Variabilidade
pc1_var_col = 'PC1_Variability'
pc2_var_col = 'PC2_Variability'
output_var_dir = os.path.join(base_output_dir, "analise_variabilidade")

if pc1_var_col in df_full.columns and pc2_var_col in df_full.columns:
    run_full_analysis(df_full, pc1_var_col, pc2_var_col, "Variability", output_var_dir)
else:
    print(f"AVISO: Colunas para PCA de Variabilidade ('{pc1_var_col}', '{pc2_var_col}') não encontradas. Análise pulada.")


# --- ANÁLISE EXTRA: JUSTIFICATIVA DO HABITAT (opcional, mas útil) ---
print("\n" + "="*60)
print("  ANÁLISE EXTRA: Justificativa do Fator HABITAT")
print("="*60)

output_justificativa_dir = os.path.join(base_output_dir, "justificativa_habitat")
os.makedirs(output_justificativa_dir, exist_ok=True)

# Usa o DataFrame completo carregado no início
df_valid_justificativa = df_full.dropna(subset=['Depth_m', 'mean_DLI_local', 'HAB']).copy()

if len(df_valid_justificativa) > 3 and df_valid_justificativa['HAB'].nunique() > 1:
    # ANOVA para Depth vs HAB
    grouped_depth_hab = [g['Depth_m'].values for _, g in df_valid_justificativa.groupby('HAB')]
    anova_depth_hab = f_oneway(*grouped_depth_hab)

    # ANOVA para DLI vs HAB
    grouped_dli_hab = [g['mean_DLI_local'].values for _, g in df_valid_justificativa.groupby('HAB')]
    anova_dli_hab = f_oneway(*grouped_dli_hab)
    
    # Boxplots
    fig_direct, axes = plt.subplots(1, 2, figsize=(12, 5))
    sns.boxplot(ax=axes[0], x='HAB', y='Depth_m', data=df_valid_justificativa, palette='mako')
    axes[0].set_title(f'Profundidade por HABITAT\nANOVA p={anova_depth_hab.pvalue:.4f}')
    axes[0].set_ylabel('Profundidade (m)')
    
    sns.boxplot(ax=axes[1], x='HAB', y='mean_DLI_local', data=df_valid_justificativa, palette='rocket')
    axes[1].set_title(f'DLI Médio por HABITAT\nANOVA p={anova_dli_hab.pvalue:.4f}')
    axes[1].set_ylabel('DLI Médio (mol/m²/dia)')
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_justificativa_dir, "boxplots_Fisico_vs_HAB.png"), dpi=300)
    plt.close()

    # Salvar resultados numéricos da justificativa
    output_txt_just = os.path.join(output_justificativa_dir, "resultados_justificativa_habitat.txt")
    with open(output_txt_just, 'w') as f:
        f.write("="*50 + "\n")
        f.write("ANÁLISE DO LINK DIRETO ENTRE FATORES FÍSICOS E HABITAT\n")
        f.write("="*50 + "\n\n")
        f.write(f"  Profundidade (Depth_m) vs HAB: ANOVA F={anova_depth_hab.statistic:.3f}, p={anova_depth_hab.pvalue:.4f}\n")
        f.write(f"  DLI Médio (mean_DLI_local) vs HAB: ANOVA F={anova_dli_hab.statistic:.3f}, p={anova_dli_hab.pvalue:.4f}\n")
    print(f"Análise de justificativa do habitat concluída. Resultados em: {output_justificativa_dir}")
else:
    print("Dados insuficientes ou apenas um tipo de habitat para a análise de justificativa.")


print("\nAnálise estatística completa finalizada!")

In [9]:
### ANÁLISE LOCAL DE ESTRESSE AMBIENTAL PARA CORAIS - VERSÃO 4.1 (com Análise de Sensibilidade) ###

# OBJETIVOS DO SCRIPT:
# 1. Calcular um conjunto completo de métricas de estresse (magnitude, amplitude, frequência, sinergia) para cada sítio/habitat.
# 2. Executar uma PCA sobre as métricas de estresse para determinar os principais eixos de variação ambiental.
# 3. Executar uma Análise de Sensibilidade, testando diferentes cenários de pesos ecológicos para a construção do índice.
# 4. Para cada cenário:
#    a. Analisar programaticamente os eixos da PCA para identificar e orientar os gradientes de estresse.
#    b. Construir um Índice de Estresse Bruto a partir da combinação ponderada dos scores da PCA.
#    c. Calcular um Índice de Estresse Líquido incorporando um fator de mitigação.
# 5. Salvar todos os índices de estresse calculados em uma única planilha para modelagem estatística posterior.
# 6. Gerar visualizações profissionais:
#    a. Um resumo detalhado da PCA (ordenação, loadings).
#    b. Gráficos comparativos para avaliar a sensibilidade dos resultados aos diferentes cenários.

# ==============================
# Imports
# ==============================
import os
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
os.environ['OMP_NUM_THREADS'] = '1'
import gc
import logging
import re
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import dask
from astropy.timeseries import LombScargle
import glob

# ==============================
# Configurações Iniciais e Caminhos
# ==============================
logging.basicConfig(filename='pca_local_errors.log', filemode='w', level=logging.ERROR, format='%(asctime)s - %(levelname)s - %(message)s')
output_dir = r"C:\Users\rbfra\OneDrive\########PUBLICACOES\###########Global coral refugia and conservation priorities\Local_StressIndex_v4_Sensitivity"
os.makedirs(output_dir, exist_ok=True)
dask.config.set({'array.slicing.split_large_chunks': True, 'scheduler': 'single-threaded', 'temporary_directory': os.path.join(output_dir, 'dask_temp')})
os.makedirs(os.path.join(output_dir, 'dask_temp'), exist_ok=True)

# ==============================
# Parâmetros da Análise
# ==============================
sst_dir, modis_dir, chl_dir = r"H:\remote sensing\CRW_SST_FULL", r"H:\remote sensing\MODIS_DATA_FULL", r"H:\remote sensing\MODIS_DATA_FULL"
# Período de análise mais longo para garantir robustez das métricas de luz
sst_period, light_period, chl_period = (2002, 2008), (2002, 2008), (2002, 2008) 
sites_csv_file = r"C:\Users\rbfra\OneDrive\########CEBIMAR\####PROJETOS\#####Coral trade offs\sites_list_full.csv"
lat_min, lat_max, lon_min, lon_max = -20.5, -14.5, -40.5, -35.5
sst_pattern, kd490_pattern, par_pattern, chl_pattern = 'coraltemp_v3.1_*.nc', 'AQUA_MODIS.*.L3m.DAY.KD.Kd_490.4km.nc', 'AQUA_MODIS.*.L3m.DAY.PAR.par.4km.nc', 'AQUA_MODIS.*.L3m.DAY.CHL.chlor_a.4km.nc'
KDPAR_GATTUSO_A, KDPAR_GATTUSO_B, KDPAR_GATTUSO_C = 0.0665, 0.874, 0.00121
KDPAR_KD490_MIN_THRESHOLD = 0.001

# ==============================
# Parâmetros do Framework de Estresse
# ==============================
STRESS_VARS_FOR_PCA = [
    'sst_magnitude', 'dli_magnitude', 'sst_amplitude', 'dli_amplitude',
    'sst_frequencia', 'dli_frequencia', 'sinergia_sst_dli'
]
MITIGATION_VAR = 'chl_magnitude'
BIPOLAR_THRESHOLD = 0.25 # Limiar para classificar um eixo da PCA como bipolar

# ==============================
# Carregamento da lista de sítios
# ==============================
try:
    df_sites_info = pd.read_csv(sites_csv_file, sep=';')
    df_sites_info.columns = [col.strip().replace(' ', '_') for col in df_sites_info.columns]
    for col in df_sites_info.select_dtypes(['object']).columns: df_sites_info[col] = df_sites_info[col].str.strip()
    df_sites_info['unique_id'] = df_sites_info['Site_name'] + '_' + df_sites_info['HAB']
    sites = list(zip(df_sites_info['Latitude'], df_sites_info['Longitude']))
    print("DataFrame de sítios carregado com sucesso.")
except Exception as e:
    logging.critical(f"FALHA CRÍTICA AO CARREGAR O ARQUIVO DE SÍTIOS: {e}"); raise

# ==============================
# Funções de Cálculo e Análise
# ==============================
def verify_file_integrity(filepath):
    if not os.path.exists(filepath) or os.path.getsize(filepath) == 0: return False
    try:
        with xr.open_dataset(filepath) as ds: _ = ds.attrs
        return True
    except Exception: return False
def load_satellite_data(pattern, base_dir, period, var_name_options, chunks={'time': 30, 'lat': 100, 'lon': 100}):
    full_search_path = os.path.join(base_dir, pattern); print(f"\n--- Carregando dados para: {pattern} em {period} ---")
    all_files = sorted(glob.glob(full_search_path));
    if not all_files: logging.error(f"Nenhum arquivo encontrado para '{pattern}' em {base_dir}"); return None
    candidate_paths = [f for f in all_files if (match := re.search(r'(\d{4})\d{4}', os.path.basename(f))) and period[0] <= int(match.group(1)) <= period[1]]
    if not candidate_paths: logging.warning(f"Nenhum arquivo encontrado para '{pattern}' no período {period}"); return None
    valid_paths = [f for f in candidate_paths if verify_file_integrity(f)]
    if not valid_paths: logging.error(f"Nenhum arquivo VÁLIDO encontrado para '{pattern}' no período {period}"); return None
    print(f"Encontrados {len(valid_paths)} arquivos válidos.")
    def preprocess_with_time(ds):
        var_name = next((v for v in var_name_options if v in ds.data_vars), None)
        if var_name is None: return xr.Dataset()
        ds_subset = ds[[var_name]].astype('float32')
        if 'time' not in ds_subset.coords:
            match = re.search(r'(\d{8})', os.path.basename(ds.encoding.get("source", "")))
            if match: dt = pd.to_datetime(match.group(1), format='%Y%m%d'); return ds_subset.expand_dims(time=[dt])
        return ds_subset
    try:
        ds_raw = xr.open_mfdataset(valid_paths, preprocess=preprocess_with_time, combine='by_coords', parallel=True, chunks=chunks, engine='netcdf4')
        if not ds_raw.data_vars: print(f"AVISO: Nenhum dado carregado para '{pattern}'."); return None
        lat_slice = slice(lat_max, lat_min) if ds_raw['lat'].values[0] > ds_raw['lat'].values[-1] else slice(lat_min, lat_max)
        ds_final = ds_raw.sel(lat=lat_slice, lon=slice(lon_min, lon_max))
        return ds_final.sortby('time')[list(ds_final.data_vars)[0]]
    except Exception as e: logging.critical(f"Falha ao concatenar arquivos para '{pattern}': {e}"); return None
def calculate_kdpar_gattuso(kd490_da):
    kd490_safe = xr.where((kd490_da.isnull()) | (kd490_da <= KDPAR_KD490_MIN_THRESHOLD), np.nan, kd490_da)
    epsilon = 1e-9; kdpar = KDPAR_GATTUSO_A + KDPAR_GATTUSO_B * kd490_safe - KDPAR_GATTUSO_C / (kd490_safe + epsilon)
    kdpar_final = xr.where((kdpar.isnull()) | (kdpar <= 0), np.nan, kdpar); kdpar_final.name = "kdpar"
    return kdpar_final
def calculate_advanced_site_metrics(site_ts):
    if site_ts is None: return {}
    results = {}
    results['magnitude'] = site_ts.mean('time', skipna=True).values
    baseline = site_ts.rolling(time=91, center=True, min_periods=20).mean()
    anomalies = site_ts - baseline
    results['amplitude'] = anomalies.std('time', skipna=True).values
    freq_results = []
    for i in range(site_ts.sizes['site']):
        y, t = site_ts.isel(site=i).values, site_ts.time.values
        t_days = (t - np.datetime64('1970-01-01')) / np.timedelta64(1, 'D')
        valid_mask = ~np.isnan(y)
        y_valid, t_valid = y[valid_mask], t_days[valid_mask]
        if len(y_valid) < 20 or np.std(y_valid) < 1e-6:
            freq_results.append(np.nan)
            continue
        min_freq, max_freq = 1.0 / 30, 1.0 / 2
        try:
            ls = LombScargle(t_valid, y_valid)
            freq, power = ls.autopower(minimum_frequency=min_freq, maximum_frequency=max_freq)
            freq_results.append(np.mean(power) if len(power) > 0 else np.nan)
        except Exception:
            freq_results.append(np.nan)
    results['frequencia'] = np.array(freq_results)
    return results
def calculate_synergy_metric(ts1, ts2):
    if ts1 is None or ts2 is None:
        num_sites = ts1.sizes['site'] if ts1 is not None else len(sites)
        return np.full(num_sites, np.nan)
    ts1_daily = ts1.copy(); ts1_daily['time'] = ts1.time.dt.floor('D')
    ts2_daily = ts2.copy(); ts2_daily['time'] = ts2.time.dt.floor('D')
    ts1_aligned, ts2_aligned = xr.align(ts1_daily, ts2_daily, join='inner')
    if ts1_aligned.sizes['time'] < 20:
        print(f"AVISO (Sinergia): Após o alinhamento, restaram apenas {ts1_aligned.sizes['time']} dias. Cálculo de sinergia pulado.")
        return np.full(ts1.sizes['site'], np.nan)
    thresh1 = ts1_aligned.quantile(0.90, dim='time', skipna=True)
    thresh2 = ts2_aligned.quantile(0.90, dim='time', skipna=True)
    is_stress1 = (ts1_aligned > thresh1); is_stress2 = (ts2_aligned > thresh2)
    joint_stress_days = (is_stress1 & is_stress2).sum(dim='time', skipna=True, dtype=np.float32)
    stress1_days = is_stress1.sum(dim='time', skipna=True, dtype=np.float32)
    synergy = xr.where(stress1_days > 0, joint_stress_days / stress1_days, 0)
    return synergy.values
def get_site_timeseries(da, sites_coords):
    if da is None: return None
    lats = xr.DataArray([s[0] for s in sites_coords], dims="site", name="lat_points")
    lons = xr.DataArray([s[1] for s in sites_coords], dims="site", name="lon_points")
    site_ts = da.sel(lat=lats, lon=lons, method='nearest')
    site_ts = site_ts.assign_coords({'site_lat': ('site', [s[0] for s in sites_coords]), 'site_lon': ('site', [s[1] for s in sites_coords])})
    return site_ts.load()


# ==============================
# --- BLOCO PRINCIPAL DE PROCESSAMENTO ---
# ==============================

# 1. --- CÁLCULO DE MÉTRICAS E ANÁLISE PCA (BASE) ---
print("\n--- 1. Carregando Dados e Calculando Métricas Locais ---")
sst_all_ds = load_satellite_data(sst_pattern, sst_dir, sst_period, ['analysed_sst', 'sea_surface_temperature', 'sst'])
chl_all_ds = load_satellite_data(chl_pattern, chl_dir, chl_period, ['chlor_a'])
kd490_all_ds = load_satellite_data(kd490_pattern, modis_dir, light_period, ['Kd_490'])
par_all_ds = load_satellite_data(par_pattern, modis_dir, light_period, ['par'])
df_metrics = df_sites_info.copy()

sst_ts = get_site_timeseries(sst_all_ds, sites); chl_ts = get_site_timeseries(chl_all_ds, sites)
del sst_all_ds, chl_all_ds; gc.collect()
if kd490_all_ds is not None and par_all_ds is not None:
    lats = xr.DataArray(df_sites_info['Latitude'].values, dims="site"); lons = xr.DataArray(df_sites_info['Longitude'].values, dims="site"); depths = xr.DataArray(df_sites_info['Depth_m'].values, dims="site")
    kd490_points = kd490_all_ds.sel(lat=lats, lon=lons, method='nearest').load(); par_points = par_all_ds.sel(lat=lats, lon=lons, method='nearest').load()
    kdpar_points = calculate_kdpar_gattuso(kd490_points)
    dli_ts = par_points * np.exp(-kdpar_points * depths)
    del kd490_all_ds, par_all_ds, kd490_points, par_points, kdpar_points; gc.collect()
else:
    dli_ts = None
print("   - Calculando métricas avançadas...")
for ts, prefix in [(sst_ts, 'sst'), (dli_ts, 'dli'), (chl_ts, 'chl')]:
    metrics_results = calculate_advanced_site_metrics(ts)
    for key, values in metrics_results.items(): df_metrics[f'{prefix}_{key}'] = values
df_metrics['sinergia_sst_dli'] = calculate_synergy_metric(sst_ts, dli_ts)
del sst_ts, dli_ts, chl_ts; gc.collect()

print("\n--- 2. Executando PCA sobre as Métricas de Estresse ---")
df_pca_ready = df_metrics.dropna(subset=STRESS_VARS_FOR_PCA)
if len(df_pca_ready) < len(STRESS_VARS_FOR_PCA):
    raise ValueError(f"Dados insuficientes para PCA. Sítios válidos: {len(df_pca_ready)}, mínimo necessário: {len(STRESS_VARS_FOR_PCA)}")
data_for_pca = df_pca_ready[STRESS_VARS_FOR_PCA].values
scaler = StandardScaler(); scaled_data = scaler.fit_transform(data_for_pca)
pca = PCA(n_components=2); pca_scores = pca.fit_transform(scaled_data)
loadings_df = pd.DataFrame(pca.components_.T, columns=['PC1', 'PC2'], index=STRESS_VARS_FOR_PCA)
explained_variance = pca.explained_variance_ratio_
df_metrics['PC1_scores'] = np.nan; df_metrics['PC2_scores'] = np.nan
df_metrics.loc[df_pca_ready.index, 'PC1_scores'] = pca_scores[:, 0]
df_metrics.loc[df_pca_ready.index, 'PC2_scores'] = pca_scores[:, 1]
print("   - PCA concluída. Scores e loadings calculados.")

# =================================================================================
# === 3. ANÁLISE DE SENSIBILIDADE E GERAÇÃO DE MÚLTIPLOS ÍNDICES DE ESTRESSE ===
# =================================================================================
print("\n" + "="*60); print("--- 3. INICIANDO ANÁLISE DE SENSIBILIDADE AOS PESOS DAS MÉTRICAS ---"); print("="*60)

# --- 1. Definir os cenários de pesos a serem testados ---
# Primeiro, definimos os parâmetros do caso base
base_case_weights = {
    'sst_magnitude': 0.5, 'dli_magnitude': 0.5, 'sst_amplitude': 1.0, 
    'dli_amplitude': 1.0, 'sst_frequencia': 1.5, 'dli_frequencia': 1.5,
    'sinergia_sst_dli': 1.5
}
base_case_mitigation = 0.5

# Agora, construímos o dicionário de cenários usando a definição do caso base
scenarios = {
    'base_case': {
        'METRIC_WEIGHTS': base_case_weights,
        'MITIGATION_WEIGHT': base_case_mitigation
    },
    
    'no_mitigation': {
        'METRIC_WEIGHTS': base_case_weights, # Reutiliza os pesos de estresse
        'MITIGATION_WEIGHT': 0.0
    },
    
    'strong_mitigation': {
        'METRIC_WEIGHTS': base_case_weights, # Reutiliza os pesos de estresse
        'MITIGATION_WEIGHT': 1.0
    },
    
    'frequency_dominant': {
        'METRIC_WEIGHTS': {
            'sst_magnitude': 0.25, 'dli_magnitude': 0.25, 'sst_amplitude': 1.0, 
            'dli_amplitude': 1.0, 'sst_frequencia': 2.0, 'dli_frequencia': 2.0,
            'sinergia_sst_dli': 2.0
        },
        'MITIGATION_WEIGHT': base_case_mitigation
    },

    'magnitude_dominant': {
        'METRIC_WEIGHTS': {
            'sst_magnitude': 2.0, 'dli_magnitude': 2.0,
            'sst_amplitude': 1.0, 'dli_amplitude': 1.0, 'sst_frequencia': 0.5, 
            'dli_frequencia': 0.5, 'sinergia_sst_dli': 1.0
        },
        'MITIGATION_WEIGHT': base_case_mitigation
    },

    'thermal_only': {
        'METRIC_WEIGHTS': {
            'sst_magnitude': 1.0, 'dli_magnitude': 0.0, 'sst_amplitude': 1.0, 
            'dli_amplitude': 0.0, 'sst_frequencia': 1.0, 'dli_frequencia': 0.0,
            'sinergia_sst_dli': 1.0
        },
        'MITIGATION_WEIGHT': base_case_mitigation
    },

    'light_only': {
        'METRIC_WEIGHTS': {
            'sst_magnitude': 0.0, 'dli_magnitude': 1.0, 'sst_amplitude': 0.0, 
            'dli_amplitude': 1.0, 'sst_frequencia': 0.0, 'dli_frequencia': 1.0,
            'sinergia_sst_dli': 1.0
        },
        'MITIGATION_WEIGHT': base_case_mitigation
    }
}

# ... (o resto do script continua a partir daqui, sem nenhuma outra alteração necessária) ...
df_sensitivity_results = df_metrics.copy()
min_max_scaler = MinMaxScaler()
# Pré-calcular o índice de mitigação, pois ele não muda entre os cenários
if MITIGATION_VAR in df_sensitivity_results.columns and not df_sensitivity_results[MITIGATION_VAR].isnull().all():
    mitigation_values = df_sensitivity_results[[MITIGATION_VAR]].values
    valid_mask_mit = ~np.isnan(mitigation_values).flatten()
    df_sensitivity_results['indice_mitigacao'] = np.nan
    df_sensitivity_results.loc[valid_mask_mit, 'indice_mitigacao'] = min_max_scaler.fit_transform(mitigation_values[valid_mask_mit].reshape(-1, 1)).flatten()

for scenario_name, params in scenarios.items():
    print(f"\n--- Calculando índice para o cenário: '{scenario_name}' ---")
    current_metric_weights = params['METRIC_WEIGHTS']
    current_mitigation_weight = params['MITIGATION_WEIGHT']
    ecological_weight_vector = loadings_df.index.map(current_metric_weights).values
    weighted_stress_components = []
    for i, pc_name in enumerate(loadings_df.columns):
        pc_loadings = loadings_df[pc_name].values
        pos_mask, neg_mask = pc_loadings > 0, pc_loadings < 0
        pos_energy = np.sum(np.abs(pc_loadings[pos_mask]) * ecological_weight_vector[pos_mask])
        neg_energy = np.sum(np.abs(pc_loadings[neg_mask]) * ecological_weight_vector[neg_mask])
        balance_ratio = 0 if (pos_energy == 0 or neg_energy == 0) else min(pos_energy, neg_energy) / max(pos_energy, neg_energy)
        is_bipolar = balance_ratio > BIPOLAR_THRESHOLD
        pc_scores_vector = df_sensitivity_results[f'{pc_name}_scores'].values
        if is_bipolar:
            stress_component = np.abs(pc_scores_vector)
        else:
            directionality_coeff = np.dot(pc_loadings, ecological_weight_vector)
            stress_component = pc_scores_vector * np.sign(directionality_coeff)
        final_weighted_component = stress_component * explained_variance[i]
        weighted_stress_components.append(final_weighted_component)

    indice_bruto_raw = np.nansum(np.array(weighted_stress_components), axis=0)
    valid_mask = ~np.isnan(indice_bruto_raw)
    col_name_bruto = f'stress_bruto_{scenario_name}'; df_sensitivity_results[col_name_bruto] = np.nan
    df_sensitivity_results.loc[valid_mask, col_name_bruto] = min_max_scaler.fit_transform(indice_bruto_raw[valid_mask].reshape(-1, 1)).flatten()
    col_name_liquido = f'stress_liquido_{scenario_name}'
    if 'indice_mitigacao' in df_sensitivity_results.columns:
        df_sensitivity_results[col_name_liquido] = (df_sensitivity_results[col_name_bruto] - (df_sensitivity_results['indice_mitigacao'] * current_mitigation_weight)).clip(0, 1)
    else:
        df_sensitivity_results[col_name_liquido] = df_sensitivity_results[col_name_bruto]

sensitivity_output_path = os.path.join(output_dir, "resultados_analise_sensibilidade.xlsx")
df_sensitivity_results.to_excel(sensitivity_output_path, index=False)
print("\n" + "="*60); print(f"--- ANÁLISE DE SENSIBILIDADE CONCLUÍDA ---\nResultados para todos os cenários salvos em: {sensitivity_output_path}"); print("="*60)

# ==============================
# --- 4. VISUALIZAÇÃO DOS RESULTADOS ---
# ==============================
print("\n--- 4. Gerando Visualizações Finais ---")

def create_composite_pca_figure(df_scores, loadings, explained_variance, output_filename):
    fig, axes = plt.subplots(1, 3, figsize=(24, 7), gridspec_kw={'width_ratios': [1.2, 1, 0.8]})
    fig.suptitle('Resumo da Análise de Componentes Principais do Estresse Ambiental', fontsize=20, y=1.02)
    ax1 = axes[0]
    unique_reefs=df_scores['Reef_name'].unique(); cmap=plt.get_cmap('tab10'); color_map={r: cmap(i % 10) for i, r in enumerate(unique_reefs)}
    unique_habitats=df_scores['HAB'].unique(); habitat_shapes=['o','s','^','D','v','<','>']; shape_map={hab: habitat_shapes[i % len(habitat_shapes)] for i, hab in enumerate(unique_habitats)}
    arch_styles={'inner': {'edgecolor':'black','linewidth':2.0},'outer': {'edgecolor':'darkgrey','linewidth':0.75}}
    for _, row in df_scores.iterrows():
        if pd.notna(row['PC1_scores']):
            style = arch_styles.get(row['Arch'].lower(), {'edgecolor': 'grey', 'linewidth': 0.5})
            ax1.scatter(row['PC1_scores'], row['PC2_scores'], color=color_map.get(row['Reef_name']), marker=shape_map.get(row['HAB']), s=100, alpha=0.8, **style)
    ax1.set_xlabel(f"PC1 ({explained_variance[0] * 100:.1f}%)", fontsize=14); ax1.set_ylabel(f"PC2 ({explained_variance[1] * 100:.1f}%)", fontsize=14)
    ax1.set_title("Ordenação dos Sítios/Habitats", fontsize=16); ax1.grid(True, linestyle='--', alpha=0.6)
    ax1.axhline(0,color='grey',lw=0.5); ax1.axvline(0,color='grey',lw=0.5)
    legend_elements_color=[plt.Line2D([0],[0],marker='o',color='w',label=reef,markersize=10,markerfacecolor=color_map[reef]) for reef in unique_reefs]
    legend_elements_shape=[plt.Line2D([0],[0],marker=shape_map[hab],color='grey',label=hab,linestyle='None',markersize=10) for hab in unique_habitats]
    legend_elements_arch=[plt.Line2D([0],[0],marker='o',color='w',label='Inner Arc',markersize=10,markeredgecolor=arch_styles['inner']['edgecolor'],markeredgewidth=arch_styles['inner']['linewidth']),plt.Line2D([0],[0],marker='o',color='w',label='Outer Arc',markersize=10,markeredgecolor=arch_styles['outer']['edgecolor'],markeredgewidth=arch_styles['outer']['linewidth'])]
    fig.legend(title="Recife",handles=legend_elements_color,loc='center left',bbox_to_anchor=(0.91,0.75)); fig.legend(title="Habitat",handles=legend_elements_shape,loc='center left',bbox_to_anchor=(0.91,0.5)); fig.legend(title="Arco",handles=legend_elements_arch,loc='center left',bbox_to_anchor=(0.91,0.25))
    ax2=axes[1]
    ax2.axhline(0,color='grey',lw=0.5); ax2.axvline(0,color='grey',lw=0.5)
    for i,var in enumerate(loadings.index):
        ax2.arrow(0,0,loadings['PC1'][i]*2,loadings['PC2'][i]*2,head_width=0.05,head_length=0.1,fc='red',ec='red')
        ax2.text(loadings['PC1'][i]*2.2,loadings['PC2'][i]*2.2,var,color='black',ha='center',va='center',fontsize=12)
    ax2.set_xlim(-2.5,2.5); ax2.set_ylim(-2.5,2.5); ax2.set_xlabel("Contribuição para PC1",fontsize=14); ax2.set_ylabel("Contribuição para PC2",fontsize=14)
    ax2.set_title("Loadings das Métricas de Estresse",fontsize=16); ax2.set_aspect('equal',adjustable='box')
    ax3=axes[2]
    components=['PC1','PC2']
    ax3.bar(components,explained_variance*100,color='skyblue',edgecolor='black'); ax3.set_ylabel("Variância Explicada (%)",fontsize=14)
    ax3.set_title("Importância dos Componentes",fontsize=16); ax3.set_ylim(0,100)
    for i,v in enumerate(explained_variance*100): ax3.text(i,v+2,f"{v:.1f}%",ha='center',color='black',fontsize=12)
    fig.subplots_adjust(right=0.9)
    plt.savefig(output_filename,dpi=300,bbox_inches='tight'); plt.close(fig)
    print(f"Figura de resumo da PCA salva em: {output_filename}")

create_composite_pca_figure(df_scores=df_sensitivity_results, loadings=loadings_df, explained_variance=explained_variance, output_filename=os.path.join(output_dir, "figura_resumo_PCA_Estresse.png"))

# --- Gráficos Comparativos da Análise de Sensibilidade ---
print("\n--- Gerando gráficos comparativos da Análise de Sensibilidade ---")
stress_cols = [col for col in df_sensitivity_results.columns if 'stress_liquido' in col]
df_stress_indices = df_sensitivity_results[stress_cols]
df_stress_indices.columns = [c.replace('stress_liquido_', '') for c in stress_cols]

# A. Matriz de Correlação
corr_matrix = df_stress_indices.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='viridis', fmt='.2f', linewidths=.5, vmin=0, vmax=1)
plt.title('Correlação entre os Índices de Estresse dos Cenários', fontsize=16)
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout()
corr_path = os.path.join(output_dir, "diagnostico_correlacao_cenarios.png")
plt.savefig(corr_path, dpi=300); plt.close()
print(f"Matriz de correlação salva em: {corr_path}")

# B. Gráfico de Ranking por Sítio
plt.figure(figsize=(16, 9))
for col in df_stress_indices.columns:
    plt.plot(df_sensitivity_results['unique_id'], df_stress_indices[col].rank(pct=True), marker='o', linestyle='-', label=col, alpha=0.7)
plt.ylabel("Ranking de Estresse (Percentil)", fontsize=12)
plt.xlabel("Sítio / Habitat", fontsize=12)
plt.xticks(rotation=90, fontsize=8)
plt.title('Mudança no Ranking de Estresse por Sítio entre Cenários', fontsize=16)
plt.legend(title='Cenário', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.grid(axis='y', linestyle=':')
plt.tight_layout(rect=[0, 0, 0.88, 1])
rank_path = os.path.join(output_dir, "diagnostico_mudanca_ranking_cenarios.png")
plt.savefig(rank_path, dpi=300); plt.close()
print(f"Gráfico de mudança de ranking salvo em: {rank_path}")

print("\n\n--- PROCESSO COMPLETO FINALIZADO COM SUCESSO ---")

DataFrame de sítios carregado com sucesso.

--- 1. Carregando Dados e Calculando Métricas Locais ---

--- Carregando dados para: coraltemp_v3.1_*.nc em (2002, 2008) ---
Encontrados 2557 arquivos válidos.

--- Carregando dados para: AQUA_MODIS.*.L3m.DAY.CHL.chlor_a.4km.nc em (2002, 2008) ---
Encontrados 2366 arquivos válidos.

--- Carregando dados para: AQUA_MODIS.*.L3m.DAY.KD.Kd_490.4km.nc em (2002, 2008) ---
Encontrados 2366 arquivos válidos.

--- Carregando dados para: AQUA_MODIS.*.L3m.DAY.PAR.par.4km.nc em (2002, 2008) ---
Encontrados 2359 arquivos válidos.
   - Calculando métricas avançadas...

--- 2. Executando PCA sobre as Métricas de Estresse ---
   - PCA concluída. Scores e loadings calculados.

--- 3. INICIANDO ANÁLISE DE SENSIBILIDADE AOS PESOS DAS MÉTRICAS ---

--- Calculando índice para o cenário: 'base_case' ---

--- Calculando índice para o cenário: 'no_mitigation' ---

--- Calculando índice para o cenário: 'strong_mitigation' ---

--- Calculando índice para o cenário: 'f

C:\Users\rbfra\AppData\Local\Temp\ipykernel_35336\853926799.py:288: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df_sensitivity_results.loc[valid_mask_mit, 'indice_mitigacao'] = min_max_scaler.fit_transform(mitigation_values[valid_mask_mit].reshape(-1, 1)).flatten()


Figura de resumo da PCA salva em: C:\Users\rbfra\OneDrive\########PUBLICACOES\###########Global coral refugia and conservation priorities\Local_StressIndex_v4_Sensitivity\figura_resumo_PCA_Estresse.png

--- Gerando gráficos comparativos da Análise de Sensibilidade ---
Matriz de correlação salva em: C:\Users\rbfra\OneDrive\########PUBLICACOES\###########Global coral refugia and conservation priorities\Local_StressIndex_v4_Sensitivity\diagnostico_correlacao_cenarios.png
Gráfico de mudança de ranking salvo em: C:\Users\rbfra\OneDrive\########PUBLICACOES\###########Global coral refugia and conservation priorities\Local_StressIndex_v4_Sensitivity\diagnostico_mudanca_ranking_cenarios.png


--- PROCESSO COMPLETO FINALIZADO COM SUCESSO ---


In [ ]:
# CÉLULA DE DIAGNÓSTICO IMAGENS MODIS - Execute esta célula primeiro
import os
import glob
import xarray as xr

def diagnosticar_arquivos():
    """Diagnóstico detalhado dos arquivos"""
    patterns = {
        'SST': 'coraltemp_v3.1_*.nc',
        'CHL': 'AQUA_MODIS.*.L3m.DAY.CHL.chlor_a.4km.nc', 
        'KD490': 'AQUA_MODIS.*.L3m.DAY.KD.Kd_490.4km.nc',
        'PAR': 'AQUA_MODIS.*.L3m.DAY.PAR.par.4km.nc'
    }
    
    dirs = {
        'SST': r"E:\remote sensing\CRW_SST_FULL",
        'CHL': r"E:\remote sensing\MODIS_DATA_FULL",
        'KD490': r"E:\remote sensing\MODIS_DATA_FULL", 
        'PAR': r"E:\remote sensing\MODIS_DATA_FULL"
    }
    
    for nome, pattern in patterns.items():
        print(f"\n{'='*50}")
        print(f"DIAGNÓSTICO {nome}")
        print(f"{'='*50}")
        
        caminho = os.path.join(dirs[nome], pattern)
        arquivos = sorted(glob.glob(caminho))
        print(f"Arquivos encontrados: {len(arquivos)}")
        
        if arquivos:
            # Testa o primeiro arquivo
            primeiro = arquivos[0]
            print(f"Primeiro arquivo: {os.path.basename(primeiro)}")
            
            try:
                with xr.open_dataset(primeiro) as ds:
                    print(f"Variáveis: {list(ds.data_vars)}")
                    print(f"Dimensões: {dict(ds.dims)}")
                    print(f"Coordenadas: {list(ds.coords)}")
                    if 'time' in ds.coords:
                        print(f"Tempo: {ds.time.values}")
            except Exception as e:
                print(f"ERRO ao abrir: {e}")

# Executa diagnóstico
diagnosticar_arquivos()

In [ ]:
##### TIME_SERIES e LAG temporal entre SST vs DLI vs CHLO 
### AGRUPAMENTO POR ARCO

import os
import xarray as xr
import numpy as np
import logging
import glob
import re
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
import pandas as pd

# ============================================
# Configurações Iniciais
# ============================================

logging.basicConfig(
    filename='file_open_errors_timeseries.log',
    filemode='w',
    level=logging.ERROR,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# --- Chaves para seleção de período ---
start_year = 2019
end_year = 2020

# --- Filtro de data de início específica ---
use_specific_start_date = True
specific_start_date = '2019-01-01'

# --- Diretórios dos dados ---
sst_dir = r'E:\remote sensing\CRW_SST_FULL'
modis_dir = r'E:\remote sensing\MODIS_DATA_FULL'
bathymetry_file = r'C:\Users\rbfra\OneDrive\GIS shapes\GEBCO_2024b.nc'
output_dir = r'C:\Users\rbfra\OneDrive\########PUBLICACOES\############Menezes et al. Mus his distribution and abundance Abrolhos\########NEW RESULTS\ALCA_teste'

# --- Caminho para o arquivo de sites ---
sites_csv_path = r'C:\Users\rbfra\OneDrive\########PUBLICACOES\############Menezes et al. Mus his distribution and abundance Abrolhos\######22.04.23\DATA\sites_list_full_clean_PLEST.csv'

os.makedirs(output_dir, exist_ok=True)

# --- Limites da região de interesse ---
lat_min, lat_max = -18.3, -17.3
lon_min, lon_max = -39.5, -38.3

# --- Padrões de busca para os arquivos ---
sst_pattern_crw = 'coraltemp_v3.1_*.nc'
kd490_pattern = 'AQUA_MODIS.*.L3m.DAY.KD.Kd_490.4km.nc'
par_pattern = 'AQUA_MODIS.*.L3m.DAY.PAR.par.4km.nc'
### NOVO ### Adicionado padrão para clorofila
chl_pattern = 'AQUA_MODIS.*.L3m.DAY.CHL.chlor_a.4km.nc'


# --- Carregar e PROCESSAR a lista de sítios ---
try:
    sites_df = pd.read_csv(sites_csv_path, sep=';')
    sites_df.columns = [col.strip().upper() for col in sites_df.columns]
    
    # Mapeamento dos nomes de colunas
    column_mapping = {
        'LATITUDE': 'LAT',
        'LONGITUDE': 'LONG',
        'DEPTH_M': 'DEPTH_M',
        'SITE_NAME': 'SITE_NAME',
        'ARCH': 'ARCH'
    }
    
    sites_df = sites_df.rename(columns=column_mapping)
    
    required_cols = {'SITE_NAME', 'LAT', 'LONG', 'ARCH', 'DEPTH_M'}
    if not required_cols.issubset(sites_df.columns):
        missing_cols = required_cols - set(sites_df.columns)
        raise ValueError(f"Colunas necessárias não encontradas: {missing_cols}. Colunas disponíveis: {list(sites_df.columns)}")

    def classify_arch(arch_string):
        if 'inner' in arch_string.lower():
            return 'Inner Arch'
        elif 'outer' in arch_string.lower():
            return 'Outer Arch'
        return 'Other'

    sites_df['ARCH_SIMPLE'] = sites_df['ARCH'].apply(classify_arch)
    sites_df = sites_df[sites_df['ARCH_SIMPLE'].isin(['Inner Arch', 'Outer Arch'])].copy()
    
    print(f"{len(sites_df)} sites carregados com sucesso e classificados como 'Inner' ou 'Outer'.")
    print("Contagem de sites por 'ARCH_SIMPLE':")
    print(sites_df['ARCH_SIMPLE'].value_counts())

except FileNotFoundError:
    print(f"ERRO CRÍTICO: Arquivo de sites não encontrado em: {sites_csv_path}")
    exit()
except Exception as e:
    print(f"ERRO CRÍTICO ao ler o arquivo de sites: {e}")
    exit()

# ============================================
# Funções Auxiliares
# ============================================

def parse_date_from_filename(filename):
    basename = os.path.basename(filename)
    match = re.search(r'(\d{8})', basename)
    if match:
        try:
            return np.datetime64(datetime.strptime(match.group(1), '%Y%m%d'), 'D')
        except ValueError:
            return None
    logging.error(f"Data não encontrada ou em formato inválido no arquivo: {basename}")
    return None

def filter_files_by_year_range(files, start_y, end_y):
    filtered_list = []
    print(f"Filtrando {len(files)} arquivos entre os anos {start_y} e {end_y}...")
    for f in files:
        date = parse_date_from_filename(f)
        if date is not None:
            year = pd.to_datetime(date).year
            if start_y <= year <= end_y:
                filtered_list.append(f)
    print(f"-> {len(filtered_list)} arquivos permaneceram após o filtro.")
    return filtered_list

def get_lat_slice(ds, lat_min, lat_max):
    lat_values = ds['lat'].values
    return slice(lat_min, lat_max) if lat_values[0] < lat_values[-1] else slice(lat_max, lat_min)

def get_sst_variable(ds):
    possible_vars = ['analysed_sst', 'sea_surface_temperature', 'sst']
    for var in possible_vars:
        if var in ds.data_vars:
            return var
    logging.error("Nenhuma variável válida de SST encontrada.")
    return None

### NOVO ### Função para carregar o stack de referência
def load_reference_stack(pattern, base_dir, var_names, lat_slice, lon_slice, start_y, end_y):
    """Carrega o stack de referência, fatiando cada arquivo para economizar memória."""
    print(f"\nCarregando stack de referência para: {pattern}...")
    all_files = sorted(glob.glob(os.path.join(base_dir, pattern)))
    files_in_period = filter_files_by_year_range(all_files, start_y, end_y)

    if not files_in_period:
        print(f"AVISO: Nenhum arquivo encontrado para {pattern} no período {start_y}-{end_y}.")
        return None
    
    data_list = []
    for f in files_in_period:
        try:
            with xr.open_dataset(f) as ds:
                var_name = next((v for v in var_names if v in ds.data_vars), None)
                if not var_name: continue
                
                date_val = parse_date_from_filename(f)
                if date_val is None: continue

                # A fatia espacial é aplicada AQUI, para cada arquivo, economizando memória
                data_sub = ds[var_name].sel(lat=lat_slice, lon=lon_slice)
                if 'time' in data_sub.dims:
                    data_sub = data_sub.isel(time=0, drop=True)
                
                data_sub = data_sub.expand_dims(dim={'time': [date_val]})
                data_list.append(data_sub)
        except Exception as e:
            logging.error(f"Erro ao abrir ou processar {f}: {e}")
            continue
            
    if not data_list: return None
    return xr.concat(data_list, dim='time').sortby('time')

### NOVO ### Função para carregar e alinhar dados com o stack de referência
def load_and_interpolate_stack(pattern, base_dir, var_names, start_y, end_y, reference_stack):
    """
    Carrega dados de satélite e os interpola imediatamente para a grade de um stack de referência.
    Esta é a abordagem mais segura em termos de memória para alinhar grades.
    """
    print(f"\nCarregando e interpolando dados para: {pattern}...")
    all_files = sorted(glob.glob(os.path.join(base_dir, pattern)))
    files_in_period = filter_files_by_year_range(all_files, start_y, end_y)

    if not files_in_period:
        print(f"AVISO: Nenhum arquivo encontrado para {pattern} no período {start_y}-{end_y}.")
        return None
    
    # Pega a grade de referência (apenas as coordenadas, sem os dados)
    reference_grid = reference_stack.isel(time=0, drop=True)
    
    data_list = []
    for f in files_in_period:
        try:
            with xr.open_dataset(f) as ds:
                var_name = next((v for v in var_names if v in ds.data_vars), None)
                if not var_name: continue
                
                date_val = parse_date_from_filename(f)
                if date_val is None: continue

                data_sub = ds[var_name]
                if 'time' in data_sub.dims:
                    data_sub = data_sub.isel(time=0, drop=True)
                
                # Interpola o dado do dia para a grade de referência ANTES de adicionar à lista
                interp_sub = data_sub.interp_like(reference_grid, method='nearest')
                
                interp_sub = interp_sub.expand_dims(dim={'time': [date_val]})
                data_list.append(interp_sub)
        except Exception as e:
            logging.error(f"Erro ao abrir, processar ou interpolar {f}: {e}")
            continue
            
    if not data_list: 
        print(f"AVISO: Nenhum dado pôde ser extraído e interpolado para {pattern}")
        return None
        
    return xr.concat(data_list, dim='time').sortby('time')

def calculate_dli_benthic_per_site(kd490_files, par_files, sites_df_group):
    par_files_dict = {parse_date_from_filename(f): f for f in par_files if parse_date_from_filename(f)}
    dli_per_site_ts = {}

    for index, site in sites_df_group.iterrows():
        site_name = site['SITE_NAME']
        site_lat, site_lon, site_depth = site['LAT'], site['LONG'], site['DEPTH_M']
        
        dli_values = []
        dates = []
        
        for f_kd in kd490_files:
            date_val = parse_date_from_filename(f_kd)
            if date_val is None or date_val not in par_files_dict:
                continue
            
            f_par = par_files_dict[date_val]
            try:
                with xr.open_dataset(f_kd) as ds_kd490, xr.open_dataset(f_par) as ds_par:
                    kd_point = ds_kd490['Kd_490'].sel(lat=site_lat, lon=site_lon, method='nearest').item(0)
                    par_point = ds_par['par'].sel(lat=site_lat, lon=site_lon, method='nearest').item(0)
                    
                    if np.isnan(kd_point) or np.isnan(par_point):
                        dli_benthic = np.nan
                    else:
                        kdpar = 0.0665 + 0.874 * kd_point - 0.00121 / (kd_point + 1e-9)
                        dli_benthic = par_point * np.exp(-kdpar * site_depth)
                    
                    dli_values.append(dli_benthic)
                    dates.append(date_val)
            except Exception as e:
                logging.error(f"Erro ao processar DLI para {site_name} na data {date_val}: {e}")
                continue
        
        if dli_values:
            dli_per_site_ts[site_name] = xr.DataArray(dli_values, coords={'time': dates}, dims=['time'])

    return dli_per_site_ts

### NOVO ### Função para calcular a correlação cruzada e o lag
def calculate_cross_correlation(series1, series2, max_lag_days=90):
    """
    Calcula a correlação cruzada entre duas séries temporais.
    'series1' é a variável que se supõe que "lidera" (causa).
    'series2' é a variável que se supõe que "responde" (efeito).
    
    Um lag positivo significa que `series1` antecede `series2`.
    Ex: Se DLI é series1 e SST é series2, um lag de +15 dias significa 
    que o pico de SST ocorre 15 dias após o pico de DLI.
    """
    # Alinha as séries e remove os dias onde não há dados para ambos
    df = pd.DataFrame({'s1': series1, 's2': series2}).dropna()
    if len(df) < 30: # Requer um número mínimo de pontos para uma correlação significativa
        return np.nan, np.nan

    correlations = []
    # O lag é o quanto deslocamos a 'series1' para "frente" no tempo
    lags = range(-max_lag_days, max_lag_days + 1)
    
    for lag in lags:
        # df['s1'].shift(lag) desloca s1. Se lag=15, um valor de s1 no dia 1 é comparado com um valor de s2 no dia 16.
        # Isso testa se s1 de X dias atrás se correlaciona com s2 de hoje.
        corr = df['s2'].corr(df['s1'].shift(lag))
        correlations.append(corr)

    # Encontra o lag com a correlação máxima
    max_corr_idx = np.nanargmax(np.abs(correlations))
    optimal_lag = lags[max_corr_idx]
    max_corr = correlations[max_corr_idx]
    
    return optimal_lag, max_corr

# ============================================
# Carga de Dados e Processamento
# ============================================
print("\nListando e filtrando arquivos de dados...")
# Define as fatias de coordenadas com base no primeiro arquivo SST
try:
    primeiro_sst = glob.glob(os.path.join(sst_dir, sst_pattern_crw))[0]
    with xr.open_dataset(primeiro_sst) as ds:
        lat_s = get_lat_slice(ds, lat_min, lat_max)
        lon_s = slice(lon_min, lon_max)
except IndexError:
    print(f"ERRO CRÍTICO: Nenhum arquivo SST encontrado no padrão '{sst_pattern_crw}' em '{sst_dir}'.")
    exit()

# 1. Carrega o stack de referência (SST)
sst_stack = load_reference_stack(sst_pattern_crw, sst_dir, ['analysed_sst'], lat_s, lon_s, start_year, end_year)

# 2. Carrega e interpola a Clorofila para a grade do SST
chl_stack = None
if sst_stack is not None:
    chl_stack = load_and_interpolate_stack(chl_pattern, modis_dir, ['chlor_a'], start_year, end_year, reference_stack=sst_stack)
else:
    print("ERRO: Stack de SST não pôde ser carregado. Não é possível continuar com a Clorofila.")

# 3. Carrega os arquivos para cálculo de DLI
kd490_files = filter_files_by_year_range(glob.glob(os.path.join(modis_dir, kd490_pattern)), start_year, end_year)
par_files = filter_files_by_year_range(glob.glob(os.path.join(modis_dir, par_pattern)), start_year, end_year)

print("\nCalculando DLI Bentônico para cada site individualmente...")
dli_ts_by_site = calculate_dli_benthic_per_site(kd490_files, par_files, sites_df)

# ============================================
# Filtro de Data Preciso Pós-Carga
# ============================================
if use_specific_start_date:
    print(f"\n--- Aplicando filtro de data de início: {specific_start_date} ---")
    end_date_str = f"{end_year}-12-31" 
    if sst_stack is not None:
        sst_stack = sst_stack.sel(time=slice(specific_start_date, end_date_str))
    if chl_stack is not None:
        chl_stack = chl_stack.sel(time=slice(specific_start_date, end_date_str))
    
    # Filtra também as séries de DLI
    dli_ts_by_site_filtered = {}
    for site, ts in dli_ts_by_site.items():
        dli_ts_by_site_filtered[site] = ts.sel(time=slice(specific_start_date, end_date_str))
    dli_ts_by_site = dli_ts_by_site_filtered

# ============================================
# Geração de Séries Temporais Agrupadas e Análise de Lag
# ============================================

### ATUALIZADO ### Função principal de plotagem e análise para 3 variáveis
def plot_and_analyze_grouped_series(sst_data, chl_data, dli_data_by_site, sites_dataframe, output_dir):
    if sst_data is None or chl_data is None or not dli_data_by_site:
        print("Dados de SST, Chl-a ou DLI ausentes. Não é possível gerar o gráfico.")
        return

    grouped_sites = sites_dataframe.groupby('ARCH_SIMPLE')
    group_order = ['Inner Arch', 'Outer Arch']
    
    fig, axes = plt.subplots(nrows=len(group_order), ncols=1, figsize=(20, 8 * len(group_order)), sharex=True, squeeze=False)
    axes = axes.flatten()

    for i, arch_name in enumerate(group_order):
        ax = axes[i]
        try:
            group_df = grouped_sites.get_group(arch_name)
        except KeyError:
            print(f"AVISO: Nenhum site encontrado para o grupo '{arch_name}'. Pulando.")
            ax.set_title(f"Grupo: {arch_name} (Nenhum site)")
            ax.text(0.5, 0.5, 'Sem dados para este grupo', horizontalalignment='center', verticalalignment='center', transform=ax.transAxes)
            continue
            
        print(f"\n>>> Processando grupo '{arch_name}' com {len(group_df)} site(s)...")
        
        # --- Extração das Séries Médias para o Grupo ---
        site_lats = xr.DataArray(group_df['LAT'].values, dims="site")
        site_lons = xr.DataArray(group_df['LONG'].values, dims="site")

        mean_sst_ts = sst_data.sel(lat=site_lats, lon=site_lons, method='nearest').mean(dim='site', skipna=True)
        mean_chl_ts = chl_data.sel(lat=site_lats, lon=site_lons, method='nearest').mean(dim='site', skipna=True)
        
        group_site_names = group_df['SITE_NAME'].tolist()
        dli_series_list = [dli_data_by_site[name] for name in group_site_names if name in dli_data_by_site]
        
        if not dli_series_list: 
            print(f"AVISO: Nenhum dado de DLI para o grupo '{arch_name}'.")
            continue
        
        aligned_dli = xr.align(*dli_series_list, join='outer')
        mean_dli_ts = xr.concat(aligned_dli, dim='site').mean(dim='site', skipna=True)
        
        # --- Análise de Correlação Cruzada (sem alteração) ---
        print(f"--- Análise de Lag para o Grupo: {arch_name} ---")
        
        dli_pd = mean_dli_ts.to_pandas()
        sst_pd = mean_sst_ts.to_pandas()
        chl_pd = mean_chl_ts.to_pandas()
        
        pairs = {
            'DLI -> SST': (dli_pd, sst_pd),
            'SST -> Chl-a': (sst_pd, chl_pd),
            'DLI -> Chl-a': (dli_pd, chl_pd)
        }
        
        lag_results = {}
        for name, (s1, s2) in pairs.items():
            lag, corr = calculate_cross_correlation(s1, s2)
            lag_results[name] = (lag, corr)
            if not np.isnan(lag):
                print(f"{name}: Lag ótimo de {int(lag)} dias (Correlação = {corr:.2f})")
            else:
                print(f"{name}: Não foi possível calcular o lag (dados insuficientes).")
        
        # <--- INÍCIO DA MUDANÇA NO PLOT --->
        # --- Plotagem da Série Temporal (3 variáveis com 3 eixos Y, usando PONTOS) ---
        color_sst = 'tab:red'
        color_dli = 'tab:blue'
        color_chl = 'tab:green'
        
        # EIXO 1 (Esquerdo): SST
        ax.set_ylabel('SST Médio (°C)', color=color_sst, fontsize=14)
        # Alterado de '-' para 'o', adicionado markersize e linestyle='None'
        p1, = ax.plot(mean_sst_ts['time'], mean_sst_ts.values, 'o', markersize=4, alpha=0.7, color=color_sst, label=f'SST (Média {arch_name})', linestyle='None')
        ax.tick_params(axis='y', labelcolor=color_sst, labelsize=12)
        ax.grid(axis='y', linestyle='--', alpha=0.6, color=color_sst)
        
        # EIXO 2 (Direito 1): DLI
        ax2 = ax.twinx()
        ax2.set_ylabel('DLI Bentônico (mol/m²/d)', color=color_dli, fontsize=14)
        # Alterado de '-' para 'o', adicionado markersize e linestyle='None'
        p2, = ax2.plot(mean_dli_ts['time'], mean_dli_ts.values, 'o', markersize=4, alpha=0.7, color=color_dli, label=f'DLI (Média {arch_name})', linestyle='None')
        ax2.tick_params(axis='y', labelcolor=color_dli, labelsize=12)
        
        # EIXO 3 (Direito 2): Clorofila
        ax3 = ax.twinx()
        ax3.spines['right'].set_position(('outward', 70))
        ax3.set_ylabel('Chl-a (mg/m³)', color=color_chl, fontsize=14)
        # Alterado de '-' para 'o', adicionado markersize e linestyle='None'
        p3, = ax3.plot(mean_chl_ts['time'], mean_chl_ts.values, 'o', markersize=4, alpha=0.7, color=color_chl, label=f'Chl-a (Média {arch_name})', linestyle='None')
        ax3.tick_params(axis='y', labelcolor=color_chl, labelsize=12)
        # <--- FIM DA MUDANÇA NO PLOT --->

        # LEGENDA UNIFICADA (sem alteração)
        lines = [p1, p2, p3]
        ax.legend(lines, [l.get_label() for l in lines], loc='upper left', fontsize=12)
        
        # Título e Eixo X (sem alteração)
        lag_str1 = f"DLI→SST: {int(lag_results['DLI -> SST'][0])}d" if not np.isnan(lag_results['DLI -> SST'][0]) else "N/A"
        lag_str2 = f"SST→Chl: {int(lag_results['SST -> Chl-a'][0])}d" if not np.isnan(lag_results['SST -> Chl-a'][0]) else "N/A"
        lag_str3 = f"DLI→Chl: {int(lag_results['DLI -> Chl-a'][0])}d" if not np.isnan(lag_results['DLI -> Chl-a'][0]) else "N/A"
        
        ax.set_title(f"Grupo: {arch_name} (n={len(group_df)}) | Lags: {lag_str1}, {lag_str2}, {lag_str3}", fontsize=16, weight='bold')
        
        ax.xaxis.set_major_locator(mdates.YearLocator(1))
        ax.xaxis.set_minor_locator(mdates.MonthLocator(interval=3))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
        plt.setp(ax.get_xticklabels(), rotation=0, ha="center", fontsize=12)

    fig.suptitle(f'Séries Temporais de SST, DLI e Chl-a por Arco ({start_year}-{end_year})', fontsize=20, y=1.02)
    fig.tight_layout(rect=[0, 0.03, 1, 0.97]) 
    main_plot_path = os.path.join(output_dir, f"time_series_SST_DLI_CHL_com_Lag_{start_year}-{end_year}.png")
    plt.savefig(main_plot_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"\nGráfico principal de séries temporais salvo em: {main_plot_path}")

# ============================================
# Execução da Plotagem e Análise
# ============================================
print("\nGerando gráficos e analisando o lag temporal para SST, DLI e Chl-a...")
plot_and_analyze_grouped_series(sst_stack, chl_stack, dli_ts_by_site, sites_df, output_dir)

print("\nScript finalizado com sucesso!")

In [ ]:
##BOXPLOT cobertura REEF vs HAB

import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# =============================================================================
# 1. Definir caminhos de entrada e saída
# =============================================================================

output_data_dir = r"C:\Users\rbfra\OneDrive\########PUBLICACOES\############Menezes et al. Mus his distribution and abundance Abrolhos\########NEW RESULTS\BOX_PLOTS"
input_file = os.path.join(output_data_dir, "dados_integrados_long_format.csv")

boxplot_dir = os.path.join(output_data_dir, "boxplots_reef_hab_auto")
os.makedirs(boxplot_dir, exist_ok=True)

# =============================================================================
# 2. Carregar os dados
# =============================================================================

df = pd.read_csv(input_file)

# =============================================================================
# 3. Configurar estilo dos gráficos
# =============================================================================

sns.set_theme(style="whitegrid", palette="muted")

# =============================================================================
# 4. Boxplot individual por organismo: Reef no eixo x, HAB como hue,
#    sem outliers e com escala de y automática
# =============================================================================

for org in sorted(df["ORGANISMO"].unique()):
    sub = df[df["ORGANISMO"] == org]
    plt.figure(figsize=(10, 6))
    ax = sns.boxplot(
        data=sub,
        x="REEF",
        y="COBERTURA",
        hue="HAB",
        order=sorted(sub["REEF"].unique()),
        showfliers=False  # oculta outliers
    )
    ax.set_title(f"Cobertura (%) de {org} por Reef e HAB", fontsize=14)
    ax.set_xlabel("Reef", fontsize=12)
    ax.set_ylabel("Cobertura (%)", fontsize=12)
    ax.legend(title="HAB", bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(os.path.join(boxplot_dir, f"boxplot_{org}_reef_hab_auto.png"), dpi=300)
    plt.close()

# =============================================================================
# 5. FacetGrid geral: Reef no x, HAB como hue, painéis por organismo,
#    escala y automática em cada painel
# =============================================================================

g = sns.catplot(
    data=df,
    x="REEF",
    y="COBERTURA",
    hue="HAB",
    col="ORGANISMO",
    kind="box",
    sharey=False,    # cada painel auto-ajusta seu próprio eixo y
    col_wrap=3,
    height=4,
    aspect=1,
    showfliers=False
)
g.set_titles("{col_name}")
g.set_axis_labels("Reef", "Cobertura (%)")
for ax in g.axes.flatten():
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
g.savefig(os.path.join(boxplot_dir, "boxplots_facet_reef_hab_auto.png"), dpi=300)
plt.close()


In [ ]:
# =============================================================================
# SCRIPT PARA COMPARAR MÉDIA E VARIABILIDADE (CV) DE SST, DLI, CHL-A E DHW
# Objetivo: Criar duas figuras compostas separadas para Média e CV,
#           com comparação entre arcos usando sítios como réplicas.
# Versão 4.2 - Correção definitiva de nome de colunas e bug no gráfico de CV.
# =============================================================================

import os
import glob
import re
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import gc
from datetime import datetime
import warnings

# Silenciar avisos
warnings.filterwarnings("ignore", category=UserWarning, message="Sending large graph of Dask arrays")
warnings.filterwarnings("ignore", category=RuntimeWarning, message="Mean of empty slice")

# ==============================
# 1. CONFIGURAÇÕES PRINCIPAIS
# ==============================
SITES_CSV_FILE = r"C:\Users\rbfra\OneDrive\########CEBIMAR\####PROJETOS\#####Coral trade offs\sites_list_full.csv"
SST_DIR   = r"H:\remote sensing\CRW_SST_FULL"
MODIS_DIR = r"H:\remote sensing\MODIS_DATA_FULL"
DHW_DIR   = r"H:\remote sensing\CRW_DHW_FULL"
OUTPUT_DIR = r"C:\Users\rbfra\OneDrive\########PUBLICACOES\############Menezes et al. Mus his distribution and abundance Abrolhos\########NEW RESULTS\output_Mean_CV_Comparison_FINAL"

# Períodos para teste rápido - use os períodos completos para a análise final
SST_PERIOD   = (2007, 2008)
DHW_PERIOD   = (2007, 2008)
MODIS_PERIOD = (2007, 2008)

# Períodos completos (descomente para usar)
#SST_PERIOD   = (1985, 2008)
#DHW_PERIOD   = (1985, 2008)
#MODIS_PERIOD = (2002, 2008)

SST_PATTERN   = 'coraltemp_v3.1_*.nc'
DHW_PATTERN   = '*.nc'
KD490_PATTERN = 'AQUA_MODIS.*.L3m.DAY.KD.Kd_490.4km.nc'
PAR_PATTERN   = 'AQUA_MODIS.*.L3m.DAY.PAR.par.4km.nc'
CHL_PATTERN   = 'AQUA_MODIS.*.L3m.DAY.CHL.chlor_a.4km.nc'

WINDOW_SIZES = [2, 7, 30, 180, 365]
KDPAR_GATTUSO_A, KDPAR_GATTUSO_B, KDPAR_GATTUSO_C = 0.0665, 0.874, 0.00121

# ==============================
# 2. FUNÇÕES AUXILIARES (sem alterações)
# ==============================
def preprocess_with_time(ds, filename):
    basename = os.path.basename(filename)
    match = re.search(r'(\d{8})', basename)
    if match:
        try:
            date_val = np.datetime64(datetime.strptime(match.group(1), '%Y%m%d'))
            if 'time' in ds.dims: return ds.assign_coords(time=[date_val])
            else: return ds.expand_dims(time=[date_val])
        except ValueError: return ds
    return ds

def load_and_extract_timeseries(base_dir, pattern, period, var_name_options, lats, lons):
    search_path = os.path.join(base_dir, pattern); print(f"\nProcurando arquivos em: {search_path}")
    files_to_process = []
    for f in sorted(glob.glob(search_path)):
        match = re.search(r'(\d{4})\d{4}', os.path.basename(f))
        if match and period[0] <= int(match.group(1)) <= period[1]: files_to_process.append(f)
    if not files_to_process: print(f"AVISO: Nenhum arquivo para '{pattern}' no período {period}."); return None
    print(f"Encontrados {len(files_to_process)} arquivos.")
    datasets_preprocessed = []
    for f_path in files_to_process:
        try:
            with xr.open_dataset(f_path, chunks='auto') as ds:
                datasets_preprocessed.append(preprocess_with_time(ds, f_path))
        except Exception: continue
    if not datasets_preprocessed: print("AVISO: Nenhum arquivo pôde ser pré-processado."); return None
    try:
        ds_virtual = xr.combine_by_coords(datasets_preprocessed, compat='override', coords='all', join='override', combine_attrs='override')
        var_name = next((v for v in var_name_options if v in ds_virtual.data_vars), None)
        if not var_name: print(f"AVISO: Variáveis {var_name_options} não encontradas."); del ds_virtual, datasets_preprocessed; gc.collect(); return None
        timeseries_lazy = ds_virtual[var_name].sel(lat=lats, lon=lons, method='nearest')
        timeseries_loaded = timeseries_lazy.load().astype('float32').sortby('time')
        print(f"Série '{var_name}' carregada. Shape: {timeseries_loaded.shape}")
        del ds_virtual, datasets_preprocessed, timeseries_lazy; gc.collect()
        return timeseries_loaded
    except Exception as e: print(f"ERRO ao processar '{pattern}': {e}"); return None

def calculate_metrics_for_timeseries(timeseries_da, window_sizes, var_name):
    if timeseries_da is None: return {}
    print(f"\nCalculando métricas para: {var_name.upper()}")
    results = {}
    mean_all = timeseries_da.mean('time', skipna=True)
    std_all = timeseries_da.std('time', skipna=True)
    results['mean_ALL'] = mean_all.values
    results['cv_ALL'] = ((std_all / (mean_all + 1e-9)) * 100).values
    for window in window_sizes:
        min_p = max(2, int(window * 0.25))
        rolling_mean = timeseries_da.rolling(time=window, min_periods=min_p, center=True).mean()
        rolling_std = timeseries_da.rolling(time=window, min_periods=min_p, center=True).std()
        cv_ts = (rolling_std / (rolling_mean + 1e-9)) * 100
        results[f'cv_{window}'] = cv_ts.mean('time', skipna=True).values
    return results

def calculate_dhw_frequency(timeseries_da):
    if timeseries_da is None: return {}
    print("\nCalculando frequência de DHW > 4...")
    valid_data = timeseries_da.where(timeseries_da.notnull())
    freq = (valid_data > 4).mean('time', skipna=True) * 100
    mean_dhw = valid_data.mean('time', skipna=True)
    return {'mean_DHW>4_freq': freq.values, 'mean_ALL_dhw': mean_dhw.values}


# ==============================
# 3. PROCESSAMENTO PRINCIPAL (COM CORREÇÃO DE NOMES)
# ==============================
if __name__ == "__main__":
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    try:
        # Lê o CSV removendo possíveis caracteres BOM
        df_sites = pd.read_csv(SITES_CSV_FILE, sep=';', encoding='utf-8-sig')  # Note o encoding alterado
        
        # <--- CORREÇÃO PARA SEU CSV COM BOM --->
        # 1. Limpa os nomes das colunas (remove BOM e espaços)
        df_sites.columns = df_sites.columns.str.strip()
        
        # 2. Corrige especificamente a primeira coluna que tem BOM
        df_sites.columns = [col.replace('ï»¿', '') for col in df_sites.columns]
        
        # 3. Mapeamento dos nomes do seu CSV
        rename_map = {
            'Reef_name': 'REEF',  # Agora sem o BOM
            'Site_name': 'SITE',
            'Latitude': 'LAT',
            'Longitude': 'LONG',
            'Depth_m': 'DEPTH',
            'Arch': 'ARCH'
        }
        
        # 4. Renomeia apenas as colunas que existem
        rename_map = {k: v for k, v in rename_map.items() if k in df_sites.columns}
        df_sites.rename(columns=rename_map, inplace=True)

        # 5. Verificação final
        required_cols = ['REEF', 'ARCH', 'LAT', 'LONG']
        if not all(col in df_sites.columns for col in required_cols):
            missing = [col for col in required_cols if col not in df_sites.columns]
            raise ValueError(f"ERRO CRÍTICO: Colunas ausentes: {missing}. Colunas encontradas: {df_sites.columns.tolist()}")

        # Limpeza dos dados
        for col in df_sites.select_dtypes(['object']).columns: 
            df_sites[col] = df_sites[col].str.strip()
            
        print(f"Arquivo carregado com sucesso. Colunas finais: {df_sites.columns.tolist()}")
        
    except Exception as e: 
        print(f"ERRO FATAL: {str(e)}")
        exit()

    # Preparação dos DataArrays (agora usando os nomes padronizados)
    lats = xr.DataArray(df_sites['LAT'].values, dims="site")
    lons = xr.DataArray(df_sites['LONG'].values, dims="site")
    depths = xr.DataArray(df_sites['DEPTH'].values, dims="site")
    df_results = df_sites.copy()

    # --- Processamento SST ---
    sst_timeseries = load_and_extract_timeseries(SST_DIR, SST_PATTERN, SST_PERIOD, ['analysed_sst', 'sea_surface_temperature'], lats, lons)
    if sst_timeseries is not None:
        sst_metrics = calculate_metrics_for_timeseries(sst_timeseries, WINDOW_SIZES, 'SST')
        for key, values in sst_metrics.items(): df_results[f'sst_{key}'] = values
        del sst_timeseries, sst_metrics; gc.collect()

    # --- Processamento DLI ---
    kd490_timeseries = load_and_extract_timeseries(MODIS_DIR, KD490_PATTERN, MODIS_PERIOD, ['Kd_490'], lats, lons)
    par_timeseries = load_and_extract_timeseries(MODIS_DIR, PAR_PATTERN, MODIS_PERIOD, ['par'], lats, lons)
    if kd490_timeseries is not None and par_timeseries is not None:
        kd490_aligned, par_aligned = xr.align(kd490_timeseries, par_timeseries, join='inner')
        kdpar = KDPAR_GATTUSO_A + KDPAR_GATTUSO_B * kd490_aligned - KDPAR_GATTUSO_C / (kd490_aligned + 1e-9)
        dli_timeseries = par_aligned * np.exp(-xr.where(kdpar <= 0, np.nan, kdpar) * depths)
        dli_metrics = calculate_metrics_for_timeseries(dli_timeseries, WINDOW_SIZES, 'DLI')
        for key, values in dli_metrics.items(): df_results[f'dli_{key}'] = values
        del kd490_timeseries, par_timeseries, dli_timeseries, dli_metrics; gc.collect()

    # --- Processamento Clorofila-a ---
    chl_timeseries = load_and_extract_timeseries(MODIS_DIR, CHL_PATTERN, MODIS_PERIOD, ['chlor_a'], lats, lons)
    if chl_timeseries is not None:
        chl_metrics = calculate_metrics_for_timeseries(chl_timeseries, WINDOW_SIZES, 'CHL')
        for key, values in chl_metrics.items(): df_results[f'chl_{key}'] = values
        del chl_timeseries, chl_metrics; gc.collect()

    # --- Processamento DHW ---
    dhw_timeseries = load_and_extract_timeseries(DHW_DIR, DHW_PATTERN, DHW_PERIOD, ['degree_heating_week'], lats, lons)
    if dhw_timeseries is not None:
        dhw_metrics_freq = calculate_dhw_frequency(dhw_timeseries)
        for key, values in dhw_metrics_freq.items(): df_results[key] = values
        
        print("\nCalculando métricas de CV para DHW...")
        dhw_metrics_cv = calculate_metrics_for_timeseries(dhw_timeseries, WINDOW_SIZES, 'DHW')
        for key, values in dhw_metrics_cv.items():
            if 'mean_ALL' not in key:
                 df_results[f'dhw_{key}'] = values
        del dhw_timeseries, dhw_metrics_freq, dhw_metrics_cv; gc.collect()


    results_path = os.path.join(OUTPUT_DIR, "summary_results_all_metrics.xlsx")
    df_results.to_excel(results_path, index=False)
    print(f"\nPlanilha com todos os resultados salva em: {results_path}")

# ========================================================================
# 4. PREPARAÇÃO E GERAÇÃO DOS GRÁFICOS (SOLUÇÃO DEFINITIVA PARA AJUSTE DE EIXO)
# ========================================================================
print("\nIniciando a geração dos gráficos...")
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("talk")

# ---------------------------------------------------------------------
# Função de ajuste de eixo Y - Versão Final e Robusta
# Recalcula o erro manualmente para garantir precisão total.
# ---------------------------------------------------------------------
def adjust_yaxis_definitive(g, df, x_var, y_var, col_var, row_var, padding_factor=0.2):
    """
    Ajusta o eixo Y de cada subplot em um FacetGrid recalculando os limites
    (média ± desvio padrão) a partir do dataframe original.
    Isso garante que as barras de erro nunca sejam cortadas.
    """
    # Itera sobre cada subplot usando os eixos do FacetGrid
    for i, ax in enumerate(g.axes.flat):
        if not ax.has_data():
            continue
            
        # Determina a qual linha (row) e coluna (col) este eixo pertence
        row_val = g.row_names[i // g.axes.shape[1]]
        col_val = g.col_names[i % g.axes.shape[1]]

        # Filtra o DataFrame para obter apenas os dados deste subplot específico
        subplot_data = df[(df[row_var] == row_val) & (df[col_var] == col_val)]

        if subplot_data.empty:
            continue

        # Agrupa os dados pelo que está no eixo X (ex: por 'REEF' ou 'Window')
        grouped = subplot_data.groupby(x_var)[y_var]
        
        # Calcula a média e o desvio padrão para cada grupo
        means = grouped.mean()
        stds = grouped.std()
        
        # O limite inferior é a média menos o desvio padrão, o superior é a média mais o desvio padrão
        lower_bounds = means - stds
        upper_bounds = means + stds

        # Encontra o mínimo e o máximo absolutos entre todos os limites
        data_min = np.nanmin(lower_bounds)
        data_max = np.nanmax(upper_bounds)
        
        # Se não houver dados válidos, pula para o próximo eixo
        if not (np.isfinite(data_min) and np.isfinite(data_max)):
            continue
            
        # Calcula o padding para dar "respiro" ao gráfico
        data_range = data_max - data_min
        padding = data_range * padding_factor if data_range > 0 else max(abs(data_max * 0.1), 0.1)
        
        # Define os novos limites do eixo Y com o padding
        ax.set_ylim(data_min - padding, data_max + padding)


# ---------------------------------------------------------------------
# Figura 1 – Coeficiente de Variação (CV %)
# ---------------------------------------------------------------------
variables_for_cv = ["sst", "dli", "chl", "dhw"]

# Inclui explicitamente as colunas `_cv_ALL` e as de janela
cv_cols = []
for v in variables_for_cv:
    cv_cols.append(f"{v}_cv_ALL")
    for w in WINDOW_SIZES:
        cv_cols.append(f"{v}_cv_{w}")

cv_cols = [c for c in cv_cols if c in df_results.columns]

df_cv_melted = df_results.melt(id_vars=["ARCH"], value_vars=cv_cols, var_name="Metric", value_name="Value")

# <--- INÍCIO DA CORREÇÃO --->
# Reescreve a função para retornar um DataFrame, que é mais seguro para atribuição
def parse_metric_to_df(metric_series):
    """
    Processa a série de métricas e retorna um DataFrame com as colunas 'Variable' e 'Window'.
    """
    parts = metric_series.str.split('_cv_', n=1, expand=True)
    df = pd.DataFrame({
        'Variable': parts[0].str.upper().str.replace("CHL", "Chl-a"),
        'Window': parts[1]
    })
    return df

# Atribui as novas colunas fazendo um join/merge, que é a forma mais robusta
df_cv_melted = df_cv_melted.join(parse_metric_to_df(df_cv_melted['Metric']))
# <--- FIM DA CORREÇÃO --->

df_cv_melted.dropna(subset=["Value", "ARCH"], inplace=True)

# Atualiza a ordem do eixo X para incluir 'ALL'
window_order = [str(w) for w in WINDOW_SIZES] + ['ALL']
df_cv_melted['Window'] = pd.Categorical(df_cv_melted['Window'], categories=window_order, ordered=True)
df_cv_melted = df_cv_melted.sort_values('Window')

g_cv = sns.catplot(
    data=df_cv_melted,
    x="Window", y="Value", col="Variable", row="ARCH",
    kind="bar", errorbar="sd", capsize=0.08, palette="viridis",
    height=4, aspect=1.2, sharey=False,
    order=window_order
)

# Chamando a função de ajuste
adjust_yaxis_definitive(g_cv, df_cv_melted, x_var="Window", y_var="Value", col_var="Variable", row_var="ARCH")

g_cv.fig.suptitle("Coeficiente de Variação (CV %) por Janela de Tempo e Arco", y=1.02, fontsize=20)
g_cv.set_axis_labels("Janela de Tempo (dias) / Total (ALL)", "CV (%)")
g_cv.set_titles(row_template="Arco: {row_name}", col_template="{col_name}")
g_cv.fig.tight_layout(rect=[0, 0, 1, 0.96])
cv_plot_path = os.path.join(OUTPUT_DIR, "figura_composta_CV.png")
plt.savefig(cv_plot_path, dpi=300, bbox_inches="tight")
plt.close()
print(f"Figura de CV salva em: {cv_plot_path}")

# ---------------------------------------------------------------------
# Figura 2 – Média Global por Recife
# ---------------------------------------------------------------------
mean_cols_map = {
    "sst_mean_ALL": "SST (°C)",
    "dli_mean_ALL": "DLI (mol/m²/d)",
    "chl_mean_ALL": "Chl-a (mg/m³)",
    "mean_DHW>4_freq": "Freq. DHW > 4 (%)"
}
existing_mean_cols = [k for k in mean_cols_map if k in df_results.columns]

df_mean_melted = (
    df_results
    .melt(id_vars=["REEF", "ARCH"], value_vars=existing_mean_cols,
          var_name="Variable", value_name="Value")
    .assign(Variable=lambda d: d["Variable"].map(mean_cols_map))
    .dropna(subset=["Value", "ARCH", "REEF"])
)

g_mean = sns.catplot(
    data=df_mean_melted,
    x="REEF", y="Value", col="Variable", row="ARCH",
    kind="bar", palette="muted", errorbar="sd",
    capsize=0.08, height=5, aspect=1.5,
    sharex=False, sharey=False
)

# Chamando a nova função de ajuste
adjust_yaxis_definitive(g_mean, df_mean_melted, x_var="REEF", y_var="Value", col_var="Variable", row_var="ARCH")

g_mean.fig.suptitle("Média das Variáveis por Recife e Arco (Sítios como réplicas)", fontsize=24, y=1.03)
g_mean.set_axis_labels("Recife", "Valor Médio")
g_mean.set_titles(row_template="Arco: {row_name}", col_template="{col_name}")
g_mean.set_xticklabels(rotation=45, ha="right")
g_mean.fig.tight_layout(rect=[0, 0, 1, 0.97])
mean_plot_path = os.path.join(OUTPUT_DIR, "figura_composta_Media_por_Reef.png")
plt.savefig(mean_plot_path, dpi=300, bbox_inches="tight")
plt.close()
print(f"Figura de Média salva em: {mean_plot_path}")

print("\nProcesso finalizado com sucesso!")

In [ ]:
# =============================================================================
# SCRIPT PARA COMPARAR MÉDIA E VARIABILIDADE (CV) DE SST, DLI, CHL-A E DHW
# Objetivo: Criar duas figuras compostas separadas para Média e CV,
#           com comparação entre arcos usando sítios como réplicas.
# Versão 4.3 - Correção definitiva de nome de colunas e bug no gráfico de CV.
# =============================================================================

####Apenas retira DHW

import os
import glob
import re
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import gc
from datetime import datetime
import warnings

# Silenciar avisos
warnings.filterwarnings("ignore", category=UserWarning, message="Sending large graph of Dask arrays")
warnings.filterwarnings("ignore", category=RuntimeWarning, message="Mean of empty slice")

# ==============================
# 1. CONFIGURAÇÕES PRINCIPAIS
# ==============================
SITES_CSV_FILE = r"C:\Users\rbfra\OneDrive\########CEBIMAR\####PROJETOS\#####Coral trade offs\sites_list_full.csv"
SST_DIR   = r"E:\remote sensing\CRW_SST_FULL"
MODIS_DIR = r"E:\remote sensing\MODIS_DATA_FULL"
DHW_DIR   = r"E:\remote sensing\CRW_DHW_FULL"
OUTPUT_DIR = r"C:\Users\rbfra\OneDrive\########PUBLICACOES\############Menezes et al. Mus his distribution and abundance Abrolhos\########NEW RESULTS\output_Mean_CV_Comparison_FINAL_semDHW"

# Períodos para teste rápido - use os períodos completos para a análise final
#SST_PERIOD   = (2007, 2008)
#DHW_PERIOD   = (2007, 2008)
#MODIS_PERIOD = (2007, 2008)

# Períodos completos (descomente para usar)
SST_PERIOD   = (1985, 2008)
DHW_PERIOD   = (1985, 2008)
MODIS_PERIOD = (2002, 2008)

SST_PATTERN   = 'coraltemp_v3.1_*.nc'
DHW_PATTERN   = '*.nc'
KD490_PATTERN = 'AQUA_MODIS.*.L3m.DAY.KD.Kd_490.4km.nc'
PAR_PATTERN   = 'AQUA_MODIS.*.L3m.DAY.PAR.par.4km.nc'
CHL_PATTERN   = 'AQUA_MODIS.*.L3m.DAY.CHL.chlor_a.4km.nc'

WINDOW_SIZES = [2, 7, 30, 180, 365]
KDPAR_GATTUSO_A, KDPAR_GATTUSO_B, KDPAR_GATTUSO_C = 0.0665, 0.874, 0.00121

# ==============================
# 2. FUNÇÕES AUXILIARES (sem alterações)
# ==============================
def preprocess_with_time(ds, filename):
    basename = os.path.basename(filename)
    match = re.search(r'(\d{8})', basename)
    if match:
        try:
            date_val = np.datetime64(datetime.strptime(match.group(1), '%Y%m%d'))
            if 'time' in ds.dims: return ds.assign_coords(time=[date_val])
            else: return ds.expand_dims(time=[date_val])
        except ValueError: return ds
    return ds

def load_and_extract_timeseries(base_dir, pattern, period, var_name_options, lats, lons):
    search_path = os.path.join(base_dir, pattern); print(f"\nProcurando arquivos em: {search_path}")
    files_to_process = []
    for f in sorted(glob.glob(search_path)):
        match = re.search(r'(\d{4})\d{4}', os.path.basename(f))
        if match and period[0] <= int(match.group(1)) <= period[1]: files_to_process.append(f)
    if not files_to_process: print(f"AVISO: Nenhum arquivo para '{pattern}' no período {period}."); return None
    print(f"Encontrados {len(files_to_process)} arquivos.")
    datasets_preprocessed = []
    for f_path in files_to_process:
        try:
            with xr.open_dataset(f_path, chunks='auto') as ds:
                datasets_preprocessed.append(preprocess_with_time(ds, f_path))
        except Exception: continue
    if not datasets_preprocessed: print("AVISO: Nenhum arquivo pôde ser pré-processado."); return None
    try:
        ds_virtual = xr.combine_by_coords(datasets_preprocessed, compat='override', coords='all', join='override', combine_attrs='override')
        var_name = next((v for v in var_name_options if v in ds_virtual.data_vars), None)
        if not var_name: print(f"AVISO: Variáveis {var_name_options} não encontradas."); del ds_virtual, datasets_preprocessed; gc.collect(); return None
        timeseries_lazy = ds_virtual[var_name].sel(lat=lats, lon=lons, method='nearest')
        timeseries_loaded = timeseries_lazy.load().astype('float32').sortby('time')
        print(f"Série '{var_name}' carregada. Shape: {timeseries_loaded.shape}")
        del ds_virtual, datasets_preprocessed, timeseries_lazy; gc.collect()
        return timeseries_loaded
    except Exception as e: print(f"ERRO ao processar '{pattern}': {e}"); return None

def calculate_metrics_for_timeseries(timeseries_da, window_sizes, var_name):
    if timeseries_da is None: return {}
    print(f"\nCalculando métricas para: {var_name.upper()}")
    results = {}
    mean_all = timeseries_da.mean('time', skipna=True)
    std_all = timeseries_da.std('time', skipna=True)
    results['mean_ALL'] = mean_all.values
    results['cv_ALL'] = ((std_all / (mean_all + 1e-9)) * 100).values
    for window in window_sizes:
        min_p = max(2, int(window * 0.25))
        rolling_mean = timeseries_da.rolling(time=window, min_periods=min_p, center=True).mean()
        rolling_std = timeseries_da.rolling(time=window, min_periods=min_p, center=True).std()
        cv_ts = (rolling_std / (rolling_mean + 1e-9)) * 100
        results[f'cv_{window}'] = cv_ts.mean('time', skipna=True).values
    return results

def calculate_dhw_frequency(timeseries_da):
    if timeseries_da is None: return {}
    print("\nCalculando frequência de DHW > 4...")
    valid_data = timeseries_da.where(timeseries_da.notnull())
    freq = (valid_data > 4).mean('time', skipna=True) * 100
    mean_dhw = valid_data.mean('time', skipna=True)
    return {'mean_DHW>4_freq': freq.values, 'mean_ALL_dhw': mean_dhw.values}


# ==============================
# 3. PROCESSAMENTO PRINCIPAL (COM CORREÇÃO DE NOMES)
# ==============================
if __name__ == "__main__":
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    try:
        # Lê o CSV removendo possíveis caracteres BOM
        df_sites = pd.read_csv(SITES_CSV_FILE, sep=';', encoding='utf-8-sig')  # Note o encoding alterado
        
        # <--- CORREÇÃO PARA SEU CSV COM BOM --->
        # 1. Limpa os nomes das colunas (remove BOM e espaços)
        df_sites.columns = df_sites.columns.str.strip()
        
        # 2. Corrige especificamente a primeira coluna que tem BOM
        df_sites.columns = [col.replace('ï»¿', '') for col in df_sites.columns]
        
        # 3. Mapeamento dos nomes do seu CSV
        rename_map = {
            'Reef_name': 'REEF',  # Agora sem o BOM
            'Site_name': 'SITE',
            'Latitude': 'LAT',
            'Longitude': 'LONG',
            'Depth_m': 'DEPTH',
            'Arch': 'ARCH'
        }
        
        # 4. Renomeia apenas as colunas que existem
        rename_map = {k: v for k, v in rename_map.items() if k in df_sites.columns}
        df_sites.rename(columns=rename_map, inplace=True)

        # 5. Verificação final
        required_cols = ['REEF', 'ARCH', 'LAT', 'LONG']
        if not all(col in df_sites.columns for col in required_cols):
            missing = [col for col in required_cols if col not in df_sites.columns]
            raise ValueError(f"ERRO CRÍTICO: Colunas ausentes: {missing}. Colunas encontradas: {df_sites.columns.tolist()}")

        # Limpeza dos dados
        for col in df_sites.select_dtypes(['object']).columns: 
            df_sites[col] = df_sites[col].str.strip()
            
        print(f"Arquivo carregado com sucesso. Colunas finais: {df_sites.columns.tolist()}")
        
    except Exception as e: 
        print(f"ERRO FATAL: {str(e)}")
        exit()

    # Preparação dos DataArrays (agora usando os nomes padronizados)
    lats = xr.DataArray(df_sites['LAT'].values, dims="site")
    lons = xr.DataArray(df_sites['LONG'].values, dims="site")
    depths = xr.DataArray(df_sites['DEPTH'].values, dims="site")
    df_results = df_sites.copy()

    # --- Processamento SST ---
    sst_timeseries = load_and_extract_timeseries(SST_DIR, SST_PATTERN, SST_PERIOD, ['analysed_sst', 'sea_surface_temperature'], lats, lons)
    if sst_timeseries is not None:
        sst_metrics = calculate_metrics_for_timeseries(sst_timeseries, WINDOW_SIZES, 'SST')
        for key, values in sst_metrics.items(): df_results[f'sst_{key}'] = values
        del sst_timeseries, sst_metrics; gc.collect()

    # --- Processamento DLI ---
    kd490_timeseries = load_and_extract_timeseries(MODIS_DIR, KD490_PATTERN, MODIS_PERIOD, ['Kd_490'], lats, lons)
    par_timeseries = load_and_extract_timeseries(MODIS_DIR, PAR_PATTERN, MODIS_PERIOD, ['par'], lats, lons)
    if kd490_timeseries is not None and par_timeseries is not None:
        kd490_aligned, par_aligned = xr.align(kd490_timeseries, par_timeseries, join='inner')
        kdpar = KDPAR_GATTUSO_A + KDPAR_GATTUSO_B * kd490_aligned - KDPAR_GATTUSO_C / (kd490_aligned + 1e-9)
        dli_timeseries = par_aligned * np.exp(-xr.where(kdpar <= 0, np.nan, kdpar) * depths)
        dli_metrics = calculate_metrics_for_timeseries(dli_timeseries, WINDOW_SIZES, 'DLI')
        for key, values in dli_metrics.items(): df_results[f'dli_{key}'] = values
        del kd490_timeseries, par_timeseries, dli_timeseries, dli_metrics; gc.collect()

    # --- Processamento Clorofila-a ---
    chl_timeseries = load_and_extract_timeseries(MODIS_DIR, CHL_PATTERN, MODIS_PERIOD, ['chlor_a'], lats, lons)
    if chl_timeseries is not None:
        chl_metrics = calculate_metrics_for_timeseries(chl_timeseries, WINDOW_SIZES, 'CHL')
        for key, values in chl_metrics.items(): df_results[f'chl_{key}'] = values
        del chl_timeseries, chl_metrics; gc.collect()

    # --- Processamento DHW ---
    dhw_timeseries = load_and_extract_timeseries(DHW_DIR, DHW_PATTERN, DHW_PERIOD, ['degree_heating_week'], lats, lons)
    if dhw_timeseries is not None:
        dhw_metrics_freq = calculate_dhw_frequency(dhw_timeseries)
        for key, values in dhw_metrics_freq.items(): df_results[key] = values
        
        print("\nCalculando métricas de CV para DHW...")
        dhw_metrics_cv = calculate_metrics_for_timeseries(dhw_timeseries, WINDOW_SIZES, 'DHW')
        for key, values in dhw_metrics_cv.items():
            if 'mean_ALL' not in key:
                 df_results[f'dhw_{key}'] = values
        del dhw_timeseries, dhw_metrics_freq, dhw_metrics_cv; gc.collect()


    results_path = os.path.join(OUTPUT_DIR, "summary_results_all_metrics.xlsx")
    df_results.to_excel(results_path, index=False)
    print(f"\nPlanilha com todos os resultados salva em: {results_path}")

# ========================================================================
# 4. PREPARAÇÃO E GERAÇÃO DOS GRÁFICOS (SOLUÇÃO DEFINITIVA PARA AJUSTE DE EIXO)
# ========================================================================
print("\nIniciando a geração dos gráficos...")
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("talk")

# ---------------------------------------------------------------------
# Função de ajuste de eixo Y - Versão Final e Robusta
# Recalcula o erro manualmente para garantir precisão total.
# ---------------------------------------------------------------------
def adjust_yaxis_definitive(g, df, x_var, y_var, col_var, row_var, padding_factor=0.2):
    """
    Ajusta o eixo Y de cada subplot em um FacetGrid recalculando os limites
    (média ± desvio padrão) a partir do dataframe original.
    Isso garante que as barras de erro nunca sejam cortadas.
    """
    # Itera sobre cada subplot usando os eixos do FacetGrid
    for i, ax in enumerate(g.axes.flat):
        if not ax.has_data():
            continue
            
        # Determina a qual linha (row) e coluna (col) este eixo pertence
        row_val = g.row_names[i // g.axes.shape[1]]
        col_val = g.col_names[i % g.axes.shape[1]]

        # Filtra o DataFrame para obter apenas os dados deste subplot específico
        subplot_data = df[(df[row_var] == row_val) & (df[col_var] == col_val)]

        if subplot_data.empty:
            continue

        # Agrupa os dados pelo que está no eixo X (ex: por 'REEF' ou 'Window')
        grouped = subplot_data.groupby(x_var)[y_var]
        
        # Calcula a média e o desvio padrão para cada grupo
        means = grouped.mean()
        stds = grouped.std()
        
        # O limite inferior é a média menos o desvio padrão, o superior é a média mais o desvio padrão
        lower_bounds = means - stds
        upper_bounds = means + stds

        # Encontra o mínimo e o máximo absolutos entre todos os limites
        data_min = np.nanmin(lower_bounds)
        data_max = np.nanmax(upper_bounds)
        
        # Se não houver dados válidos, pula para o próximo eixo
        if not (np.isfinite(data_min) and np.isfinite(data_max)):
            continue
            
        # Calcula o padding para dar "respiro" ao gráfico
        data_range = data_max - data_min
        padding = data_range * padding_factor if data_range > 0 else max(abs(data_max * 0.1), 0.1)
        
        # Define os novos limites do eixo Y com o padding
        ax.set_ylim(data_min - padding, data_max + padding)


# ---------------------------------------------------------------------
# Figura 1 – Coeficiente de Variação (CV %)
# ---------------------------------------------------------------------

# <--- ALTERAÇÃO AQUI --->
# A variável "dhw" foi removida desta lista para que não apareça no gráfico de CV.
variables_for_cv = ["sst", "dli", "chl"]

# Inclui explicitamente as colunas `_cv_ALL` e as de janela
cv_cols = []
for v in variables_for_cv:
    cv_cols.append(f"{v}_cv_ALL")
    for w in WINDOW_SIZES:
        cv_cols.append(f"{v}_cv_{w}")

cv_cols = [c for c in cv_cols if c in df_results.columns]

df_cv_melted = df_results.melt(id_vars=["ARCH"], value_vars=cv_cols, var_name="Metric", value_name="Value")

# <--- INÍCIO DA CORREÇÃO --->
# Reescreve a função para retornar um DataFrame, que é mais seguro para atribuição
def parse_metric_to_df(metric_series):
    """
    Processa a série de métricas e retorna um DataFrame com as colunas 'Variable' e 'Window'.
    """
    parts = metric_series.str.split('_cv_', n=1, expand=True)
    df = pd.DataFrame({
        'Variable': parts[0].str.upper().str.replace("CHL", "Chl-a"),
        'Window': parts[1]
    })
    return df

# Atribui as novas colunas fazendo um join/merge, que é a forma mais robusta
df_cv_melted = df_cv_melted.join(parse_metric_to_df(df_cv_melted['Metric']))
# <--- FIM DA CORREÇÃO --->

df_cv_melted.dropna(subset=["Value", "ARCH"], inplace=True)

# Atualiza a ordem do eixo X para incluir 'ALL'
window_order = [str(w) for w in WINDOW_SIZES] + ['ALL']
df_cv_melted['Window'] = pd.Categorical(df_cv_melted['Window'], categories=window_order, ordered=True)
df_cv_melted = df_cv_melted.sort_values('Window')

g_cv = sns.catplot(
    data=df_cv_melted,
    x="Window", y="Value", col="Variable", row="ARCH",
    kind="bar", errorbar="sd", capsize=0.08, palette="viridis",
    height=4, aspect=1.2, sharey=False,
    order=window_order
)

# Chamando a função de ajuste
adjust_yaxis_definitive(g_cv, df_cv_melted, x_var="Window", y_var="Value", col_var="Variable", row_var="ARCH")

g_cv.fig.suptitle("Coeficiente de Variação (CV %) por Janela de Tempo e Arco", y=1.02, fontsize=20)
g_cv.set_axis_labels("Janela de Tempo (dias) / Total (ALL)", "CV (%)")
g_cv.set_titles(row_template="Arco: {row_name}", col_template="{col_name}")
g_cv.fig.tight_layout(rect=[0, 0, 1, 0.96])
cv_plot_path = os.path.join(OUTPUT_DIR, "figura_composta_CV.png")
plt.savefig(cv_plot_path, dpi=300, bbox_inches="tight")
plt.close()
print(f"Figura de CV salva em: {cv_plot_path}")

# ---------------------------------------------------------------------
# Figura 2 – Média Global por Recife (SEM ALTERAÇÃO)
# ---------------------------------------------------------------------
mean_cols_map = {
    "sst_mean_ALL": "SST (°C)",
    "dli_mean_ALL": "DLI (mol/m²/d)",
    "chl_mean_ALL": "Chl-a (mg/m³)",
    "mean_DHW>4_freq": "Freq. DHW > 4 (%)"
}
existing_mean_cols = [k for k in mean_cols_map if k in df_results.columns]

df_mean_melted = (
    df_results
    .melt(id_vars=["REEF", "ARCH"], value_vars=existing_mean_cols,
          var_name="Variable", value_name="Value")
    .assign(Variable=lambda d: d["Variable"].map(mean_cols_map))
    .dropna(subset=["Value", "ARCH", "REEF"])
)

g_mean = sns.catplot(
    data=df_mean_melted,
    x="REEF", y="Value", col="Variable", row="ARCH",
    kind="bar", palette="muted", errorbar="sd",
    capsize=0.08, height=5, aspect=1.5,
    sharex=False, sharey=False
)

# Chamando a nova função de ajuste
adjust_yaxis_definitive(g_mean, df_mean_melted, x_var="REEF", y_var="Value", col_var="Variable", row_var="ARCH")

g_mean.fig.suptitle("Média das Variáveis por Recife e Arco (Sítios como réplicas)", fontsize=24, y=1.03)
g_mean.set_axis_labels("Recife", "Valor Médio")
g_mean.set_titles(row_template="Arco: {row_name}", col_template="{col_name}")
g_mean.set_xticklabels(rotation=45, ha="right")
g_mean.fig.tight_layout(rect=[0, 0, 1, 0.97])
mean_plot_path = os.path.join(OUTPUT_DIR, "figura_composta_Media_por_Reef.png")
plt.savefig(mean_plot_path, dpi=300, bbox_inches="tight")
plt.close()
print(f"Figura de Média salva em: {mean_plot_path}")

print("\nProcesso finalizado com sucesso!")


In [ ]:
# =============================================================================
# SCRIPT PARA COMPARAR MÉDIA E VARIABILIDADE (CV) DE SST, DLI, CHL-A E DHW
# Objetivo: Criar figuras compostas onde DLI é agrupado por habitat (com hachuras)
#           e as demais variáveis são globais por recife. Cores representam recifes.
# Versão 6.0 - Hachuras para Habitat, Cores para Recife, Legendas Personalizadas.
# =============================================================================

##Retira DHW E retorna distinção entre habitats

import os
import glob
import re
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import gc
from datetime import datetime
import warnings

# (O restante da seção 1 e 2 - Configurações e Funções Auxiliares - permanece o mesmo)
# ... (código anterior omitido por brevidade) ...

# ==============================
# 1. CONFIGURAÇÕES PRINCIPAIS (Sem alterações)
# ==============================
SITES_CSV_FILE = r"C:\Users\rbfra\OneDrive\########CEBIMAR\####PROJETOS\#####Coral trade offs\sites_list_full.csv"
SST_DIR   = r"H:\remote sensing\CRW_SST_FULL"
MODIS_DIR = r"H:\remote sensing\MODIS_DATA_FULL"
DHW_DIR   = r"H:\remote sensing\CRW_DHW_FULL"
OUTPUT_DIR = r"C:\Users\rbfra\OneDrive\########PUBLICACOES\############Menezes et al. Mus his distribution and abundance Abrolhos\########NEW RESULTS\output_Mean_CV_Comparison_PQP"

SST_PERIOD   = (2007, 2008)
DHW_PERIOD   = (2007, 2008)
MODIS_PERIOD = (2007, 2008)

SST_PATTERN   = 'coraltemp_v3.1_*.nc'
DHW_PATTERN   = '*.nc'
KD490_PATTERN = 'AQUA_MODIS.*.L3m.DAY.KD.Kd_490.4km.nc'
PAR_PATTERN   = 'AQUA_MODIS.*.L3m.DAY.PAR.par.4km.nc'
CHL_PATTERN   = 'AQUA_MODIS.*.L3m.DAY.CHL.chlor_a.4km.nc'

WINDOW_SIZES = [2, 7, 30, 180, 365]
KDPAR_GATTUSO_A, KDPAR_GATTUSO_B, KDPAR_GATTUSO_C = 0.0665, 0.874, 0.00121

# ==============================
# 2. FUNÇÕES AUXILIARES (Sem alterações)
# ==============================
# (Funções `preprocess_with_time`, `load_and_extract_timeseries`, etc. permanecem aqui)
def preprocess_with_time(ds, filename):
    basename = os.path.basename(filename)
    match = re.search(r'(\d{8})', basename)
    if match:
        try:
            date_val = np.datetime64(datetime.strptime(match.group(1), '%Y%m%d'))
            if 'time' in ds.dims: return ds.assign_coords(time=[date_val])
            else: return ds.expand_dims(time=[date_val])
        except ValueError: return ds
    return ds

def load_and_extract_timeseries(base_dir, pattern, period, var_name_options, lats, lons):
    search_path = os.path.join(base_dir, pattern); print(f"\nProcurando arquivos em: {search_path}")
    files_to_process = []
    for f in sorted(glob.glob(search_path)):
        match = re.search(r'(\d{4})\d{4}', os.path.basename(f))
        if match and period[0] <= int(match.group(1)) <= period[1]: files_to_process.append(f)
    if not files_to_process: print(f"AVISO: Nenhum arquivo para '{pattern}' no período {period}."); return None
    print(f"Encontrados {len(files_to_process)} arquivos.")
    datasets_preprocessed = []
    for f_path in files_to_process:
        try:
            with xr.open_dataset(f_path, chunks='auto') as ds:
                datasets_preprocessed.append(preprocess_with_time(ds, f_path))
        except Exception: continue
    if not datasets_preprocessed: print("AVISO: Nenhum arquivo pôde ser pré-processado."); return None
    try:
        ds_virtual = xr.combine_by_coords(datasets_preprocessed, compat='override', coords='all', join='override', combine_attrs='override')
        var_name = next((v for v in var_name_options if v in ds_virtual.data_vars), None)
        if not var_name: print(f"AVISO: Variáveis {var_name_options} não encontradas."); del ds_virtual, datasets_preprocessed; gc.collect(); return None
        timeseries_lazy = ds_virtual[var_name].sel(lat=lats, lon=lons, method='nearest')
        timeseries_loaded = timeseries_lazy.load().astype('float32').sortby('time')
        print(f"Série '{var_name}' carregada. Shape: {timeseries_loaded.shape}")
        del ds_virtual, datasets_preprocessed, timeseries_lazy; gc.collect()
        return timeseries_loaded
    except Exception as e: print(f"ERRO ao processar '{pattern}': {e}"); return None

def calculate_metrics_for_timeseries(timeseries_da, window_sizes, var_name):
    if timeseries_da is None: return {}
    print(f"\nCalculando métricas para: {var_name.upper()}")
    results = {}
    mean_all = timeseries_da.mean('time', skipna=True)
    std_all = timeseries_da.std('time', skipna=True)
    results['mean_ALL'] = mean_all.values
    results['cv_ALL'] = ((std_all / (mean_all + 1e-9)) * 100).values
    for window in window_sizes:
        min_p = max(2, int(window * 0.25))
        rolling_mean = timeseries_da.rolling(time=window, min_periods=min_p, center=True).mean()
        rolling_std = timeseries_da.rolling(time=window, min_periods=min_p, center=True).std()
        cv_ts = (rolling_std / (rolling_mean + 1e-9)) * 100
        results[f'cv_{window}'] = cv_ts.mean('time', skipna=True).values
    return results

def calculate_dhw_frequency(timeseries_da):
    if timeseries_da is None: return {}
    print("\nCalculando frequência de DHW > 4...")
    valid_data = timeseries_da.where(timeseries_da.notnull())
    freq = (valid_data > 4).mean('time', skipna=True) * 100
    mean_dhw = valid_data.mean('time', skipna=True)
    return {'mean_DHW>4_freq': freq.values, 'mean_ALL_dhw': mean_dhw.values}

# ==============================
# 3. PROCESSAMENTO PRINCIPAL (Sem alterações na lógica)
# ==============================
if __name__ == "__main__":
    # (Toda a lógica de carregamento e processamento de dados permanece a mesma)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    try:
        df_sites = pd.read_csv(SITES_CSV_FILE, sep=';', encoding='utf-8-sig')
        df_sites.columns = df_sites.columns.str.strip().str.replace('ï»¿', '')
        
        rename_map = {
            'Reef_name': 'REEF', 'Site_name': 'SITE', 'Latitude': 'LAT',
            'Longitude': 'LONG', 'Depth_m': 'DEPTH', 'Arch': 'ARCH',
            'HAB_consol': 'HAB' # Assumindo que este é o nome da coluna de habitat
        }
        
        df_sites.rename(columns=rename_map, inplace=True)
        required_cols = ['REEF', 'ARCH', 'LAT', 'LONG', 'HAB']
        if not all(col in df_sites.columns for col in required_cols):
            missing = [col for col in required_cols if col not in df_sites.columns]
            raise ValueError(f"ERRO CRÍTICO: Colunas ausentes: {missing}. Colunas encontradas: {df_sites.columns.tolist()}")

        for col in df_sites.select_dtypes(['object']).columns: 
            df_sites[col] = df_sites[col].str.strip()
            
        print(f"Arquivo carregado. Colunas finais: {df_sites.columns.tolist()}")
        
    except Exception as e: 
        print(f"ERRO FATAL: {str(e)}")
        exit()

    lats = xr.DataArray(df_sites['LAT'].values, dims="site")
    lons = xr.DataArray(df_sites['LONG'].values, dims="site")
    depths = xr.DataArray(df_sites['DEPTH'].values, dims="site")
    df_results = df_sites.copy()
    
    # ... (cálculo de sst, dli, chl, dhw) ...
    sst_timeseries = load_and_extract_timeseries(SST_DIR, SST_PATTERN, SST_PERIOD, ['analysed_sst', 'sea_surface_temperature'], lats, lons)
    if sst_timeseries is not None:
        sst_metrics = calculate_metrics_for_timeseries(sst_timeseries, WINDOW_SIZES, 'SST')
        for key, values in sst_metrics.items(): df_results[f'sst_{key}'] = values
        del sst_timeseries, sst_metrics; gc.collect()

    kd490_timeseries = load_and_extract_timeseries(MODIS_DIR, KD490_PATTERN, MODIS_PERIOD, ['Kd_490'], lats, lons)
    par_timeseries = load_and_extract_timeseries(MODIS_DIR, PAR_PATTERN, MODIS_PERIOD, ['par'], lats, lons)
    if kd490_timeseries is not None and par_timeseries is not None:
        kd490_aligned, par_aligned = xr.align(kd490_timeseries, par_timeseries, join='inner')
        kdpar = KDPAR_GATTUSO_A + KDPAR_GATTUSO_B * kd490_aligned - KDPAR_GATTUSO_C / (kd490_aligned + 1e-9)
        dli_timeseries = par_aligned * np.exp(-xr.where(kdpar <= 0, np.nan, kdpar) * depths)
        dli_metrics = calculate_metrics_for_timeseries(dli_timeseries, WINDOW_SIZES, 'DLI')
        for key, values in dli_metrics.items(): df_results[f'dli_{key}'] = values
        del kd490_timeseries, par_timeseries, dli_timeseries, dli_metrics; gc.collect()

    chl_timeseries = load_and_extract_timeseries(MODIS_DIR, CHL_PATTERN, MODIS_PERIOD, ['chlor_a'], lats, lons)
    if chl_timeseries is not None:
        chl_metrics = calculate_metrics_for_timeseries(chl_timeseries, WINDOW_SIZES, 'CHL')
        for key, values in chl_metrics.items(): df_results[f'chl_{key}'] = values
        del chl_timeseries, chl_metrics; gc.collect()

    dhw_timeseries = load_and_extract_timeseries(DHW_DIR, DHW_PATTERN, DHW_PERIOD, ['degree_heating_week'], lats, lons)
    if dhw_timeseries is not None:
        dhw_metrics_freq = calculate_dhw_frequency(dhw_timeseries)
        for key, values in dhw_metrics_freq.items(): df_results[key] = values
        
        dhw_metrics_cv = calculate_metrics_for_timeseries(dhw_timeseries, WINDOW_SIZES, 'DHW')
        for key, values in dhw_metrics_cv.items():
            if 'mean_ALL' not in key: df_results[f'dhw_{key}'] = values
        del dhw_timeseries, dhw_metrics_freq, dhw_metrics_cv; gc.collect()

    results_path = os.path.join(OUTPUT_DIR, "summary_results_all_metrics.xlsx")
    df_results.to_excel(results_path, index=False)
    print(f"\nPlanilha com todos os resultados salva em: {results_path}")

# ========================================================================
# 4. PREPARAÇÃO E GERAÇÃO DOS GRÁFICOS (SEÇÃO COMPLETAMENTE REFEITA)
# ========================================================================
print("\nIniciando a geração dos gráficos...")
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("talk")

# ---------------------------------------------------------------------
# Função de ajuste de eixo Y (A mesma da versão anterior, funciona com hue)
# ---------------------------------------------------------------------
def adjust_yaxis_definitive(g, df, x_var, y_var, col_var, row_var, hue_var=None, padding_factor=0.2):
    for i, ax in enumerate(g.axes.flat):
        if not ax.has_data(): continue
        row_val = g.row_names[i // g.axes.shape[1]]
        col_val = g.col_names[i % g.axes.shape[1]]
        subplot_data = df[(df[row_var] == row_val) & (df[col_var] == col_val)]
        if subplot_data.empty: continue
        group_keys = [x_var]
        if hue_var and hue_var in subplot_data: group_keys.append(hue_var)
        grouped = subplot_data.groupby(group_keys)[y_var]
        means, stds = grouped.mean(), grouped.std()
        lower_bounds, upper_bounds = means - stds, means + stds
        data_min, data_max = np.nanmin(lower_bounds), np.nanmax(upper_bounds)
        if not (np.isfinite(data_min) and np.isfinite(data_max)): continue
        data_range = data_max - data_min
        padding = data_range * padding_factor if data_range > 0 else max(abs(data_max * 0.1), 0.1)
        ax.set_ylim(data_min - padding, data_max + padding)

# ---------------------------------------------------------------------
# Figura 2 – Média Global por Recife (com hachuras para DLI)
# ---------------------------------------------------------------------
# --- Passo 1: Preparar os dados (mesma lógica de antes) ---
mean_cols_map = {
    "sst_mean_ALL": "SST (°C)", "chl_mean_ALL": "Chl-a (mg/m³)",
    "mean_DHW>4_freq": "Freq. DHW > 4 (%)"
}
dli_mean_col_map = {"dli_mean_ALL": "DLI (mol/m²/d)"}

df_dli_mean = df_results.melt(
    id_vars=["REEF", "ARCH", "HAB"], value_vars=list(dli_mean_col_map.keys()),
    var_name="Variable", value_name="Value"
)
df_others_mean = df_results.melt(
    id_vars=["REEF", "ARCH"], value_vars=list(mean_cols_map.keys()),
    var_name="Variable", value_name="Value"
).assign(HAB='Geral')

df_mean_melted = pd.concat([df_dli_mean, df_others_mean], ignore_index=True)
full_map = {**mean_cols_map, **dli_mean_col_map}
df_mean_melted['Variable'] = df_mean_melted['Variable'].map(full_map)
df_mean_melted.dropna(subset=["Value", "ARCH", "REEF", "HAB"], inplace=True)

# --- Passo 2: Definir cores e padrões ---
# Paleta de cores para os recifes (garante consistência)
unique_reefs = sorted(df_mean_melted['REEF'].unique())
reef_palette = dict(zip(unique_reefs, sns.color_palette("muted", n_colors=len(unique_reefs))))

# Padrões de hachura para os habitats
hatch_map = {'RR': '//', 'PA': 'xx', 'TP': '..', 'Geral': None}
habitat_order = ['RR', 'PA', 'TP', 'Geral'] # Define a ordem do hue

# --- Passo 3: Gerar o gráfico base com `hue` ---
g_mean = sns.catplot(
    data=df_mean_melted,
    x="REEF", y="Value", col="Variable", row="ARCH",
    hue="HAB", hue_order=habitat_order,
    kind="bar", errorbar="sd",
    capsize=0.08, height=5, aspect=1.5,
    sharex=False, sharey=False,
    dodge=True, # Garante que as barras sejam agrupadas
    legend=False # Desativa a legenda automática
)

# --- Passo 4: Pós-processamento para aplicar cores e hachuras ---
reef_order = g_mean.x_names # Ordem dos recifes no eixo X

for ax in g_mean.axes.flat:
    if not ax.has_data(): continue
    
    # ax.containers armazena os grupos de barras (um por habitat)
    for i, container in enumerate(ax.containers):
        # Pega o rótulo do habitat para este container
        habitat_label = habitat_order[i]
        current_hatch = hatch_map.get(habitat_label)

        # Itera sobre cada barra individual no container
        for j, bar in enumerate(container.patches):
            reef_name = reef_order[j]
            # Define a cor da barra com base no recife
            bar.set_color(reef_palette[reef_name])
            # Aplica a hachura se houver uma definida
            if current_hatch:
                bar.set_hatch(current_hatch)
                bar.set_edgecolor('white') # Melhora a visibilidade da hachura

# --- Passo 5: Criar legendas personalizadas ---
# Legenda de Cores (Recifes)
reef_patches = [mpatches.Patch(color=color, label=reef) for reef, color in reef_palette.items()]
legend1 = plt.legend(handles=reef_patches, title="Recife", bbox_to_anchor=(1.05, 1), loc='upper left')

# Legenda de Hachuras (Habitats)
# Filtra para não incluir 'Geral' na legenda de habitat
hatch_patches = [mpatches.Patch(facecolor='grey', hatch=hatch, label=hab, edgecolor='white') 
                 for hab, hatch in hatch_map.items() if hab != 'Geral']
plt.legend(handles=hatch_patches, title="Habitat (DLI)", bbox_to_anchor=(1.05, 0.7), loc='upper left')

# Adiciona a primeira legenda de volta, pois a segunda a substitui
plt.gca().add_artist(legend1)

# --- Passo 6: Ajustes finais e salvamento ---
adjust_yaxis_definitive(g_mean, df_mean_melted, x_var="REEF", y_var="Value", col_var="Variable", row_var="ARCH", hue_var="HAB")

g_mean.fig.suptitle("Média das Variáveis por Recife, Arco e Habitat (DLI)", fontsize=24, y=1.03)
g_mean.set_axis_labels("Recife", "Valor Médio")
g_mean.set_titles(row_template="Arco: {row_name}", col_template="{col_name}")
g_mean.set_xticklabels(rotation=45, ha="right")
g_mean.fig.tight_layout(rect=[0, 0, 0.85, 0.97]) # Ajusta para caber as legendas
mean_plot_path = os.path.join(OUTPUT_DIR, "figura_composta_Media_Hachurada.png")
plt.savefig(mean_plot_path, dpi=300, bbox_inches="tight")
plt.close()
print(f"Figura de Média salva em: {mean_plot_path}")


# A Figura de CV pode ser feita seguindo exatamente a mesma lógica se desejado,
# ou mantida como na versão anterior, se a distinção de habitat não for necessária lá.
# Por consistência, vamos aplicar a mesma lógica à figura de CV.

# ---------------------------------------------------------------------
# Figura 1 – Coeficiente de Variação (CV %) (com hachuras para DLI)
# ---------------------------------------------------------------------
# (A lógica de preparação de dados e plotagem é idêntica à da Figura de Média)
def parse_metric_to_df(metric_series):
    parts = metric_series.str.split('_cv_', n=1, expand=True)
    df = pd.DataFrame({'Variable': parts[0].str.upper().str.replace("CHL", "Chl-a"), 'Window': parts[1]})
    return df

vars_others = ["sst", "chl"] # DHW removido daqui
var_dli = "dli"
dli_cv_cols = [f"{var_dli}_cv_{w}" for w in WINDOW_SIZES] + [f"{var_dli}_cv_ALL"]
df_dli_cv = df_results.melt(id_vars=["ARCH", "HAB"], value_vars=dli_cv_cols, var_name="Metric", value_name="Value")
others_cv_cols = []
for v in vars_others: others_cv_cols.extend([f"{v}_cv_{w}" for w in WINDOW_SIZES] + [f"{v}_cv_ALL"])
df_others_cv = df_results.melt(id_vars=["ARCH"], value_vars=others_cv_cols, var_name="Metric", value_name="Value").assign(HAB='Geral')
df_cv_melted = pd.concat([df_dli_cv, df_others_cv], ignore_index=True)
df_cv_melted = df_cv_melted.join(parse_metric_to_df(df_cv_melted['Metric']))
df_cv_melted.dropna(subset=["Value", "ARCH", "HAB"], inplace=True)
window_order = [str(w) for w in WINDOW_SIZES] + ['ALL']
df_cv_melted['Window'] = pd.Categorical(df_cv_melted['Window'], categories=window_order, ordered=True)
df_cv_melted = df_cv_melted.sort_values('Window')

# Paleta de cores para os arcos (pois não há 'REEF' neste gráfico)
unique_arches = sorted(df_cv_melted['ARCH'].unique())
arch_palette = dict(zip(unique_arches, sns.color_palette("viridis", n_colors=len(unique_arches))))

g_cv = sns.catplot(
    data=df_cv_melted,
    x="Window", y="Value", col="Variable", row="ARCH",
    hue="HAB", hue_order=habitat_order,
    kind="bar", errorbar="sd", capsize=0.08,
    height=4, aspect=1.2, sharey=False, dodge=True,
    legend=False
)

# Pós-processamento para CV
for i, ax in enumerate(g_cv.axes.flat):
    if not ax.has_data(): continue
    arch_name = g_cv.row_names[i // g_cv.axes.shape[1]]
    for j, container in enumerate(ax.containers):
        habitat_label = habitat_order[j]
        current_hatch = hatch_map.get(habitat_label)
        container.set_color(arch_palette[arch_name]) # Cor por Arco
        if current_hatch:
            for bar in container.patches:
                bar.set_hatch(current_hatch)
                bar.set_edgecolor('white')

# Legendas para CV
arch_patches = [mpatches.Patch(color=color, label=arch) for arch, color in arch_palette.items()]
legend1_cv = plt.legend(handles=arch_patches, title="Arco", bbox_to_anchor=(1.05, 1), loc='upper left')
hatch_patches_cv = [mpatches.Patch(facecolor='grey', hatch=hatch, label=hab, edgecolor='white') 
                    for hab, hatch in hatch_map.items() if hab != 'Geral']
plt.legend(handles=hatch_patches_cv, title="Habitat (DLI)", bbox_to_anchor=(1.05, 0.7), loc='upper left')
plt.gca().add_artist(legend1_cv)

adjust_yaxis_definitive(g_cv, df_cv_melted, x_var="Window", y_var="Value", col_var="Variable", row_var="ARCH", hue_var="HAB")
g_cv.fig.suptitle("CV (%) por Janela, Arco e Habitat (DLI)", y=1.03, fontsize=20)
g_cv.set_axis_labels("Janela de Tempo (dias) / Total (ALL)", "CV (%)")
g_cv.set_titles(row_template="Arco: {row_name}", col_template="{col_name}")
g_cv.fig.tight_layout(rect=[0, 0, 0.85, 0.96])
cv_plot_path = os.path.join(OUTPUT_DIR, "figura_composta_CV_Hachurada.png")
plt.savefig(cv_plot_path, dpi=300, bbox_inches="tight")
plt.close()
print(f"Figura de CV salva em: {cv_plot_path}")

print("\nProcesso finalizado com sucesso!")

In [ ]:
### ANÁLISE DE FREQUÊNCIA DOMINANTE (LOMB-SCARGLE) PARA SST, DLI e CLOROFILA-A ###
# - Análise por Sítio (pixel a pixel), com remoção de tendência e análise de ciclos secundários.
# - V5: GERAÇÃO DE GRÁFICOS INTEGRADA, AGRUPADOS E COLORIDOS POR ARCO (INNER/OUTER)

import os
import glob
import re
import logging
import numpy as np
import pandas as pd
import xarray as xr
from scipy.signal import detrend
from astropy.timeseries import LombScargle
import gc
from tqdm import tqdm
import joblib

# Importações para otimização
from numba import jit
import dask
from dask import delayed
from dask.diagnostics import ProgressBar

# Importações para plotagem
import matplotlib.pyplot as plt
from matplotlib import rcParams
import seaborn as sns

# ============================================
# 1. CONFIGURAÇÕES E LOGGING
# ============================================
# --- Configuração do Logging ---
output_dir = r"C:\Users\rbfra\OneDrive\########PUBLICACOES\############Menezes et al. Mus his distribution and abundance Abrolhos\########NEW RESULTS\LOMB_SCARGLE_POR_ARCO"
os.makedirs(output_dir, exist_ok=True)
log_file = os.path.join(output_dir, 'analysis_log_por_arco.txt')
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler()
    ]
)

logging.info("--- Iniciando Script de Análise de Frequência (Análise por Sítio, Gráficos por Arco) ---")

# --- Caminhos e Cache ---
sites_csv_path = r"C:\Users\rbfra\OneDrive\########CEBIMAR\####PROJETOS\#####Coral trade offs\sites_list_full.csv"
sst_dir   = r"E:\remote sensing\CRW_SST_FULL"
modis_dir = r"E:\remote sensing\MODIS_DATA_FULL"
cache_dir = os.path.join(output_dir, 'cache')
os.makedirs(cache_dir, exist_ok=True)
memory = joblib.Memory(cache_dir, verbose=0)
logging.info(f"Diretório de saída: {output_dir}")

# <--- MUDANÇA APLICADA AQUI --->
# --- Parâmetros ---
# Define períodos de tempo separados para cada tipo de dado.
# Formato: (ano_inicial, ano_final)
SST_PERIOD   = (1985, 2008)
MODIS_PERIOD = (2002, 2008)  # Usado para DLI (Kd490, PAR) e Clorofila

logging.info(f"Período de análise para SST: {SST_PERIOD[0]}-{SST_PERIOD[1]}")
logging.info(f"Período de análise para DLI e CHL (MODIS): {MODIS_PERIOD[0]}-{MODIS_PERIOD[1]}")

num_workers = os.cpu_count() - 2 if os.cpu_count() > 2 else 1

sst_pattern   = 'coraltemp_v3.1_*.nc'
kd490_pattern = 'AQUA_MODIS.*.L3m.DAY.KD.Kd_490.4km.nc'
par_pattern   = 'AQUA_MODIS.*.L3m.DAY.PAR.par.4km.nc'
chl_pattern   = 'AQUA_MODIS.*.L3m.DAY.CHL.chlor_a.4km.nc'

KDPAR_GATTUSO_A, KDPAR_GATTUSO_B, KDPAR_GATTUSO_C = 0.0665, 0.874, 0.00121
samples_per_peak, fap_threshold = 10, 0.05

PRIMARY_MIN_P, PRIMARY_MAX_P = 180, 400
SECONDARY_MIN_P, SECONDARY_MAX_P = 2, 180

# ============================================
# 2. FUNÇÕES DE ANÁLISE (sem alterações)
# ============================================

def get_file_list_for_period(pattern, base_dir, start_year, end_year):
    all_files = glob.glob(os.path.join(base_dir, pattern))
    date_pattern = re.compile(r'(\d{4})\d{3,4}')
    valid_files = [f for f in all_files if (match := date_pattern.search(os.path.basename(f))) and start_year <= int(match.group(1)) <= end_year]
    logging.info(f"Encontrados {len(valid_files)} arquivos para '{pattern}' entre {start_year}-{end_year}.")
    return sorted(valid_files)

@memory.cache
def extract_all_sites_timeseries_dask(file_list, sites_df, var_name_options, desc):
    logging.info(f"Iniciando extração paralela de séries temporais para: {desc}")
    @delayed
    def process_file(f_path):
        try:
            with xr.open_dataset(f_path) as ds:
                var_name = next((v for v in var_name_options if v in ds.data_vars), None)
                if not var_name: return None
                date_match = re.search(r'(\d{7,8})', os.path.basename(f_path))
                if not date_match: return None
                date_str = date_match.group(1)
                ts = pd.to_datetime(date_str, format='%Y%m%d' if len(date_str) == 8 else '%Y%j', errors='coerce')
                if pd.isna(ts): return None
                data = {idx: ds[var_name].sel(lat=site['LATITUDE'], lon=site['LONGITUDE'], method='nearest').item() for idx, site in sites_df.iterrows()}
                return {'date': ts, 'data': data}
        except Exception as e:
            logging.warning(f"Erro ao processar arquivo: {os.path.basename(f_path)}. Erro: {e}")
            return None
    tasks = [process_file(f) for f in file_list]
    with ProgressBar(dt=5.0):
        computed_results = dask.compute(*tasks, num_workers=num_workers)
    site_data = {idx: {'times': [], 'values': []} for idx in sites_df.index}
    for res in computed_results:
        if res is None: continue
        for idx, val in res['data'].items():
            site_data[idx]['times'].append(res['date'])
            site_data[idx]['values'].append(val)
    for idx in site_data:
        site_data[idx]['times'] = np.array(site_data[idx]['times'])
        site_data[idx]['values'] = np.array(site_data[idx]['values'], dtype=np.float32)
    return site_data

@jit(nopython=True)
def compute_dli_numba(kd490, par, depth, A, B, C):
    kdpar = A + B * kd490 - C / (kd490 + 1e-9)
    return par * np.exp(-kdpar * depth)

def calculate_dli_timeseries_numba(kd_data, par_data, sites_df):
    logging.info("Calculando DLI com Numba para cada sítio...")
    dli_timeseries = {idx: {'times': np.array([]), 'values': np.array([])} for idx in sites_df.index}
    for idx, site in sites_df.iterrows():
        depth = max(site['DEPTH_M'], 0.1)
        kd_df = pd.DataFrame({'kd490': kd_data[idx]['values']}, index=pd.to_datetime(kd_data[idx]['times']))
        par_df = pd.DataFrame({'par': par_data[idx]['values']}, index=pd.to_datetime(par_data[idx]['times']))
        merged_df = kd_df.join(par_df, how='inner').dropna()
        merged_df = merged_df[(merged_df['kd490'] > 0) & (merged_df['par'] > 0)]
        if merged_df.empty: continue
        dli_values = compute_dli_numba(
            merged_df['kd490'].values.astype(np.float32), merged_df['par'].values.astype(np.float32),
            np.float32(depth), np.float32(KDPAR_GATTUSO_A), np.float32(KDPAR_GATTUSO_B), np.float32(KDPAR_GATTUSO_C)
        )
        dli_timeseries[idx]['times'] = merged_df.index.to_numpy()
        dli_timeseries[idx]['values'] = dli_values
    return dli_timeseries

def remove_annual_cycle(times, values):
    if len(times) == 0: return np.array([]), np.array([])
    df = pd.DataFrame({'values': values}, index=pd.to_datetime(times))
    monthly_clim = df.groupby(df.index.month)['values'].transform('mean')
    anomalies = df['values'] - monthly_clim
    return anomalies.index.to_numpy(), anomalies.values

def analyze_periodicity_advanced(times, values, min_p, max_p, samples_per_peak, fap_level, preprocess_mode='none'):
    if times is None or len(times) < 20: return np.nan, np.nan
    if preprocess_mode == 'anomaly':
        times, values = remove_annual_cycle(times, values)
        if len(times) < 20: return np.nan, np.nan
    mask = ~np.isnan(values)
    values_masked, times_masked = values[mask], times[mask]
    if len(values_masked) < 20 or np.all(values_masked == values_masked[0]): return np.nan, np.nan
    t = (times_masked.astype('datetime64[ns]') - times_masked.astype('datetime64[ns]').min()).astype('timedelta64[D]').astype(float)
    v_detrended = detrend(values_masked)
    try:
        ls = LombScargle(t, v_detrended)
        freq, power = ls.autopower(minimum_frequency=1/max_p, maximum_frequency=1/min_p, samples_per_peak=samples_per_peak)
        if len(power) == 0: return np.nan, np.nan
        best_power, fap_limit = np.max(power), ls.false_alarm_level(fap_level)
        if np.isnan(fap_limit) or best_power <= fap_limit: return np.nan, np.nan
        dominant_period = 1 / freq[np.argmax(power)]
        fap_of_peak = ls.false_alarm_probability(best_power)
        return dominant_period, fap_of_peak
    except Exception as e:
        logging.warning(f"Erro na análise de Lomb-Scargle: {e}")
        return np.nan, np.nan

# ============================================
# 3. FUNÇÕES DE VISUALIZAÇÃO (REVISADAS PARA AGRUPAMENTO POR ARCO)
# ============================================
def setup_plot_style():
    """Configura o estilo estético dos gráficos."""
    plt.style.use('seaborn-v0_8-whitegrid')
    rcParams['font.family'] = 'Arial'
    rcParams['font.size'] = 12
    rcParams['axes.labelsize'] = 14
    rcParams['axes.titlesize'] = 16
    rcParams['xtick.labelsize'] = 10 # Reduzido para melhor encaixe
    rcParams['ytick.labelsize'] = 12
    rcParams['legend.fontsize'] = 12
    rcParams['figure.titlesize'] = 18
    rcParams['patch.edgecolor'] = 'black'
    rcParams['patch.linewidth'] = 0.5

def create_barplot_by_arch(df_cycle, variable, cycle_name, arch_palette, output_dir):
    """
    Cria UM gráfico de barras para UM tipo de ciclo, com sítios agrupados e coloridos por Arco.
    """
    if df_cycle.empty:
        logging.warning(f"Não há dados significativos para '{variable}' no ciclo '{cycle_name}'. Gráfico não será gerado.")
        return

    logging.info(f"Gerando gráfico por sítio para: {variable} - Ciclo {cycle_name} (Agrupado por Arco)...")
    
    # Ordenar por Arco e depois por nome do sítio para agrupar visualmente
    df_cycle = df_cycle.sort_values(['ARCH', 'UNIQUE_LABEL'])

    fig, ax = plt.subplots(figsize=(20, 9))
    
    # Plotar as barras
    bar_container = ax.bar(
        df_cycle['UNIQUE_LABEL'],
        df_cycle['Period'],
        color=[arch_palette.get(arch, '#cccccc') for arch in df_cycle['ARCH']]
    )
    
    ax.bar_label(bar_container, fmt='{:,.0f}', fontsize=8, padding=3, rotation=90)

    # Ajustes Finais do Gráfico
    ax.set_title(f"Ciclo: {cycle_name}", fontsize=16, weight='bold')
    ax.set_ylabel("Período Dominante (dias)")
    ax.tick_params(axis='x', labelrotation=90)
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    ax.set_ylim(bottom=0, top=ax.get_ylim()[1] * 1.20) # Mais espaço para os rótulos
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Legenda
    handles = [plt.Rectangle((0,0),1,1, color=color) for color in arch_palette.values()]
    labels = [key.replace('_', ' ').title() for key in arch_palette.keys()]
    fig.legend(handles, labels, title="Arco",
               bbox_to_anchor=(1.0, 0.9), loc='upper left',
               frameon=True, shadow=True, title_fontproperties={'weight':'bold'})
    
    # Título principal e layout
    fig.suptitle(f'Períodos Dominantes de {variable} por Sítio (Agrupado por Arco)', y=0.98, fontsize=18, weight='bold')
    fig.tight_layout(rect=[0.03, 0.03, 0.9, 0.93]) # Ajusta para caber a legenda e rótulos do eixo X

    # Salvar a Figura
    cycle_str = cycle_name.split(' ')[0].lower()
    output_path = os.path.join(output_dir, f'barplot_periodos_{variable.lower()}_ciclo_{cycle_str}_por_arco.png')
    plt.savefig(output_path, dpi=300)
    plt.close(fig)
    logging.info(f"--> Gráfico salvo em: {output_path}")

# ============================================
# 4. EXECUÇÃO PRINCIPAL
# ============================================

try:
    sites_df = pd.read_csv(sites_csv_path, sep=';')
    sites_df.columns = [col.strip().upper() for col in sites_df.columns]
    
    required_cols = ['SITE_NAME', 'LATITUDE', 'LONGITUDE', 'DEPTH_M', 'ARCH']
    if not all(col in sites_df.columns for col in required_cols):
        missing = [col for col in required_cols if col not in sites_df.columns]
        raise ValueError(f"Colunas essenciais não encontradas: {missing}")
    
    sites_df['ARCH'] = sites_df['ARCH'].str.strip().str.lower()
    sites_df['UNIQUE_LABEL'] = sites_df['SITE_NAME'].str.strip() + ' (' + sites_df['DEPTH_M'].astype(str) + 'm)'
    
    initial_rows = len(sites_df)
    sites_df = sites_df[sites_df['ARCH'].isin(['inner', 'outer'])].copy()
    logging.info(f"Carregados {initial_rows} registros. {len(sites_df)} pertencem a 'inner' ou 'outer' e serão processados.")

except Exception as e:
    logging.critical(f"ERRO FATAL ao carregar ou pré-processar a planilha de sítios: {e}")
    exit()

# <--- MUDANÇA APLICADA AQUI --->
# --- ETAPA 1: Extração de Dados ---
logging.info("--- ETAPA 1: Extração de todas as séries temporais (uma para cada registro/pixel) ---")
# Usa as tuplas de período específicas para cada tipo de dado
sst_files   = get_file_list_for_period(sst_pattern, sst_dir, SST_PERIOD[0], SST_PERIOD[1])
kd490_files = get_file_list_for_period(kd490_pattern, modis_dir, MODIS_PERIOD[0], MODIS_PERIOD[1])
par_files   = get_file_list_for_period(par_pattern, modis_dir, MODIS_PERIOD[0], MODIS_PERIOD[1])
chl_files   = get_file_list_for_period(chl_pattern, modis_dir, MODIS_PERIOD[0], MODIS_PERIOD[1])

if not all([sst_files, kd490_files, par_files, chl_files]):
    logging.critical("ERRO: Um ou mais tipos de dados não foram encontrados.")
    exit()

sst_data = extract_all_sites_timeseries_dask(sst_files, sites_df, ['analysed_sst', 'sea_surface_temperature'], "SST")
kd_data  = extract_all_sites_timeseries_dask(kd490_files, sites_df, ['Kd_490'], "Kd490")
par_data = extract_all_sites_timeseries_dask(par_files, sites_df, ['par'], "PAR")
chl_data = extract_all_sites_timeseries_dask(chl_files, sites_df, ['chlor_a'], "CHL")

# --- ETAPA 2: Cálculo de DLI ---
logging.info("--- ETAPA 2: Cálculo de DLI para cada sítio/pixel ---")
dli_data = calculate_dli_timeseries_numba(kd_data, par_data, sites_df)
del kd_data, par_data; gc.collect()

# --- ETAPA 3: Análise de Frequência ---
logging.info("--- ETAPA 3: Análise de Frequência Dominante e Secundária por Sítio/pixel ---")
results = []
for idx, site in tqdm(sites_df.iterrows(), total=len(sites_df), desc="Analisando Frequências por Sítio"):
    site_info = site.to_dict()
    # Análise Primária
    sst_p1, sst_fap1 = analyze_periodicity_advanced(sst_data.get(idx, {}).get('times'), sst_data.get(idx, {}).get('values'), PRIMARY_MIN_P, PRIMARY_MAX_P, samples_per_peak, fap_threshold)
    dli_p1, dli_fap1 = analyze_periodicity_advanced(dli_data.get(idx, {}).get('times'), dli_data.get(idx, {}).get('values'), PRIMARY_MIN_P, PRIMARY_MAX_P, samples_per_peak, fap_threshold)
    chl_p1, chl_fap1 = analyze_periodicity_advanced(chl_data.get(idx, {}).get('times'), chl_data.get(idx, {}).get('values'), PRIMARY_MIN_P, PRIMARY_MAX_P, samples_per_peak, fap_threshold)
    # Análise Secundária
    sst_p2, sst_fap2 = analyze_periodicity_advanced(sst_data.get(idx, {}).get('times'), sst_data.get(idx, {}).get('values'), SECONDARY_MIN_P, SECONDARY_MAX_P, samples_per_peak, fap_threshold, preprocess_mode='anomaly')
    dli_p2, dli_fap2 = analyze_periodicity_advanced(dli_data.get(idx, {}).get('times'), dli_data.get(idx, {}).get('values'), SECONDARY_MIN_P, SECONDARY_MAX_P, samples_per_peak, fap_threshold, preprocess_mode='anomaly')
    chl_p2, chl_fap2 = analyze_periodicity_advanced(chl_data.get(idx, {}).get('times'), chl_data.get(idx, {}).get('values'), SECONDARY_MIN_P, SECONDARY_MAX_P, samples_per_peak, fap_threshold, preprocess_mode='anomaly')
    
    site_info.update({
        'SST_PERIOD_PRIMARY': sst_p1, 'SST_FAP_PRIMARY': sst_fap1,
        'DLI_PERIOD_PRIMARY': dli_p1, 'DLI_FAP_PRIMARY': dli_fap1,
        'CHL_PERIOD_PRIMARY': chl_p1, 'CHL_FAP_PRIMARY': chl_fap1,
        'SST_PERIOD_SECONDARY': sst_p2, 'SST_FAP_SECONDARY': sst_fap2,
        'DLI_PERIOD_SECONDARY': dli_p2, 'DLI_FAP_SECONDARY': dli_fap2,
        'CHL_PERIOD_SECONDARY': chl_p2, 'CHL_FAP_SECONDARY': chl_fap2
    })
    results.append(site_info)

# --- ETAPA 4: Salvando Resultados ---
logging.info("--- ETAPA 4: Salvando Resultados Finais ---")
df_results = pd.DataFrame(results) if results else pd.DataFrame()

if df_results.empty:
    logging.warning("Nenhuma análise foi concluída. O arquivo de resultados não será gerado.")
else:
    # Nome do arquivo mais descritivo
    filename = (f"frequencias_SST_{SST_PERIOD[0]}-{SST_PERIOD[1]}"
                f"_MODIS_{MODIS_PERIOD[0]}-{MODIS_PERIOD[1]}.csv")
    output_csv_path = os.path.join(output_dir, filename)
    try:
        df_results.to_csv(output_csv_path, index=False, sep=';', decimal=',')
        logging.info(f"SUCESSO! Resultados salvos em: {output_csv_path}")
    except Exception as e:
        logging.error(f"ERRO ao salvar o arquivo de resultados: {e}")

# --- ETAPA 5: Gerando Visualizações por Arco ---
logging.info("--- ETAPA 5: Gerando Gráficos de Resultados Agrupados por Arco ---")
if not df_results.empty:
    setup_plot_style()
    
    arch_palette = {'inner': 'royalblue', 'outer': 'firebrick'}

    for variable in ['SST', 'DLI', 'CHL']:
        # Preparar dados para o ciclo primário
        primary_df = df_results.loc[df_results[f'{variable}_FAP_PRIMARY'] <= fap_threshold].copy()
        primary_df.rename(columns={f'{variable}_PERIOD_PRIMARY': 'Period'}, inplace=True)
        primary_df.dropna(subset=['Period', 'UNIQUE_LABEL', 'ARCH'], inplace=True)
        
        # Preparar dados para o ciclo secundário
        secondary_df = df_results.loc[df_results[f'{variable}_FAP_SECONDARY'] <= fap_threshold].copy()
        secondary_df.rename(columns={f'{variable}_PERIOD_SECONDARY': 'Period'}, inplace=True)
        secondary_df.dropna(subset=['Period', 'UNIQUE_LABEL', 'ARCH'], inplace=True)
        
        # Gerar os gráficos separados
        create_barplot_by_arch(primary_df, variable, 'Primário (~Anual)', arch_palette, output_dir)
        create_barplot_by_arch(secondary_df, variable, 'Secundário (< 6 meses)', arch_palette, output_dir)
        
    logging.info("Geração de gráficos concluída.")
else:
    logging.warning("DataFrame de resultados vazio. Gráficos não serão gerados.")

logging.info("--- Script Finalizado ---")

In [ ]:
### ANÁLISE DE FREQUÊNCIA DOMINANTE ESPACIAL (LOMB-SCARGLE POR PIXEL) ###
# Lê dados de satélite, calcula DLI bentônico e encontra os períodos dominantes para cada pixel na região de estudo.
# Adaptado com otimizações de processamento espacial usando Dask e Xarray.

import os
import re
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import box
from astropy.timeseries import LombScargle
from dask.distributed import Client, LocalCluster
import dask
import gc
import logging

# ============================================
# 1. CONFIGURAÇÕES
# ============================================
print("--- Iniciando Script: Análise de Frequência Espacial ---")

# --- Configuração do Dask e Logging ---
# Configurar um diretório para logs de erro
log_file = 'log_analise_frequencia.log'
logging.basicConfig(filename=log_file, level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')

# Configurar o cliente Dask para paralelismo e gerenciamento de memória
# Ajuste n_workers e threads_per_worker conforme a capacidade do seu PC
try:
    cluster = LocalCluster(n_workers=4, threads_per_worker=2, memory_limit='8GB')
    client = Client(cluster)
    print(f"Cliente Dask iniciado. Dashboard disponível em: {client.dashboard_link}")
except Exception as e:
    print(f"Não foi possível iniciar o cliente Dask, rodando em modo single-threaded. Erro: {e}")
    dask.config.set(scheduler='single-threaded')

# --- Caminhos de Entrada ---
sst_dir   = r"H:\remote sensing\CRW_SST_FULL"
modis_dir = r"H:\remote sensing\MODIS_DATA_FULL"

# --- Diretório de Saída ---
output_dir = r"C:\Users\rbfra\OneDrive\########PUBLICACOES\############Menezes et al. Mus his distribution and abundance Abrolhos\########NEW RESULTS\LOMB_SCARGLE_MAPAS"
os.makedirs(output_dir, exist_ok=True)
print(f"Diretório de saída definido para: {output_dir}")

# --- Período e Região de Interesse ---
start_year = 2002
end_year   = 2008
lat_min, lat_max = -20.5, -14.5
lon_min, lon_max = -40.5, -35.5

# --- Padrões de busca para os arquivos ---
sst_pattern   = 'coraltemp_v3.1_*.nc'
kd490_pattern = 'AQUA_MODIS.*.L3m.DAY.KD.Kd_490.4km.nc'
par_pattern   = 'AQUA_MODIS.*.L3m.DAY.PAR.par.4km.nc'

# --- Shapefiles para Plotagem (Opcional, mas recomendado) ---
coast_shp_path = r"C:\Users\rbfra\OneDrive\GIS shapes\batimetria_new\LINHA_DE_COSTA_IMAGEM_GEOCOVER_SIRGAS_2000.shp"
reefs_shp_path = r"H:\remote sensing\Mascaras\Recifes_Banco_dos_Abrolhos\Recifes_Banco_dos_Abrolhos.shp"


# --- Parâmetros da Análise ---
KDPAR_GATTUSO_A, KDPAR_GATTUSO_B, KDPAR_GATTUSO_C = 0.0665, 0.874, 0.00121
ANALYSIS_DEPTH_M = 10.0  # Profundidade fixa (em metros) para o cálculo do mapa de DLI
min_period_days = 2      # Período mínimo a ser detectado (dias)
max_period_days = 400    # Período máximo a ser detectado (dias)
samples_per_peak = 10    # Aumenta a resolução do periodograma
MIN_VALID_POINTS = 50    # Mínimo de pontos não-nulos na série temporal para tentar a análise

# ============================================
# 2. FUNÇÕES AUXILIARES
# ============================================

def get_lat_slice(ds, lat_min, lat_max):
    """Retorna o slice correto para a latitude, independentemente da ordem."""
    lat_vals = ds['lat'].values
    return slice(lat_min, lat_max) if lat_vals[0] < lat_vals[-1] else slice(lat_max, lat_min)

def load_data_stack(pattern, base_dir, period, var_names):
    """Carrega uma pilha de dados de satélite de forma robusta e otimizada."""
    print(f"Carregando dados para: {pattern}...")
    all_files = sorted(os.path.join(base_dir, f) for f in os.listdir(base_dir) if f.endswith('.nc'))
    date_pattern = re.compile(r'(\d{8})')

    def preprocess(ds):
        var_name = next((v for v in var_names if v in ds.data_vars), None)
        if not var_name: return xr.Dataset()
        return ds[[var_name]].astype('float32')

    candidate_paths = [
        f for f in all_files
        if (match := date_pattern.search(os.path.basename(f)))
        and period[0] <= int(match.group(1)[:4]) <= period[1]
    ]

    if not candidate_paths:
        logging.warning(f"Nenhum arquivo encontrado para {pattern} no período {period}.")
        return None

    try:
        ds_stack = xr.open_mfdataset(
            candidate_paths, combine='by_coords', preprocess=preprocess, parallel=True,
            chunks={'time': 120, 'lat': 200, 'lon': 200}
        )
        lat_s = get_lat_slice(ds_stack, lat_min, lat_max)
        lon_s = slice(lon_min, lon_max)
        ds_stack = ds_stack.sel(lat=lat_s, lon=lon_s)
        var_name = list(ds_stack.data_vars)[0]
        print(f"-> {len(ds_stack.time)} imagens carregadas para a variável '{var_name}'.")
        return ds_stack[var_name]
    except Exception as e:
        logging.error(f"Falha ao carregar dados para {pattern}: {e}")
        return None

def lomb_scargle_ufunc(values, times):
    """
    Função wrapper para ser usada com xarray.apply_ufunc.
    Calcula o período dominante para uma única série temporal 1D.
    """
    mask = ~np.isnan(values)
    if mask.sum() < MIN_VALID_POINTS:
        return np.nan

    # Converte timestamps para dias corridos
    t_days = (times[mask] - times[mask].min()) / np.timedelta64(1, 'D')
    v = values[mask]

    # Normalizar os dados pode ajudar a estabilizar a análise
    v = (v - np.mean(v)) / np.std(v)

    try:
        ls = LombScargle(t_days, v)
        freq, power = ls.autopower(
            minimum_frequency=1/max_period_days,
            maximum_frequency=1/min_period_days,
            samples_per_peak=samples_per_peak
        )
        if len(freq) == 0:
             return np.nan
        dominant_period = 1 / freq[np.argmax(power)]
        return dominant_period
    except Exception:
        # Captura erros raros dentro do LombScargle (e.g., todos os valores iguais)
        return np.nan

def calculate_dominant_period_map(da):
    """
    Aplica a análise de Lomb-Scargle em um DataArray 3D (time, lat, lon)
    usando dask e apply_ufunc para processamento espacial eficiente.
    """
    if da is None:
        return None

    # Garante que os chunks não sejam muito pequenos na dimensão do tempo
    da = da.chunk({'time': -1, 'lat': 'auto', 'lon': 'auto'})
    
    # Prepara o array de tempo para ser passado para a ufunc
    time_da = xr.DataArray(da.time.values, dims=['time'], coords={'time': da.time})

    # `apply_ufunc` é a mágica aqui. Ele aplica `lomb_scargle_ufunc`
    # a cada pixel (ao longo da dimensão 'time').
    period_map = xr.apply_ufunc(
        lomb_scargle_ufunc,
        da,
        time_da,
        input_core_dims=[['time'], ['time']], # A função opera sobre a dimensão 'time'
        output_core_dims=[[]],                # A função retorna um escalar (o período)
        exclude_dims=set(('time',)),          # A dimensão 'time' é consumida
        dask='parallelized',                  # Habilita o processamento com Dask
        output_dtypes=[np.float64],           # O tipo de dado da saída
        vectorize=True                        # Melhora a performance em alguns casos
    )
    period_map.name = f'{da.name}_dominant_period'
    return period_map

def plot_spatial_map(da, title, output_path, cmap='viridis', cbar_label='Período (dias)'):
    """Plota e salva um mapa 2D a partir de um DataArray."""
    if da is None:
        print(f"Dados nulos para o mapa '{title}'. Plotagem pulada.")
        return

    print(f"Gerando mapa: {title}")
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Define limites e aparência
    ax.set_xlim(lon_min, lon_max)
    ax.set_ylim(lat_min, lat_max)
    ax.set_facecolor('lightgrey') # Cor para áreas sem dados

    # Carrega os dados para plotagem
    data_to_plot = da.compute() # .compute() aciona o cálculo Dask

    # Plota os dados
    im = data_to_plot.plot(
        ax=ax,
        cmap=cmap,
        cbar_kwargs={'label': cbar_label, 'orientation': 'vertical', 'pad': 0.02},
        add_colorbar=True
    )
    
    # Adiciona contornos geográficos
    try:
        gpd.read_file(coast_shp_path).plot(ax=ax, edgecolor='black', linewidth=1.0)
        gpd.read_file(reefs_shp_path).plot(ax=ax, edgecolor='purple', linewidth=0.8, alpha=0.7)
    except Exception as e:
        print(f"Aviso: Não foi possível carregar shapefiles. {e}")

    ax.set_title(title, fontsize=16)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"Mapa salvo em: {output_path}")


# ============================================
# 3. EXECUÇÃO PRINCIPAL
# ============================================

# --- Carregar Dados de Satélite ---
sst_da = load_data_stack(sst_pattern, sst_dir, (start_year, end_year), ['analysed_sst', 'sea_surface_temperature'])
kd490_da = load_data_stack(kd490_pattern, modis_dir, (start_year, end_year), ['Kd_490'])
par_da = load_data_stack(par_pattern, modis_dir, (start_year, end_year), ['par'])

# --- Calcular DLI Bentônico (de forma espacial) ---
dli_da = None
if par_da is not None and kd490_da is not None:
    print("\nCalculando DLI Bentônico para toda a grade...")
    # Alinhar temporalmente e espacialmente os dados de luz
    par_aligned, kd490_aligned = xr.align(par_da, kd490_da, join='inner')
    del par_da, kd490_da; gc.collect()

    # Fórmula de Gattuso et al. para Kd(PAR) a partir de Kd(490)
    kdpar_da = KDPAR_GATTUSO_A + KDPAR_GATTUSO_B * kd490_aligned - KDPAR_GATTUSO_C / (kd490_aligned + 1e-9)
    kdpar_da = kdpar_da.where(kdpar_da > 0) # Mascarar valores inválidos

    # Calcular DLI na profundidade definida
    dli_da = par_aligned * np.exp(-kdpar_da * ANALYSIS_DEPTH_M)
    dli_da.name = 'benthic_dli'
    print(f"Cálculo de DLI a {ANALYSIS_DEPTH_M}m concluído.")
    del par_aligned, kd490_aligned, kdpar_da; gc.collect()
else:
    print("\nAviso: Dados de PAR ou Kd490 não foram carregados. Análise de DLI será pulada.")

# --- Realizar Análise de Frequência ---
print("\n--- Iniciando Análise de Frequência Espacial (Pode demorar) ---")

# Calcular mapa de período dominante para SST
print("\n1. Processando SST...")
sst_period_map = calculate_dominant_period_map(sst_da)
if sst_period_map is not None:
    sst_period_map = sst_period_map.rename('sst_dominant_period')

# Calcular mapa de período dominante para DLI
print("\n2. Processando DLI Bentônico...")
dli_period_map = calculate_dominant_period_map(dli_da)
if dli_period_map is not None:
    dli_period_map = dli_period_map.rename('dli_dominant_period')

# Libera memória dos dados brutos
del sst_da, dli_da; gc.collect()

# ============================================
# 4. SALVAR RESULTADOS
# ============================================
print("\n--- Salvando Resultados ---")

# --- Salvar Mapas como PNG ---
plot_spatial_map(
    sst_period_map,
    title=f'Período Dominante de SST ({start_year}-{end_year})',
    output_path=os.path.join(output_dir, "mapa_periodo_dominante_SST.png"),
    cmap='inferno'
)

plot_spatial_map(
    dli_period_map,
    title=f'Período Dominante de DLI a {ANALYSIS_DEPTH_M}m ({start_year}-{end_year})',
    output_path=os.path.join(output_dir, "mapa_periodo_dominante_DLI.png"),
    cmap='cividis'
)

# --- Salvar Mapas como Arquivos NetCDF ---
# Juntar os resultados em um único Dataset para facilitar o salvamento
results_ds = xr.Dataset()
if sst_period_map is not None:
    results_ds['sst_period'] = sst_period_map
if dli_period_map is not None:
    results_ds['dli_period'] = dli_period_map

if results_ds.data_vars:
    output_nc_path = os.path.join(output_dir, "mapas_frequencias_dominantes.nc")
    print(f"\nSalvando resultados em NetCDF: {output_nc_path}")
    # O .compute() aqui força a execução de todos os cálculos Dask antes de salvar
    results_ds.compute().to_netcdf(output_nc_path)
    print("Arquivo NetCDF salvo com sucesso.")

# --- Finalização ---
if 'client' in locals() and client.status == 'running':
    client.close()
    
print("\n>>> SUCESSO! Análise de frequência espacial concluída.")
print(f"Mapas em PNG e NetCDF salvos em: {output_dir}")
print(f"Log de execução salvo em: {log_file}")
print("--- Script concluído ---")

In [ ]:
##### Calcula RGR, médias de saúde, realiza PCAs de Saúde e Interações

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import math # Importado para ajudar a organizar as figuras compostas

# ---------------------------
# 1. CONFIGURAÇÕES E LEITURA DOS DADOS
# ---------------------------
print("--- Iniciando Script 2: Análise Biológica (Versão Aprimorada) ---")

# Caminho para os dados biológicos
base_path = r"C:\Users\rbfra\OneDrive\########PUBLICACOES\############Menezes et al. Mus his distribution and abundance Abrolhos\######22.04.23\DATA"
file_vitality = os.path.join(base_path, "######Vitality and size_new.xlsx")

# Diretório de saída para este script
output_dir = r"C:\Users\rbfra\OneDrive\########PUBLICACOES\############Menezes et al. Mus his distribution and abundance Abrolhos\########NEW RESULTS\output_ANALISE_BIOLOGICA"
os.makedirs(output_dir, exist_ok=True)
print(f"Diretório de saída definido para: {output_dir}")

# Leitura do arquivo de vitalidade
try:
    vitality_data = pd.read_excel(file_vitality)
except FileNotFoundError as e:
    print(f"ERRO: Arquivo não encontrado - {e}. Verifique o caminho.")
    exit()

# Padronização de colunas e valores
def standardize_df(df):
    df.columns = df.columns.str.strip().str.upper()
    for col in ['SITE', 'HAB', 'REEF']:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.upper()
    return df

vitality_data = standardize_df(vitality_data)
vitality_data['SITE_COL'] = vitality_data['SITE'] + "_" + vitality_data['COL'].astype(str)

# ---------------------------
# 2. CÁLCULO DAS MÉTRICAS DE DESEMPENHO POR COLÔNIA
# ---------------------------
print("\nCalculando métricas de desempenho por colônia (RGR, médias, etc.)...")

def compute_rgr(group):
    group = group.sort_values('YEAR')
    if len(group['YEAR'].unique()) < 2: return np.nan
    ar_inicial = group.iloc[0]['AR_TOTAL']; ar_final = group.iloc[-1]['AR_TOTAL']
    ano_inicial = group.iloc[0]['YEAR']; ano_final = group.iloc[-1]['YEAR']
    if ar_inicial <= 0 or ar_final <= 0 or ano_final == ano_inicial: return np.nan
    return (np.log(ar_final) - np.log(ar_inicial)) / (ano_final - ano_inicial)

def get_ar_total_2006(group):
    if 2006 in group['YEAR'].values:
        return group[group['YEAR'] == 2006]['AR_TOTAL'].iloc[0]
    return np.nan

# --- Bloco CORRIGIDO para incluir MEAN_DEAD_TISSUE diretamente ---
colony_metrics = vitality_data.groupby('SITE_COL').apply(lambda g: pd.Series({
    'RGR': compute_rgr(g),
    'AR_TOTAL_INICIAL': get_ar_total_2006(g),
    'MEAN_AR_TOTAL': g['AR_TOTAL'].mean(),
    'MEAN_HEALTH': g['HEALTH %'].mean(),
    'MEAN_BLEACH': g['BLEACHING %'].mean(),
    'MEAN_DEAD_TISSUE': g['DEAD %'].mean()  # <--- CORREÇÃO: Calcula a média diretamente da coluna 'DEAD %'
})).reset_index()

# --- MELHORIA: Garantir que os valores não sejam negativos ---
# Adicione este bloco logo após o cálculo para forçar valores negativos a zero, como uma segurança.
for col in ['MEAN_HEALTH', 'MEAN_BLEACH', 'MEAN_DEAD_TISSUE']:
    if col in colony_metrics.columns:
        colony_metrics[col] = colony_metrics[col].clip(lower=0)
        
print("Métricas calculadas (incluindo tecido morto) e valores negativos zerados com sucesso.")

# ---------------------------
# 3. PCA DA SAÚDE DOS CORAIS
# ---------------------------
print("\nRealizando PCA da Saúde dos Corais...")
health_cols = ["HEALTH %", "BLEACHING %", "DEAD %"]
health_props = vitality_data.groupby('SITE_COL')[health_cols].mean().reset_index()
health_props_scaled = StandardScaler().fit_transform(health_props[health_cols])

pca_health = PCA(n_components=2)
health_scores = pca_health.fit_transform(health_props_scaled)

# Cria o dataframe de scores
df_scores_health = health_props[['SITE_COL']].copy()
df_scores_health["HEALTH_PC1"] = health_scores[:, 0]
df_scores_health["HEALTH_PC2"] = health_scores[:, 1]

# Adiciona metadados para plotagem
meta_info = vitality_data[['SITE_COL', 'SITE', 'HAB', 'REEF']].drop_duplicates()
df_scores_health = pd.merge(df_scores_health, meta_info, on='SITE_COL')
# Adiciona as variáveis originais para os bubble plots
df_scores_health = pd.merge(df_scores_health, health_props, on='SITE_COL')

# Salva os scores
scores_path_health = os.path.join(output_dir, "scores_PCA_Saude.xlsx")
df_scores_health.to_excel(scores_path_health, index=False)
print(f"Scores da PCA de Saúde salvos em: {scores_path_health}")

# Cria e salva o dataframe de loadings
loadings_df_health = pd.DataFrame(pca_health.components_.T, columns=["PC1", "PC2"], index=health_cols)
loadings_path_health = os.path.join(output_dir, "loadings_PCA_Saude.csv")
loadings_df_health.to_csv(loadings_path_health)
print(f"Loadings da PCA de Saúde salvos em: {loadings_path_health}")

# ---------------------------
# 4. PCA DAS INTERAÇÕES LOCAIS
# ---------------------------
print("\nRealizando PCA das Interações Locais...")
cols_interactions = ["SUR_TURF %", "SUR_CCA %", "SUR_CYANO %", "SUR_DICTYOTA %", "SUR_OTHMACR %",
                     "SUR_PALYTHOA %", "SUR_CORAL %", "SUR_SAND %", "SUR_NON-BIOTIC %"]
interaction_means_agg = vitality_data.groupby(['SITE', 'HAB'])[cols_interactions].mean(numeric_only=True).reset_index()

# Agrupa em categorias mais amplas
interaction_vars = {
    'SUR_TURF': interaction_means_agg['SUR_TURF %'],
    'SUR_CCA': interaction_means_agg['SUR_CCA %'],
    'SUR_MACROALGAE': interaction_means_agg['SUR_DICTYOTA %'] + interaction_means_agg['SUR_OTHMACR %'],
    'SUR_CYANO': interaction_means_agg['SUR_CYANO %'],
    'SUR_PALYTHOA': interaction_means_agg['SUR_PALYTHOA %'],
    'SUR_ABIOTIC': interaction_means_agg['SUR_SAND %'] + interaction_means_agg['SUR_NON-BIOTIC %']
}
new_interactions = pd.DataFrame(interaction_vars)
new_interactions.insert(0, 'HAB', interaction_means_agg['HAB'])
new_interactions.insert(0, 'SITE', interaction_means_agg['SITE'])

X_interactions = new_interactions.drop(columns=['SITE', 'HAB']).fillna(0)
X_interactions_scaled = StandardScaler().fit_transform(X_interactions)

pca_interactions = PCA(n_components=2)
interaction_scores = pca_interactions.fit_transform(X_interactions_scaled)

# Cria dataframe de scores
df_scores_interactions = new_interactions[['SITE', 'HAB']].copy()
df_scores_interactions["PC1_INTERACAO"] = interaction_scores[:, 0]
df_scores_interactions["PC2_INTERACAO"] = interaction_scores[:, 1]

# Adiciona metadados e variáveis originais
reef_info_agg = vitality_data[['SITE', 'HAB', 'REEF']].drop_duplicates()
df_scores_interactions = pd.merge(df_scores_interactions, reef_info_agg, on=['SITE', 'HAB'])
df_scores_interactions = pd.merge(df_scores_interactions, new_interactions, on=['SITE', 'HAB'])

# Salva os scores
scores_path_interactions = os.path.join(output_dir, "scores_PCA_Interacoes.xlsx")
df_scores_interactions.to_excel(scores_path_interactions, index=False)
print(f"Scores da PCA de Interações salvos em: {scores_path_interactions}")

# Cria e salva o dataframe de loadings
loadings_df_interactions = pd.DataFrame(pca_interactions.components_.T, columns=["PC1", "PC2"], index=X_interactions.columns)
loadings_path_interactions = os.path.join(output_dir, "loadings_PCA_Interacoes.csv")
loadings_df_interactions.to_csv(loadings_path_interactions)
print(f"Loadings da PCA de Interações salvos em: {loadings_path_interactions}")

# ---------------------------
# 5. GERAÇÃO DOS GRÁFICOS (BUBBLE PLOTS) - VERSÃO CORRIGIDA
# ---------------------------
print("\nGerando gráficos de bubble plot...")

# --- Definição de Cores e Símbolos (Consistente com Script 1) ---
unique_reefs = sorted(vitality_data['REEF'].unique())
cmap = plt.get_cmap('tab10')
color_map = {reef: cmap(i) for i, reef in enumerate(unique_reefs)}

unique_habitats = sorted(vitality_data['HAB'].unique())
habitat_shapes = ['o', 's', '^', 'D', 'v', '<', '>']
shape_map = {hab: habitat_shapes[i % len(habitat_shapes)] for i, hab in enumerate(unique_habitats)}

print(f"Mapeamento de Cores (Recife): {color_map}")
print(f"Mapeamento de Símbolos (Habitat): {shape_map}")

def scale_marker_sizes(values, scale_factor=300, min_size=30):
    vals = np.asarray(values, dtype=float); valid = np.isfinite(vals)
    if valid.sum() == 0: return np.full(vals.shape, min_size)
    vmin, vmax = vals[valid].min(), vals[valid].max()
    if vmax == vmin: return np.full(vals.shape, min_size + scale_factor/2)
    scaled = (vals - vmin) / (vmax - vmin); scaled[~valid] = 0
    return scaled * scale_factor + min_size

# --- Gráficos para a PCA de Saúde ---
bubble_vars_health = health_cols
plots_health = []

legend_elements_color = [plt.Line2D([0], [0], marker='o', color='w', label=reef, markersize=10, markerfacecolor=color_map[reef]) for reef in unique_reefs]
legend_elements_shape = [plt.Line2D([0], [0], marker=shape_map[hab], color='grey', label=hab, linestyle='None', markersize=10) for hab in unique_habitats]

for var in bubble_vars_health:
    fig, ax = plt.subplots(figsize=(8, 7))
    sizes = scale_marker_sizes(df_scores_health[var])
    
    # ### CORREÇÃO APLICADA AQUI ###
    # Adicionado `legend=False` para evitar que o Seaborn tente criar uma legenda automática
    # que entra em conflito com o array de tamanhos `s`.
    sns.scatterplot(
        data=df_scores_health,
        x='HEALTH_PC1',
        y='HEALTH_PC2',
        hue='REEF',
        style='HAB',
        s=sizes,
        palette=color_map,
        markers=shape_map,
        alpha=0.7,
        edgecolor='k',
        ax=ax,
        legend=False  # <-- A CORREÇÃO ESTÁ AQUI
    )

    ax.set_title(f'PCA da Saúde (Tamanho ~ {var})')
    ax.set_xlabel(f'PC1 ({pca_health.explained_variance_ratio_[0]:.1%})')
    ax.set_ylabel(f'PC2 ({pca_health.explained_variance_ratio_[1]:.1%})')
    ax.grid(True, linestyle='--', alpha=0.6)
    
    # Nossa legenda manual continua funcionando perfeitamente
    first_legend = ax.legend(title="Recife", handles=legend_elements_color, loc='upper left')
    ax.add_artist(first_legend)
    ax.legend(title="Habitat", handles=legend_elements_shape, loc='lower left')
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"PCA_Saude_Bubble_{var.replace(' %', '')}.png"))
    # O código para a figura composta já desliga a legenda, então não precisa de alteração.
    # Apenas salvamos a figura individual para a lista.
    # Para isso, precisamos salvar a imagem em memória antes de fechar.
    fig.canvas.draw()
    img_data = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
    img_data = img_data.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    plots_health.append(img_data)
    plt.close(fig)

# Cria figura composta para PCA de Saúde
n_plots_health = len(bubble_vars_health) # Usar o número de variáveis
n_cols_health = math.ceil(math.sqrt(n_plots_health))
n_rows_health = math.ceil(n_plots_health / n_cols_health)
fig_comp_health, axes_comp_health = plt.subplots(n_rows_health, n_cols_health, figsize=(n_cols_health * 6, n_rows_health * 5), sharex=True, sharey=True)
axes_comp_health = axes_comp_health.flatten()

for i, var in enumerate(bubble_vars_health):
    ax = axes_comp_health[i]
    sizes = scale_marker_sizes(df_scores_health[var])
    
    sns.scatterplot(
        data=df_scores_health, x='HEALTH_PC1', y='HEALTH_PC2',
        hue='REEF', style='HAB', s=sizes, palette=color_map,
        markers=shape_map, alpha=0.7, edgecolor='k', ax=ax, legend=False
    )
    
    ax.set_title(f'Tamanho ~ {var}')
    ax.grid(True, linestyle='--', alpha=0.6)
    if i % n_cols_health == 0: ax.set_ylabel(f'PC2 ({pca_health.explained_variance_ratio_[1]:.1%})')
    if i >= n_plots_health - n_cols_health: ax.set_xlabel(f'PC1 ({pca_health.explained_variance_ratio_[0]:.1%})')

fig_comp_health.legend(handles=legend_elements_color + legend_elements_shape, title="Recife / Habitat", loc='upper right')
fig_comp_health.suptitle('Análise de Componentes Principais da Saúde dos Corais', fontsize=16)
plt.tight_layout(rect=[0, 0, 0.85, 0.95])
plt.savefig(os.path.join(output_dir, "PCA_Saude_Figura_Composta.png"))
plt.close(fig_comp_health)

# --- Gráficos para a PCA de Interações ---
# A mesma correção deve ser aplicada aqui por consistência.
bubble_vars_interactions = ['SUR_TURF', 'SUR_CCA', 'SUR_MACROALGAE', 'SUR_PALYTHOA', 'SUR_ABIOTIC', 'SUR_CYANO']
plots_interactions = []

for var in bubble_vars_interactions:
    fig, ax = plt.subplots(figsize=(8, 7))
    sizes = scale_marker_sizes(df_scores_interactions[var])
    
    sns.scatterplot(
        data=df_scores_interactions, x='PC1_INTERACAO', y='PC2_INTERACAO',
        hue='REEF', style='HAB', s=sizes, palette=color_map,
        markers=shape_map, alpha=0.7, edgecolor='k', ax=ax,
        legend=False # <-- CORREÇÃO APLICADA AQUI TAMBÉM
    )

    ax.set_title(f'PCA de Interações (Tamanho ~ {var})')
    ax.set_xlabel(f'PC1 ({pca_interactions.explained_variance_ratio_[0]:.1%})')
    ax.set_ylabel(f'PC2 ({pca_interactions.explained_variance_ratio_[1]:.1%})')
    ax.grid(True, linestyle='--', alpha=0.6)
    
    first_legend = ax.legend(title="Recife", handles=legend_elements_color, loc='upper left')
    ax.add_artist(first_legend)
    ax.legend(title="Habitat", handles=legend_elements_shape, loc='lower left')
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"PCA_Interacoes_Bubble_{var}.png"))
    fig.canvas.draw()
    img_data = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
    img_data = img_data.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    plots_interactions.append(img_data)
    plt.close(fig)

# Cria figura composta para PCA de Interações
n_plots_inter = len(bubble_vars_interactions)
n_cols_inter = math.ceil(math.sqrt(n_plots_inter))
n_rows_inter = math.ceil(n_plots_inter / n_cols_inter)
fig_comp_inter, axes_comp_inter = plt.subplots(n_rows_inter, n_cols_inter, figsize=(n_cols_inter * 6, n_rows_inter * 5), sharex=True, sharey=True)
axes_comp_inter = axes_comp_inter.flatten()

for i, var in enumerate(bubble_vars_interactions):
    ax = axes_comp_inter[i]
    sizes = scale_marker_sizes(df_scores_interactions[var])
    sns.scatterplot(
        data=df_scores_interactions, x='PC1_INTERACAO', y='PC2_INTERACAO',
        hue='REEF', style='HAB', s=sizes, palette=color_map,
        markers=shape_map, alpha=0.7, edgecolor='k', ax=ax, legend=False
    )
    ax.set_title(f'Tamanho ~ {var}')
    ax.grid(True, linestyle='--', alpha=0.6)
    if i % n_cols_inter == 0: ax.set_ylabel(f'PC2 ({pca_interactions.explained_variance_ratio_[1]:.1%})')
    if i >= n_plots_inter - n_cols_inter: ax.set_xlabel(f'PC1 ({pca_interactions.explained_variance_ratio_[0]:.1%})')

fig_comp_inter.legend(handles=legend_elements_color + legend_elements_shape, title="Recife / Habitat", loc='upper right')
fig_comp_inter.suptitle('Análise de Componentes Principais das Interações Locais', fontsize=16)
plt.tight_layout(rect=[0, 0, 0.85, 0.95])
plt.savefig(os.path.join(output_dir, "PCA_Interacoes_Figura_Composta.png"))
plt.close(fig_comp_inter)

print("Gráficos salvos com sucesso.")

# ---------------------------
# 6. SALVAR RESULTADOS BIOLÓGICOS
# ---------------------------
# Junta todas as informações biológicas calculadas em um único dataframe
resultados_biologicos = colony_metrics.copy()
resultados_biologicos = pd.merge(resultados_biologicos, df_scores_health[['SITE_COL', 'HEALTH_PC1', 'HEALTH_PC2']], on='SITE_COL', how='left')
resultados_biologicos = pd.merge(resultados_biologicos, meta_info, on='SITE_COL', how='left')
resultados_biologicos = pd.merge(resultados_biologicos, df_scores_interactions[['SITE', 'HAB', 'PC1_INTERACAO', 'PC2_INTERACAO']], on=['SITE', 'HAB'], how='left')

# Bloco MODIFICADO para incluir a nova coluna na ordem desejada
# --- Bloco CORRIGIDO para incluir a nova coluna na ordem desejada ---
final_cols = [
    'SITE_COL', 'SITE', 'HAB', 'REEF', 
    'RGR', 'AR_TOTAL_INICIAL', 'MEAN_AR_TOTAL',
    'MEAN_HEALTH', 'MEAN_BLEACH', 'MEAN_DEAD_TISSUE', # <--- CORREÇÃO: Adicione a coluna aqui
    'HEALTH_PC1', 'HEALTH_PC2', 
    'PC1_INTERACAO', 'PC2_INTERACAO'
]
final_cols_exist = [col for col in final_cols if col in resultados_biologicos.columns]
resultados_biologicos = resultados_biologicos[final_cols_exist]

output_file = os.path.join(output_dir, "resultados_biologicos_por_colonia.csv")
resultados_biologicos.to_csv(output_file, index=False)

print(f"\nResultados biológicos consolidados salvos em: {output_file}")
print("--- Script 2 (Versão Aprimorada) concluído com sucesso! ---")

In [ ]:
##### Calcula RGR, médias de saúde, realiza PCAs de Saúde e Interações
##### Add growth boxplot

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import math # Importado para ajudar a organizar as figuras compostas

# ---------------------------
# 1. CONFIGURAÇÕES E LEITURA DOS DADOS
# ---------------------------
print("--- Iniciando Script 2: Análise Biológica (Versão Aprimorada) ---")

# Caminho para os dados biológicos
base_path = r"C:\Users\rbfra\OneDrive\########PUBLICACOES\############Menezes et al. Mus his distribution and abundance Abrolhos\######22.04.23\DATA"
file_vitality = os.path.join(base_path, "######Vitality and size_new.xlsx")

# Diretório de saída para este script
output_dir = r"C:\Users\rbfra\OneDrive\########PUBLICACOES\############Menezes et al. Mus his distribution and abundance Abrolhos\########NEW RESULTS\output_ANALISE_BIOLOGICA_boxplot"
os.makedirs(output_dir, exist_ok=True)
print(f"Diretório de saída definido para: {output_dir}")

# Leitura do arquivo de vitalidade
try:
    vitality_data = pd.read_excel(file_vitality)
except FileNotFoundError as e:
    print(f"ERRO: Arquivo não encontrado - {e}. Verifique o caminho.")
    exit()

# Padronização de colunas e valores
def standardize_df(df):
    df.columns = df.columns.str.strip().str.upper()
    for col in ['SITE', 'HAB', 'REEF']:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.upper()
    return df

vitality_data = standardize_df(vitality_data)
vitality_data['SITE_COL'] = vitality_data['SITE'] + "_" + vitality_data['COL'].astype(str)

# ---------------------------
# 2. CÁLCULO DAS MÉTRICAS DE DESEMPENHO POR COLÔNIA
# ---------------------------
print("\nCalculando métricas de desempenho por colônia (RGR, médias, etc.)...")

def compute_rgr(group):
    group = group.sort_values('YEAR')
    if len(group['YEAR'].unique()) < 2: return np.nan
    ar_inicial = group.iloc[0]['AR_TOTAL']; ar_final = group.iloc[-1]['AR_TOTAL']
    ano_inicial = group.iloc[0]['YEAR']; ano_final = group.iloc[-1]['YEAR']
    if ar_inicial <= 0 or ar_final <= 0 or ano_final == ano_inicial: return np.nan
    return (np.log(ar_final) - np.log(ar_inicial)) / (ano_final - ano_inicial)

def get_ar_total_2006(group):
    if 2006 in group['YEAR'].values:
        return group[group['YEAR'] == 2006]['AR_TOTAL'].iloc[0]
    return np.nan

# --- Bloco CORRIGIDO para incluir MEAN_DEAD_TISSUE diretamente ---
colony_metrics = vitality_data.groupby('SITE_COL').apply(lambda g: pd.Series({
    'RGR': compute_rgr(g),
    'AR_TOTAL_INICIAL': get_ar_total_2006(g),
    'MEAN_AR_TOTAL': g['AR_TOTAL'].mean(),
    'MEAN_HEALTH': g['HEALTH %'].mean(),
    'MEAN_BLEACH': g['BLEACHING %'].mean(),
    'MEAN_DEAD_TISSUE': g['DEAD %'].mean()
})).reset_index()

# --- MELHORIA: Garantir que os valores não sejam negativos ---
for col in ['MEAN_HEALTH', 'MEAN_BLEACH', 'MEAN_DEAD_TISSUE']:
    if col in colony_metrics.columns:
        colony_metrics[col] = colony_metrics[col].clip(lower=0)
        
print("Métricas calculadas (incluindo tecido morto) e valores negativos zerados com sucesso.")

# ---------------------------
# 3. PCA DA SAÚDE DOS CORAIS
# ---------------------------
print("\nRealizando PCA da Saúde dos Corais...")
health_cols = ["HEALTH %", "BLEACHING %", "DEAD %"]
health_props = vitality_data.groupby('SITE_COL')[health_cols].mean().reset_index()
health_props_scaled = StandardScaler().fit_transform(health_props[health_cols])
pca_health = PCA(n_components=2)
health_scores = pca_health.fit_transform(health_props_scaled)
df_scores_health = health_props[['SITE_COL']].copy()
df_scores_health["HEALTH_PC1"] = health_scores[:, 0]
df_scores_health["HEALTH_PC2"] = health_scores[:, 1]
meta_info = vitality_data[['SITE_COL', 'SITE', 'HAB', 'REEF']].drop_duplicates()
df_scores_health = pd.merge(df_scores_health, meta_info, on='SITE_COL')
df_scores_health = pd.merge(df_scores_health, health_props, on='SITE_COL')
scores_path_health = os.path.join(output_dir, "scores_PCA_Saude.xlsx")
df_scores_health.to_excel(scores_path_health, index=False)
print(f"Scores da PCA de Saúde salvos em: {scores_path_health}")
loadings_df_health = pd.DataFrame(pca_health.components_.T, columns=["PC1", "PC2"], index=health_cols)
loadings_path_health = os.path.join(output_dir, "loadings_PCA_Saude.csv")
loadings_df_health.to_csv(loadings_path_health)
print(f"Loadings da PCA de Saúde salvos em: {loadings_path_health}")

# ---------------------------
# 4. PCA DAS INTERAÇÕES LOCAIS
# ---------------------------
print("\nRealizando PCA das Interações Locais...")
cols_interactions = ["SUR_TURF %", "SUR_CCA %", "SUR_CYANO %", "SUR_DICTYOTA %", "SUR_OTHMACR %",
                     "SUR_PALYTHOA %", "SUR_CORAL %", "SUR_SAND %", "SUR_NON-BIOTIC %"]
interaction_means_agg = vitality_data.groupby(['SITE', 'HAB'])[cols_interactions].mean(numeric_only=True).reset_index()
interaction_vars = {
    'SUR_TURF': interaction_means_agg['SUR_TURF %'],
    'SUR_CCA': interaction_means_agg['SUR_CCA %'],
    'SUR_MACROALGAE': interaction_means_agg['SUR_DICTYOTA %'] + interaction_means_agg['SUR_OTHMACR %'],
    'SUR_CYANO': interaction_means_agg['SUR_CYANO %'],
    'SUR_PALYTHOA': interaction_means_agg['SUR_PALYTHOA %'],
    'SUR_ABIOTIC': interaction_means_agg['SUR_SAND %'] + interaction_means_agg['SUR_NON-BIOTIC %']
}
new_interactions = pd.DataFrame(interaction_vars)
new_interactions.insert(0, 'HAB', interaction_means_agg['HAB'])
new_interactions.insert(0, 'SITE', interaction_means_agg['SITE'])
X_interactions = new_interactions.drop(columns=['SITE', 'HAB']).fillna(0)
X_interactions_scaled = StandardScaler().fit_transform(X_interactions)
pca_interactions = PCA(n_components=2)
interaction_scores = pca_interactions.fit_transform(X_interactions_scaled)
df_scores_interactions = new_interactions[['SITE', 'HAB']].copy()
df_scores_interactions["PC1_INTERACAO"] = interaction_scores[:, 0]
df_scores_interactions["PC2_INTERACAO"] = interaction_scores[:, 1]
reef_info_agg = vitality_data[['SITE', 'HAB', 'REEF']].drop_duplicates()
df_scores_interactions = pd.merge(df_scores_interactions, reef_info_agg, on=['SITE', 'HAB'])
df_scores_interactions = pd.merge(df_scores_interactions, new_interactions, on=['SITE', 'HAB'])
scores_path_interactions = os.path.join(output_dir, "scores_PCA_Interacoes.xlsx")
df_scores_interactions.to_excel(scores_path_interactions, index=False)
print(f"Scores da PCA de Interações salvos em: {scores_path_interactions}")
loadings_df_interactions = pd.DataFrame(pca_interactions.components_.T, columns=["PC1", "PC2"], index=X_interactions.columns)
loadings_path_interactions = os.path.join(output_dir, "loadings_PCA_Interacoes.csv")
loadings_df_interactions.to_csv(loadings_path_interactions)
print(f"Loadings da PCA de Interações salvos em: {loadings_path_interactions}")

# ---------------------------
# 5. GERAÇÃO DOS GRÁFICOS (BUBBLE PLOTS) - VERSÃO CORRIGIDA
# ---------------------------
print("\nGerando gráficos de bubble plot...")
unique_reefs = sorted(vitality_data['REEF'].unique())
cmap = plt.get_cmap('tab10')
color_map = {reef: cmap(i) for i, reef in enumerate(unique_reefs)}
unique_habitats = sorted(vitality_data['HAB'].unique())
habitat_shapes = ['o', 's', '^', 'D', 'v', '<', '>']
shape_map = {hab: habitat_shapes[i % len(habitat_shapes)] for i, hab in enumerate(unique_habitats)}
print(f"Mapeamento de Cores (Recife): {color_map}")
print(f"Mapeamento de Símbolos (Habitat): {shape_map}")

def scale_marker_sizes(values, scale_factor=300, min_size=30):
    vals = np.asarray(values, dtype=float); valid = np.isfinite(vals)
    if valid.sum() == 0: return np.full(vals.shape, min_size)
    vmin, vmax = vals[valid].min(), vals[valid].max()
    if vmax == vmin: return np.full(vals.shape, min_size + scale_factor/2)
    scaled = (vals - vmin) / (vmax - vmin); scaled[~valid] = 0
    return scaled * scale_factor + min_size

bubble_vars_health = health_cols
plots_health = []
legend_elements_color = [plt.Line2D([0], [0], marker='o', color='w', label=reef, markersize=10, markerfacecolor=color_map[reef]) for reef in unique_reefs]
legend_elements_shape = [plt.Line2D([0], [0], marker=shape_map[hab], color='grey', label=hab, linestyle='None', markersize=10) for hab in unique_habitats]
for var in bubble_vars_health:
    fig, ax = plt.subplots(figsize=(8, 7))
    sizes = scale_marker_sizes(df_scores_health[var])
    sns.scatterplot(data=df_scores_health,x='HEALTH_PC1',y='HEALTH_PC2',hue='REEF',style='HAB',s=sizes,palette=color_map,markers=shape_map,alpha=0.7,edgecolor='k',ax=ax,legend=False)
    ax.set_title(f'PCA da Saúde (Tamanho ~ {var})'); ax.set_xlabel(f'PC1 ({pca_health.explained_variance_ratio_[0]:.1%})'); ax.set_ylabel(f'PC2 ({pca_health.explained_variance_ratio_[1]:.1%})'); ax.grid(True, linestyle='--', alpha=0.6)
    first_legend = ax.legend(title="Recife", handles=legend_elements_color, loc='upper left'); ax.add_artist(first_legend); ax.legend(title="Habitat", handles=legend_elements_shape, loc='lower left')
    plt.tight_layout(); plt.savefig(os.path.join(output_dir, f"PCA_Saude_Bubble_{var.replace(' %', '')}.png"))
    fig.canvas.draw(); img_data = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8); img_data = img_data.reshape(fig.canvas.get_width_height()[::-1] + (3,)); plots_health.append(img_data); plt.close(fig)
n_plots_health = len(bubble_vars_health); n_cols_health = math.ceil(math.sqrt(n_plots_health)); n_rows_health = math.ceil(n_plots_health / n_cols_health); fig_comp_health, axes_comp_health = plt.subplots(n_rows_health, n_cols_health, figsize=(n_cols_health * 6, n_rows_health * 5), sharex=True, sharey=True); axes_comp_health = axes_comp_health.flatten()
for i, var in enumerate(bubble_vars_health):
    ax = axes_comp_health[i]; sizes = scale_marker_sizes(df_scores_health[var])
    sns.scatterplot(data=df_scores_health, x='HEALTH_PC1', y='HEALTH_PC2',hue='REEF', style='HAB', s=sizes, palette=color_map,markers=shape_map, alpha=0.7, edgecolor='k', ax=ax, legend=False)
    ax.set_title(f'Tamanho ~ {var}'); ax.grid(True, linestyle='--', alpha=0.6)
    if i % n_cols_health == 0: ax.set_ylabel(f'PC2 ({pca_health.explained_variance_ratio_[1]:.1%})')
    if i >= n_plots_health - n_cols_health: ax.set_xlabel(f'PC1 ({pca_health.explained_variance_ratio_[0]:.1%})')
fig_comp_health.legend(handles=legend_elements_color + legend_elements_shape, title="Recife / Habitat", loc='upper right'); fig_comp_health.suptitle('Análise de Componentes Principais da Saúde dos Corais', fontsize=16); plt.tight_layout(rect=[0, 0, 0.85, 0.95]); plt.savefig(os.path.join(output_dir, "PCA_Saude_Figura_Composta.png")); plt.close(fig_comp_health)
bubble_vars_interactions = ['SUR_TURF', 'SUR_CCA', 'SUR_MACROALGAE', 'SUR_PALYTHOA', 'SUR_ABIOTIC', 'SUR_CYANO']; plots_interactions = []
for var in bubble_vars_interactions:
    fig, ax = plt.subplots(figsize=(8, 7)); sizes = scale_marker_sizes(df_scores_interactions[var])
    sns.scatterplot(data=df_scores_interactions, x='PC1_INTERACAO', y='PC2_INTERACAO',hue='REEF', style='HAB', s=sizes, palette=color_map,markers=shape_map, alpha=0.7, edgecolor='k', ax=ax,legend=False)
    ax.set_title(f'PCA de Interações (Tamanho ~ {var})'); ax.set_xlabel(f'PC1 ({pca_interactions.explained_variance_ratio_[0]:.1%})'); ax.set_ylabel(f'PC2 ({pca_interactions.explained_variance_ratio_[1]:.1%})'); ax.grid(True, linestyle='--', alpha=0.6)
    first_legend = ax.legend(title="Recife", handles=legend_elements_color, loc='upper left'); ax.add_artist(first_legend); ax.legend(title="Habitat", handles=legend_elements_shape, loc='lower left')
    plt.tight_layout(); plt.savefig(os.path.join(output_dir, f"PCA_Interacoes_Bubble_{var}.png")); fig.canvas.draw(); img_data = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8); img_data = img_data.reshape(fig.canvas.get_width_height()[::-1] + (3,)); plots_interactions.append(img_data); plt.close(fig)
n_plots_inter = len(bubble_vars_interactions); n_cols_inter = math.ceil(math.sqrt(n_plots_inter)); n_rows_inter = math.ceil(n_plots_inter / n_cols_inter); fig_comp_inter, axes_comp_inter = plt.subplots(n_rows_inter, n_cols_inter, figsize=(n_cols_inter * 6, n_rows_inter * 5), sharex=True, sharey=True); axes_comp_inter = axes_comp_inter.flatten()
for i, var in enumerate(bubble_vars_interactions):
    ax = axes_comp_inter[i]; sizes = scale_marker_sizes(df_scores_interactions[var])
    sns.scatterplot(data=df_scores_interactions, x='PC1_INTERACAO', y='PC2_INTERACAO',hue='REEF', style='HAB', s=sizes, palette=color_map,markers=shape_map, alpha=0.7, edgecolor='k', ax=ax, legend=False)
    ax.set_title(f'Tamanho ~ {var}'); ax.grid(True, linestyle='--', alpha=0.6)
    if i % n_cols_inter == 0: ax.set_ylabel(f'PC2 ({pca_interactions.explained_variance_ratio_[1]:.1%})')
    if i >= n_plots_inter - n_cols_inter: ax.set_xlabel(f'PC1 ({pca_interactions.explained_variance_ratio_[0]:.1%})')
fig_comp_inter.legend(handles=legend_elements_color + legend_elements_shape, title="Recife / Habitat", loc='upper right'); fig_comp_inter.suptitle('Análise de Componentes Principais das Interações Locais', fontsize=16); plt.tight_layout(rect=[0, 0, 0.85, 0.95]); plt.savefig(os.path.join(output_dir, "PCA_Interacoes_Figura_Composta.png")); plt.close(fig_comp_inter)
print("Gráficos salvos com sucesso.")

# ---------------------------
# 6. SALVAR RESULTADOS BIOLÓGICOS
# ---------------------------
resultados_biologicos = colony_metrics.copy()
resultados_biologicos = pd.merge(resultados_biologicos, df_scores_health[['SITE_COL', 'HEALTH_PC1', 'HEALTH_PC2']], on='SITE_COL', how='left')
resultados_biologicos = pd.merge(resultados_biologicos, meta_info, on='SITE_COL', how='left')
resultados_biologicos = pd.merge(resultados_biologicos, df_scores_interactions[['SITE', 'HAB', 'PC1_INTERACAO', 'PC2_INTERACAO']], on=['SITE', 'HAB'], how='left')
final_cols = ['SITE_COL', 'SITE', 'HAB', 'REEF', 'RGR', 'AR_TOTAL_INICIAL', 'MEAN_AR_TOTAL', 'MEAN_HEALTH', 'MEAN_BLEACH', 'MEAN_DEAD_TISSUE', 'HEALTH_PC1', 'HEALTH_PC2', 'PC1_INTERACAO', 'PC2_INTERACAO']
final_cols_exist = [col for col in final_cols if col in resultados_biologicos.columns]
resultados_biologicos = resultados_biologicos[final_cols_exist]
output_file = os.path.join(output_dir, "resultados_biologicos_por_colonia.csv")
resultados_biologicos.to_csv(output_file, index=False)
print(f"\nResultados biológicos consolidados salvos em: {output_file}")

# ---------------------------
# 7. GERAÇÃO DE BOX PLOTS (RGR)  <--- CÓDIGO NOVO INSERIDO AQUI
# ---------------------------
print("\nGerando box plots para a Taxa de Crescimento Relativa (RGR)...")

# Define o tema/estilo do seaborn para ser consistente com seus outros gráficos
sns.set_theme(style="whitegrid", palette="muted")

# É uma boa prática remover valores nulos de RGR antes de plotar
rgr_data_for_plot = resultados_biologicos.dropna(subset=['RGR'])

# Cria a figura para o box plot
plt.figure(figsize=(12, 7)) # Aumentei um pouco a largura para acomodar a legenda
ax = sns.boxplot(
    data=rgr_data_for_plot,
    x="REEF",
    y="RGR",
    hue="HAB",
    order=sorted(rgr_data_for_plot['REEF'].unique()), # Garante a ordem alfabética dos recifes
    showfliers=False  # Oculta os outliers, como no seu script de exemplo
)

# Adiciona títulos e rótulos
ax.set_title("Taxa de Crescimento Relativa (RGR) por Recife e Habitat", fontsize=16)
ax.set_xlabel("Recife", fontsize=12)
ax.set_ylabel("Taxa de Crescimento Relativa (RGR)", fontsize=12)

# Adiciona uma linha horizontal em y=0 para indicar crescimento zero
ax.axhline(0, color='red', linestyle='--', linewidth=1)

# Posiciona a legenda do lado de fora do gráfico para não obstruir os dados
ax.legend(title="Habitat", bbox_to_anchor=(1.02, 1), loc="upper left")

# Ajusta o layout para garantir que tudo (incluindo a legenda) caiba na imagem
plt.tight_layout()

# Salva a figura no diretório de saída
boxplot_path = os.path.join(output_dir, "boxplot_RGR_por_Reef_e_Hab.png")
plt.savefig(boxplot_path, dpi=300)
plt.close() # Fecha a figura para liberar memória

print(f"Box plot de RGR salvo com sucesso em: {boxplot_path}")


print("\n--- Script 2 (Versão Aprimorada) concluído com sucesso! ---")

In [ ]:
##### Calcula RGR, médias de saúde, realiza PCAs de Saúde e Interações
##### Add growth boxplot
#### New PCAs with loadings

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import math # Importado para ajudar a organizar as figuras compostas

# ---------------------------
# 1. CONFIGURAÇÕES E LEITURA DOS DADOS
# ---------------------------
print("--- Iniciando Script 2: Análise Biológica (Versão Aprimorada) ---")
base_path = r"C:\Users\rbfra\OneDrive\########PUBLICACOES\############Menezes et al. Mus his distribution and abundance Abrolhos\######22.04.23\DATA"
file_vitality = os.path.join(base_path, "######Vitality and size_new.xlsx")
output_dir = r"C:\Users\rbfra\OneDrive\########PUBLICACOES\############Menezes et al. Mus his distribution and abundance Abrolhos\########NEW RESULTS\output_ANALISE_BIOLOGICA_boxplot_PCAnew"
os.makedirs(output_dir, exist_ok=True)
print(f"Diretório de saída definido para: {output_dir}")
try:
    vitality_data = pd.read_excel(file_vitality)
except FileNotFoundError as e:
    print(f"ERRO: Arquivo não encontrado - {e}. Verifique o caminho."); exit()
def standardize_df(df):
    df.columns = df.columns.str.strip().str.upper()
    for col in ['SITE', 'HAB', 'REEF']:
        if col in df.columns: df[col] = df[col].astype(str).str.strip().str.upper()
    return df
vitality_data = standardize_df(vitality_data)
vitality_data['SITE_COL'] = vitality_data['SITE'] + "_" + vitality_data['COL'].astype(str)

# ---------------------------
# 2. CÁLCULO DAS MÉTRICAS DE DESEMPENHO POR COLÔNIA
# ---------------------------
print("\nCalculando métricas de desempenho por colônia (RGR, médias, etc.)...")
def compute_rgr(group):
    group = group.sort_values('YEAR')
    if len(group['YEAR'].unique()) < 2: return np.nan
    ar_inicial = group.iloc[0]['AR_TOTAL']; ar_final = group.iloc[-1]['AR_TOTAL']
    ano_inicial = group.iloc[0]['YEAR']; ano_final = group.iloc[-1]['YEAR']
    if ar_inicial <= 0 or ar_final <= 0 or ano_final == ano_inicial: return np.nan
    return (np.log(ar_final) - np.log(ar_inicial)) / (ano_final - ano_inicial)
def get_ar_total_2006(group):
    if 2006 in group['YEAR'].values: return group[group['YEAR'] == 2006]['AR_TOTAL'].iloc[0]
    return np.nan
colony_metrics = vitality_data.groupby('SITE_COL').apply(lambda g: pd.Series({'RGR': compute_rgr(g),'AR_TOTAL_INICIAL': get_ar_total_2006(g),'MEAN_AR_TOTAL': g['AR_TOTAL'].mean(),'MEAN_HEALTH': g['HEALTH %'].mean(),'MEAN_BLEACH': g['BLEACHING %'].mean(),'MEAN_DEAD_TISSUE': g['DEAD %'].mean()})).reset_index()
for col in ['MEAN_HEALTH', 'MEAN_BLEACH', 'MEAN_DEAD_TISSUE']:
    if col in colony_metrics.columns: colony_metrics[col] = colony_metrics[col].clip(lower=0)
print("Métricas calculadas (incluindo tecido morto) e valores negativos zerados com sucesso.")

# ---------------------------
# 3. PCA DA SAÚDE DOS CORAIS
# ---------------------------
print("\nRealizando PCA da Saúde dos Corais...")
health_cols = ["HEALTH %", "BLEACHING %", "DEAD %"]
health_props = vitality_data.groupby('SITE_COL')[health_cols].mean().reset_index()
health_props_scaled = StandardScaler().fit_transform(health_props[health_cols])
pca_health = PCA(n_components=2)
health_scores = pca_health.fit_transform(health_props_scaled)
df_scores_health = health_props[['SITE_COL']].copy()
df_scores_health["HEALTH_PC1"] = health_scores[:, 0]
df_scores_health["HEALTH_PC2"] = health_scores[:, 1]
meta_info = vitality_data[['SITE_COL', 'SITE', 'HAB', 'REEF']].drop_duplicates()
df_scores_health = pd.merge(df_scores_health, meta_info, on='SITE_COL')
df_scores_health = pd.merge(df_scores_health, health_props, on='SITE_COL')
scores_path_health = os.path.join(output_dir, "scores_PCA_Saude.xlsx")
df_scores_health.to_excel(scores_path_health, index=False)
print(f"Scores da PCA de Saúde salvos em: {scores_path_health}")
loadings_df_health = pd.DataFrame(pca_health.components_.T, columns=["PC1", "PC2"], index=health_cols)
loadings_path_health = os.path.join(output_dir, "loadings_PCA_Saude.csv")
loadings_df_health.to_csv(loadings_path_health)
print(f"Loadings da PCA de Saúde salvos em: {loadings_path_health}")

# ---------------------------
# 4. PCA DAS INTERAÇÕES LOCAIS
# ---------------------------
print("\nRealizando PCA das Interações Locais...")
cols_interactions = ["SUR_TURF %", "SUR_CCA %", "SUR_CYANO %", "SUR_DICTYOTA %", "SUR_OTHMACR %","SUR_PALYTHOA %", "SUR_CORAL %", "SUR_SAND %", "SUR_NON-BIOTIC %"]
interaction_means_agg = vitality_data.groupby(['SITE', 'HAB'])[cols_interactions].mean(numeric_only=True).reset_index()
interaction_vars = {'SUR_TURF': interaction_means_agg['SUR_TURF %'],'SUR_CCA': interaction_means_agg['SUR_CCA %'],'SUR_MACROALGAE': interaction_means_agg['SUR_DICTYOTA %'] + interaction_means_agg['SUR_OTHMACR %'],'SUR_CYANO': interaction_means_agg['SUR_CYANO %'],'SUR_PALYTHOA': interaction_means_agg['SUR_PALYTHOA %'],'SUR_ABIOTIC': interaction_means_agg['SUR_SAND %'] + interaction_means_agg['SUR_NON-BIOTIC %']}
new_interactions = pd.DataFrame(interaction_vars)
new_interactions.insert(0, 'HAB', interaction_means_agg['HAB']); new_interactions.insert(0, 'SITE', interaction_means_agg['SITE'])
X_interactions = new_interactions.drop(columns=['SITE', 'HAB']).fillna(0)
X_interactions_scaled = StandardScaler().fit_transform(X_interactions)
pca_interactions = PCA(n_components=2)
interaction_scores = pca_interactions.fit_transform(X_interactions_scaled)
df_scores_interactions = new_interactions[['SITE', 'HAB']].copy()
df_scores_interactions["PC1_INTERACAO"] = interaction_scores[:, 0]
df_scores_interactions["PC2_INTERACAO"] = interaction_scores[:, 1]
reef_info_agg = vitality_data[['SITE', 'HAB', 'REEF']].drop_duplicates()
df_scores_interactions = pd.merge(df_scores_interactions, reef_info_agg, on=['SITE', 'HAB'])
df_scores_interactions = pd.merge(df_scores_interactions, new_interactions, on=['SITE', 'HAB'])
scores_path_interactions = os.path.join(output_dir, "scores_PCA_Interacoes.xlsx")
df_scores_interactions.to_excel(scores_path_interactions, index=False)
print(f"Scores da PCA de Interações salvos em: {scores_path_interactions}")
loadings_df_interactions = pd.DataFrame(pca_interactions.components_.T, columns=["PC1", "PC2"], index=X_interactions.columns)
loadings_path_interactions = os.path.join(output_dir, "loadings_PCA_Interacoes.csv")
loadings_df_interactions.to_csv(loadings_path_interactions)
print(f"Loadings da PCA de Interações salvos em: {loadings_path_interactions}")

# =============================================================================
# 4.5. FUNÇÃO PARA PLOTAGEM COMPOSTA DA PCA (ADAPTADA DO SCRIPT DE ANÁLISE LOCAL) <--- NOVA FUNÇÃO AQUI
# =============================================================================
def create_composite_pca_figure(pca_results, output_filename):
    if pca_results is None: print(f"Não há resultados de PCA para gerar a figura {output_filename}. Pulando."); return
    df_scores = pca_results['df_scores']; loadings = pca_results['loadings']; explained_variance = pca_results['explained_variance']
    pc1_col, pc2_col = pca_results['pc1_col'], pca_results['pc2_col']; title_suffix = pca_results['title_suffix']
    unique_reefs_plot = sorted(df_scores['REEF'].unique()); cmap_plot = plt.get_cmap('tab10'); color_map_plot = {reef: cmap_plot(i) for i, reef in enumerate(unique_reefs_plot)}
    unique_habitats_plot = sorted(df_scores['HAB'].unique()); habitat_shapes_plot = ['o', 's', '^', 'D', 'v', '<', '>']; shape_map_plot = {hab: habitat_shapes_plot[i % len(habitat_shapes_plot)] for i, hab in enumerate(unique_habitats_plot)}
    fig, axes = plt.subplots(1, 3, figsize=(24, 7), gridspec_kw={'width_ratios': [1.2, 1, 0.8]}); fig.suptitle(f'Resumo da Análise de Componentes Principais - {title_suffix}', fontsize=20, y=1.02)
    ax1 = axes[0]
    sns.scatterplot(data=df_scores,x=pc1_col,y=pc2_col,hue='REEF',style='HAB',palette=color_map_plot,markers=shape_map_plot,s=100,alpha=0.8,edgecolor='k',ax=ax1,legend=False)
    ax1.set_xlabel(f"PC1 ({explained_variance[0] * 100:.1f}%)", fontsize=14)
    ax1.set_ylabel(f"PC2 ({explained_variance[1] * 100:.1f}%)", fontsize=14)
    ax1.set_title("Ordenação dos Pontos", fontsize=16)
    ax1.grid(True, linestyle='--', alpha=0.6); ax1.axhline(0, color='grey', lw=0.5); ax1.axvline(0, color='grey', lw=0.5)
    legend_elements_color = [plt.Line2D([0], [0], marker='o', color='w', label=reef, markersize=10, markerfacecolor=color_map_plot[reef]) for reef in unique_reefs_plot]
    legend_elements_shape = [plt.Line2D([0], [0], marker=shape_map_plot[hab], color='grey', label=hab, linestyle='None', markersize=10) for hab in unique_habitats_plot]
    fig.legend(title="Recife", handles=legend_elements_color, loc='center left', bbox_to_anchor=(0.91, 0.65)); fig.legend(title="Habitat", handles=legend_elements_shape, loc='center left', bbox_to_anchor=(0.91, 0.35))
    ax2 = axes[1]; ax2.axhline(0, color='grey', lw=0.5); ax2.axvline(0, color='grey', lw=0.5)
    for i, var in enumerate(loadings.index):
        ax2.arrow(0, 0, loadings['PC1'][i]*1.5, loadings['PC2'][i]*1.5, head_width=0.05, head_length=0.1, fc='red', ec='red')
        ax2.text(loadings['PC1'][i]*1.7, loadings['PC2'][i]*1.7, var, color='black', ha='center', va='center', fontsize=12)
    ax2.set_xlim(-2, 2); ax2.set_ylim(-2, 2); ax2.set_xlabel("Contribuição para PC1", fontsize=14); ax2.set_ylabel("Contribuição para PC2", fontsize=14); ax2.set_title("Loadings das Variáveis", fontsize=16); ax2.set_aspect('equal', adjustable='box')
    ax3 = axes[2]; components = ['PC1', 'PC2']
    ax3.bar(components, explained_variance, color='skyblue', edgecolor='black'); ax3.set_ylabel("Variância Explicada (%)", fontsize=14); ax3.set_title("Importância dos Componentes", fontsize=16); ax3.set_ylim(0, 100)
    for i, v in enumerate(explained_variance): ax3.text(i, v + 2, f"{v:.1f}%", ha='center', color='black', fontsize=12)
    fig.subplots_adjust(right=0.9); plt.savefig(output_filename, dpi=300, bbox_inches='tight'); plt.close(fig)
    print(f"Figura composta de resumo da PCA salva em: {output_filename}")


# ---------------------------
# 5. GERAÇÃO DOS GRÁFICOS (BUBBLE PLOTS) - VERSÃO CORRIGIDA
# ---------------------------
# ... (código dos bubble plots permanece o mesmo) ...
print("\nGerando gráficos de bubble plot...")
# ... (código omitido para brevidade) ...
print("Gráficos salvos com sucesso.")


# ------------------------------------------------------------------
# 5.5 GERAÇÃO DAS FIGURAS COMPOSTAS DE RESUMO DAS PCAS  <--- NOVA SEÇÃO AQUI
# ------------------------------------------------------------------
print("\nGerando figuras compostas de resumo das PCAs (Ordenação + Loadings + Scree Plot)...")
health_pca_results = {
    'df_scores': df_scores_health, 'loadings': loadings_df_health,
    'explained_variance': pca_health.explained_variance_ratio_ * 100,
    'pc1_col': 'HEALTH_PC1', 'pc2_col': 'HEALTH_PC2', 'title_suffix': 'Saúde dos Corais'
}
create_composite_pca_figure(health_pca_results, os.path.join(output_dir, "figura_composta_PCA_Saude.png"))
interactions_pca_results = {
    'df_scores': df_scores_interactions, 'loadings': loadings_df_interactions,
    'explained_variance': pca_interactions.explained_variance_ratio_ * 100,
    'pc1_col': 'PC1_INTERACAO', 'pc2_col': 'PC2_INTERACAO', 'title_suffix': 'Interações Locais'
}
create_composite_pca_figure(interactions_pca_results, os.path.join(output_dir, "figura_composta_PCA_Interacoes.png"))


# ---------------------------
# 6. SALVAR RESULTADOS BIOLÓGICOS
# ---------------------------
# ... (código da seção 6 permanece o mesmo) ...
resultados_biologicos = colony_metrics.copy(); resultados_biologicos = pd.merge(resultados_biologicos, df_scores_health[['SITE_COL', 'HEALTH_PC1', 'HEALTH_PC2']], on='SITE_COL', how='left'); resultados_biologicos = pd.merge(resultados_biologicos, meta_info, on='SITE_COL', how='left'); resultados_biologicos = pd.merge(resultados_biologicos, df_scores_interactions[['SITE', 'HAB', 'PC1_INTERACAO', 'PC2_INTERACAO']], on=['SITE', 'HAB'], how='left'); final_cols = ['SITE_COL', 'SITE', 'HAB', 'REEF', 'RGR', 'AR_TOTAL_INICIAL', 'MEAN_AR_TOTAL', 'MEAN_HEALTH', 'MEAN_BLEACH', 'MEAN_DEAD_TISSUE', 'HEALTH_PC1', 'HEALTH_PC2', 'PC1_INTERACAO', 'PC2_INTERACAO']; final_cols_exist = [col for col in final_cols if col in resultados_biologicos.columns]; resultados_biologicos = resultados_biologicos[final_cols_exist]; output_file = os.path.join(output_dir, "resultados_biologicos_por_colonia.csv"); resultados_biologicos.to_csv(output_file, index=False); print(f"\nResultados biológicos consolidados salvos em: {output_file}")


# ---------------------------
# 7. GERAÇÃO DE BOX PLOTS (RGR)
# ---------------------------
# ... (código da seção 7 permanece o mesmo) ...
print("\nGerando box plots para a Taxa de Crescimento Relativa (RGR)..."); sns.set_theme(style="whitegrid", palette="muted"); rgr_data_for_plot = resultados_biologicos.dropna(subset=['RGR']); plt.figure(figsize=(12, 7)); ax = sns.boxplot(data=rgr_data_for_plot,x="REEF",y="RGR",hue="HAB",order=sorted(rgr_data_for_plot['REEF'].unique()),showfliers=False); ax.set_title("Taxa de Crescimento Relativa (RGR) por Recife e Habitat", fontsize=16); ax.set_xlabel("Recife", fontsize=12); ax.set_ylabel("Taxa de Crescimento Relativa (RGR)", fontsize=12); ax.axhline(0, color='red', linestyle='--', linewidth=1); ax.legend(title="Habitat", bbox_to_anchor=(1.02, 1), loc="upper left"); plt.tight_layout(); boxplot_path = os.path.join(output_dir, "boxplot_RGR_por_Reef_e_Hab.png"); plt.savefig(boxplot_path, dpi=300); plt.close(); print(f"Box plot de RGR salvo com sucesso em: {boxplot_path}")

print("\n--- Script 2 (Versão Aprimorada) concluído com sucesso! ---")

In [ ]:
### INTEGRAÇÃO FINAL DOS DADOS (NÍVEL COLÔNIA) - VERSÃO 6 ###
# 1. Adiciona a coluna 'ARCH' ao dataframe final.
# 2. Mantém a padronização robusta de colunas para maiúsculas.
# 3. Garante que a nova coluna seja incluída na ordem correta no arquivo de saída.

import os
import pandas as pd
import glob

# ---------------------------
# 1. CONFIGURAÇÕES E CAMINHOS
# ---------------------------
print("--- Iniciando Script de Integração Final (Nível Colônia) - Versão 6 ---")

# --- Caminhos de Entrada ---
sites_csv_path = r"C:\Users\rbfra\OneDrive\########CEBIMAR\####PROJETOS\#####Coral trade offs\sites_list_full.csv"
path_pca_local = r"C:\Users\rbfra\OneDrive\########PUBLICACOES\############Menezes et al. Mus his distribution and abundance Abrolhos\########NEW RESULTS\#####output_local_PCA_CV_30_FINAL"
file_ambiental_scores = os.path.join(path_pca_local, "dados_consolidados_com_scores_das_duas_PCAs.xlsx")
path_biologico = r"C:\Users\rbfra\OneDrive\########PUBLICACOES\############Menezes et al. Mus his distribution and abundance Abrolhos\########NEW RESULTS\#output_ANALISE_BIOLOGICA"
file_resultados_biologicos = os.path.join(path_biologico, "resultados_biologicos_por_colonia.csv")
path_frequencia = r"C:\Users\rbfra\OneDrive\########PUBLICACOES\############Menezes et al. Mus his distribution and abundance Abrolhos\########NEW RESULTS\##LOMB_SCARGLE_POR_ARCO_FINAL"

# --- Diretório de Saída ---
output_dir = r"C:\Users\rbfra\OneDrive\########PUBLICACOES\############Menezes et al. Mus his distribution and abundance Abrolhos\########NEW RESULTS\output_DADOS_FINAIS_PARA_MODELAGEM_cv_30"
os.makedirs(output_dir, exist_ok=True)
print(f"Diretório de saída definido para: {output_dir}")

# ---------------------------
# 2. LEITURA E PADRONIZAÇÃO IMEDIATA
# ---------------------------
print("\nLendo e padronizando arquivos de entrada...")
try:
    df_sites_meta = pd.read_csv(sites_csv_path, sep=';')
    df_sites_meta.columns = [col.strip().upper() for col in df_sites_meta.columns]
    print(">>> Arquivo de metadados dos sítios lido e padronizado.")

    df_ambiental = pd.read_excel(file_ambiental_scores)
    df_ambiental.columns = [col.strip().upper() for col in df_ambiental.columns]
    print(">>> Arquivo de dados ambientais lido e padronizado.")

    df_biologico = pd.read_csv(file_resultados_biologicos)
    df_biologico.columns = [col.strip().upper() for col in df_biologico.columns]
    print(">>> Arquivo de dados biológicos lido e padronizado.")

    frequency_files = glob.glob(os.path.join(path_frequencia, "frequencias_*.csv"))
    if frequency_files:
        df_frequencia = pd.read_csv(frequency_files[0], sep=';', decimal=',')
        df_frequencia.columns = [col.strip().upper() for col in df_frequencia.columns]
        print(f">>> Arquivo de frequência '{os.path.basename(frequency_files[0])}' lido e padronizado.")
    else:
        df_frequencia = None
        print("AVISO: Nenhum arquivo de frequência encontrado.")

except Exception as e:
    print(f"\nERRO CRÍTICO DURANTE A LEITURA DE UM DOS ARQUIVOS DE ENTRADA: {e}")
    exit()

# ---------------------------
# 3. CRIAÇÃO DE CHAVES E INTEGRAÇÃO
# ---------------------------
print("\nCriando chaves de junção e integrando os dados...")

df_ambiental.rename(columns={'SITE_NAME': 'SITE'}, inplace=True, errors='ignore')
df_biologico.rename(columns={'SITE_NAME': 'SITE'}, inplace=True, errors='ignore')
if df_frequencia is not None:
    df_frequencia.rename(columns={'SITE_NAME': 'SITE'}, inplace=True, errors='ignore')
df_sites_meta.rename(columns={'SITE_NAME': 'SITE'}, inplace=True, errors='ignore')

if df_frequencia is not None and 'HAB' not in df_frequencia.columns:
    meta_subset = df_sites_meta[['SITE', 'HAB']].drop_duplicates()
    df_frequencia = pd.merge(df_frequencia, meta_subset, on='SITE', how='left')
    print("Coluna 'HAB' adicionada ao dataframe de frequência.")

for df, name in [(df_ambiental, 'ambiental'), (df_biologico, 'biologico'), (df_frequencia, 'frequencia')]:
    if df is not None and 'SITE' in df.columns and 'HAB' in df.columns:
        df['UNIQUE_ID'] = df['SITE'].astype(str).str.strip() + '_' + df['HAB'].astype(str).str.strip()
        print(f"Chave 'UNIQUE_ID' criada/verificada para o dataframe '{name}'.")

# --- SELEÇÃO DE COLUNAS E MERGE ---
# <--- MUDANÇA 1: Adicionar a coluna 'ARCH' à lista de colunas a manter ---
cols_ambientais_a_manter = [
    'UNIQUE_ID', 'ARCH', 'PC1_MAGNITUDE', 'PC2_MAGNITUDE', 'PC1_VARIABILITY', 'PC2_VARIABILITY',
    'SST_MEAN', 'SST_CV_ALL', 'MEAN_DLI_LOCAL', 'CV_DLI_LOCAL',
    'CHL_MEAN', 'CHL_CV_ALL', 'PROP_DHW_GT4', 'PROP_DHW_GT8', 'DEPTH_M'
]
existing_cols_ambientais = [col for col in cols_ambientais_a_manter if col in df_ambiental.columns]
df_ambiental_subset = df_ambiental[existing_cols_ambientais].drop_duplicates(subset=['UNIQUE_ID'])
df_ambiental_subset = df_ambiental_subset.rename(columns={
    'PC1_MAGNITUDE': 'PC1_ENV_MAG', 'PC2_MAGNITUDE': 'PC2_ENV_MAG',
    'PC1_VARIABILITY': 'PC1_ENV_VAR', 'PC2_VARIABILITY': 'PC2_ENV_VAR',
    'DEPTH_M': 'DEPTH'
})
print(f"\nColunas ambientais selecionadas para merge: {df_ambiental_subset.columns.tolist()}")

# --- Merge 1: Biológico + Ambiental ---
dados_finais = pd.merge(df_biologico, df_ambiental_subset, on='UNIQUE_ID', how='left')
print(f"Shape após merge com dados ambientais: {dados_finais.shape}")

# --- Merge 2: Adicionar dados de Frequência ---
if df_frequencia is not None:
    # A coluna ARCH já foi adicionada pelo merge anterior, então não precisamos pegá-la daqui.
    cols_frequencia_desejadas = [
        'UNIQUE_ID', 'SST_PERIOD_PRIMARY', 'DLI_PERIOD_PRIMARY', 'CHL_PERIOD_PRIMARY',
        'SST_PERIOD_SECONDARY', 'DLI_PERIOD_SECONDARY', 'CHL_PERIOD_SECONDARY'
    ]
    existing_cols_freq = [col for col in cols_frequencia_desejadas if col in df_frequencia.columns]
    df_frequencia_subset = df_frequencia[existing_cols_freq].drop_duplicates(subset=['UNIQUE_ID'])
    dados_finais = pd.merge(dados_finais, df_frequencia_subset, on='UNIQUE_ID', how='left')
    print(f"Shape após merge com dados de frequência: {dados_finais.shape}")

# ---------------------------
# 4. FORMATAÇÃO E SALVAMENTO FINAL
# ---------------------------
if 'MEAN_DEAD_TISSUE' not in dados_finais.columns and 'MEAN_HEALTH' in dados_finais.columns and 'MEAN_BLEACH' in dados_finais.columns:
    dados_finais['MEAN_DEAD_TISSUE'] = 100 - dados_finais['MEAN_HEALTH'] - dados_finais['MEAN_BLEACH']

# <--- MUDANÇA 2: Adicionar 'ARCH' à ordem final das colunas, logo após 'REEF' ---
final_columns_order = [
    # Identificadores da Colônia e Sítio
    'SITE_COL', 'SITE', 'HAB', 'REEF', 'ARCH', 'UNIQUE_ID',
    
    # Variáveis de Resposta Biológica
    'RGR', 'AR_TOTAL_INICIAL', 'MEAN_AR_TOTAL', 'MEAN_HEALTH', 'MEAN_BLEACH', 'MEAN_DEAD_TISSUE',
    
    # Scores das PCAs Biológicas
    'HEALTH_PC1', 'HEALTH_PC2', 'PC1_INTERACAO', 'PC2_INTERACAO',
    
    # Scores das PCAs Ambientais
    'PC1_ENV_MAG', 'PC2_ENV_MAG', 'PC1_ENV_VAR', 'PC2_ENV_VAR',
    
    # Variáveis Ambientais Brutas e de Frequência
    'SST_MEAN', 'SST_CV_ALL', 'MEAN_DLI_LOCAL', 'CV_DLI_LOCAL',
    'CHL_MEAN', 'CHL_CV_ALL', 'PROP_DHW_GT4', 'PROP_DHW_GT8', 'DEPTH',
    'SST_PERIOD_PRIMARY', 'DLI_PERIOD_PRIMARY', 'CHL_PERIOD_PRIMARY',
    'SST_PERIOD_SECONDARY', 'DLI_PERIOD_SECONDARY', 'CHL_PERIOD_SECONDARY'
]
final_columns_exist = [col for col in final_columns_order if col in dados_finais.columns]
dados_finais_ordenados = dados_finais[final_columns_exist]

print("\nVerificação de valores ausentes (NaN) na planilha final:")
print(dados_finais_ordenados.isnull().sum())

# Define um nome de arquivo um pouco diferente para esta versão
output_file = os.path.join(output_dir, "dados_finais_para_modelagem_com_ARCH.csv")
dados_finais_ordenados.to_csv(output_file, index=False, sep=';', decimal=',')

print(f"\n>>> SUCESSO! Planilha final (com ARCH) integrada salva em: {output_file}")

In [ ]:
### INTEGRAÇÃO FINAL DOS DADOS (NÍVEL COLÔNIA) - VERSÃO 7 (COM LAT/LON) ###
# 1. Adiciona as colunas 'LATITUDE' e 'LONGITUDE' ao dataframe final.
# 2. Mantém a adição da coluna 'ARCH'.
# 3. Garante que as novas colunas sejam incluídas na ordem correta no arquivo de saída.

import os
import pandas as pd
import glob

# ---------------------------
# 1. CONFIGURAÇÕES E CAMINHOS
# ---------------------------
print("--- Iniciando Script de Integração Final (Nível Colônia) - Versão 7 (com Lat/Lon) ---")

# --- Caminhos de Entrada ---
sites_csv_path = r"C:\Users\rbfra\OneDrive\########CEBIMAR\####PROJETOS\#####Coral trade offs\sites_list_full.csv"
path_pca_local = r"C:\Users\rbfra\OneDrive\########PUBLICACOES\############Menezes et al. Mus his distribution and abundance Abrolhos\########NEW RESULTS\#####output_local_PCA_CV_all_FINAL"
file_ambiental_scores = os.path.join(path_pca_local, "dados_consolidados_com_scores_das_duas_PCAs.xlsx")
path_biologico = r"C:\Users\rbfra\OneDrive\########PUBLICACOES\############Menezes et al. Mus his distribution and abundance Abrolhos\########NEW RESULTS\#output_ANALISE_BIOLOGICA"
file_resultados_biologicos = os.path.join(path_biologico, "resultados_biologicos_por_colonia.csv")
path_frequencia = r"C:\Users\rbfra\OneDrive\########PUBLICACOES\############Menezes et al. Mus his distribution and abundance Abrolhos\########NEW RESULTS\##LOMB_SCARGLE_POR_ARCO_FINAL"

# --- Diretório de Saída ---
output_dir = r"C:\Users\rbfra\OneDrive\########PUBLICACOES\############Menezes et al. Mus his distribution and abundance Abrolhos\########NEW RESULTS\output_DADOS_FINAIS_PARA_MODELAGEM_cv_all_lat"
os.makedirs(output_dir, exist_ok=True)
print(f"Diretório de saída definido para: {output_dir}")

# ---------------------------
# 2. LEITURA E PADRONIZAÇÃO IMEDIATA
# ---------------------------
print("\nLendo e padronizando arquivos de entrada...")
try:
    df_sites_meta = pd.read_csv(sites_csv_path, sep=';')
    df_sites_meta.columns = [col.strip().upper() for col in df_sites_meta.columns]
    print(">>> Arquivo de metadados dos sítios lido e padronizado.")

    df_ambiental = pd.read_excel(file_ambiental_scores)
    # Substitui vírgulas por pontos nas colunas de coordenadas e converte para numérico
    df_ambiental['Latitude'] = df_ambiental['Latitude'].astype(str).str.replace(',', '.').astype(float)
    df_ambiental['Longitude'] = df_ambiental['Longitude'].astype(str).str.replace(',', '.').astype(float)
    df_ambiental.columns = [col.strip().upper() for col in df_ambiental.columns]
    print(">>> Arquivo de dados ambientais lido e padronizado (Lat/Lon convertidas para numérico).")

    df_biologico = pd.read_csv(file_resultados_biologicos)
    df_biologico.columns = [col.strip().upper() for col in df_biologico.columns]
    print(">>> Arquivo de dados biológicos lido e padronizado.")

    frequency_files = glob.glob(os.path.join(path_frequencia, "frequencias_*.csv"))
    if frequency_files:
        df_frequencia = pd.read_csv(frequency_files[0], sep=';', decimal=',')
        df_frequencia.columns = [col.strip().upper() for col in df_frequencia.columns]
        print(f">>> Arquivo de frequência '{os.path.basename(frequency_files[0])}' lido e padronizado.")
    else:
        df_frequencia = None
        print("AVISO: Nenhum arquivo de frequência encontrado.")

except Exception as e:
    print(f"\nERRO CRÍTICO DURANTE A LEITURA DE UM DOS ARQUIVOS DE ENTRADA: {e}")
    exit()

# ---------------------------
# 3. CRIAÇÃO DE CHAVES E INTEGRAÇÃO
# ---------------------------
print("\nCriando chaves de junção e integrando os dados...")

df_ambiental.rename(columns={'SITE_NAME': 'SITE'}, inplace=True, errors='ignore')
df_biologico.rename(columns={'SITE_NAME': 'SITE'}, inplace=True, errors='ignore')
if df_frequencia is not None:
    df_frequencia.rename(columns={'SITE_NAME': 'SITE'}, inplace=True, errors='ignore')
df_sites_meta.rename(columns={'SITE_NAME': 'SITE'}, inplace=True, errors='ignore')

if df_frequencia is not None and 'HAB' not in df_frequencia.columns:
    meta_subset = df_sites_meta[['SITE', 'HAB']].drop_duplicates()
    df_frequencia = pd.merge(df_frequencia, meta_subset, on='SITE', how='left')
    print("Coluna 'HAB' adicionada ao dataframe de frequência.")

for df, name in [(df_ambiental, 'ambiental'), (df_biologico, 'biologico'), (df_frequencia, 'frequencia')]:
    if df is not None and 'SITE' in df.columns and 'HAB' in df.columns:
        df['UNIQUE_ID'] = df['SITE'].astype(str).str.strip() + '_' + df['HAB'].astype(str).str.strip()
        print(f"Chave 'UNIQUE_ID' criada/verificada para o dataframe '{name}'.")

# --- SELEÇÃO DE COLUNAS E MERGE ---
### ALTERAÇÃO 1: Adicionar 'LATITUDE' e 'LONGITUDE' à lista de colunas a manter.
cols_ambientais_a_manter = [
    'UNIQUE_ID', 'ARCH', 'LATITUDE', 'LONGITUDE', 
    'PC1_MAGNITUDE', 'PC2_MAGNITUDE', 'PC1_VARIABILITY', 'PC2_VARIABILITY',
    'SST_MEAN', 'SST_CV_ALL', 'MEAN_DLI_LOCAL', 'CV_DLI_LOCAL',
    'CHL_MEAN', 'CHL_CV_ALL', 'PROP_DHW_GT4', 'PROP_DHW_GT8', 'DEPTH_M'
]
existing_cols_ambientais = [col for col in cols_ambientais_a_manter if col in df_ambiental.columns]
df_ambiental_subset = df_ambiental[existing_cols_ambientais].drop_duplicates(subset=['UNIQUE_ID'])
df_ambiental_subset = df_ambiental_subset.rename(columns={
    'PC1_MAGNITUDE': 'PC1_ENV_MAG', 'PC2_MAGNITUDE': 'PC2_ENV_MAG',
    'PC1_VARIABILITY': 'PC1_ENV_VAR', 'PC2_VARIABILITY': 'PC2_ENV_VAR',
    'DEPTH_M': 'DEPTH'
})
print(f"\nColunas ambientais selecionadas para merge: {df_ambiental_subset.columns.tolist()}")

# --- Merge 1: Biológico + Ambiental ---
dados_finais = pd.merge(df_biologico, df_ambiental_subset, on='UNIQUE_ID', how='left')
print(f"Shape após merge com dados ambientais: {dados_finais.shape}")

# --- Merge 2: Adicionar dados de Frequência ---
if df_frequencia is not None:
    cols_frequencia_desejadas = [
        'UNIQUE_ID', 'SST_PERIOD_PRIMARY', 'DLI_PERIOD_PRIMARY', 'CHL_PERIOD_PRIMARY',
        'SST_PERIOD_SECONDARY', 'DLI_PERIOD_SECONDARY', 'CHL_PERIOD_SECONDARY'
    ]
    existing_cols_freq = [col for col in cols_frequencia_desejadas if col in df_frequencia.columns]
    df_frequencia_subset = df_frequencia[existing_cols_freq].drop_duplicates(subset=['UNIQUE_ID'])
    dados_finais = pd.merge(dados_finais, df_frequencia_subset, on='UNIQUE_ID', how='left')
    print(f"Shape após merge com dados de frequência: {dados_finais.shape}")

# ---------------------------
# 4. FORMATAÇÃO E SALVAMENTO FINAL
# ---------------------------
if 'MEAN_DEAD_TISSUE' not in dados_finais.columns and 'MEAN_HEALTH' in dados_finais.columns and 'MEAN_BLEACH' in dados_finais.columns:
    dados_finais['MEAN_DEAD_TISSUE'] = 100 - dados_finais['MEAN_HEALTH'] - dados_finais['MEAN_BLEACH']

### ALTERAÇÃO 2: Adicionar 'LATITUDE' e 'LONGITUDE' à ordem final das colunas.
final_columns_order = [
    # Identificadores da Colônia e Sítio
    'SITE_COL', 'SITE', 'HAB', 'REEF', 'ARCH', 'LATITUDE', 'LONGITUDE', 'UNIQUE_ID',
    
    # Variáveis de Resposta Biológica
    'RGR', 'AR_TOTAL_INICIAL', 'MEAN_AR_TOTAL', 'MEAN_HEALTH', 'MEAN_BLEACH', 'MEAN_DEAD_TISSUE',
    
    # Scores das PCAs Biológicas
    'HEALTH_PC1', 'HEALTH_PC2', 'PC1_INTERACAO', 'PC2_INTERACAO',
    
    # Scores das PCAs Ambientais
    'PC1_ENV_MAG', 'PC2_ENV_MAG', 'PC1_ENV_VAR', 'PC2_ENV_VAR',
    
    # Variáveis Ambientais Brutas e de Frequência
    'SST_MEAN', 'SST_CV_ALL', 'MEAN_DLI_LOCAL', 'CV_DLI_LOCAL',
    'CHL_MEAN', 'CHL_CV_ALL', 'PROP_DHW_GT4', 'PROP_DHW_GT8', 'DEPTH',
    'SST_PERIOD_PRIMARY', 'DLI_PERIOD_PRIMARY', 'CHL_PERIOD_PRIMARY',
    'SST_PERIOD_SECONDARY', 'DLI_PERIOD_SECONDARY', 'CHL_PERIOD_SECONDARY'
]
final_columns_exist = [col for col in final_columns_order if col in dados_finais.columns]
dados_finais_ordenados = dados_finais[final_columns_exist]

print("\nVerificação de valores ausentes (NaN) na planilha final:")
print(dados_finais_ordenados.isnull().sum())

# Define um nome de arquivo para esta nova versão
output_file = os.path.join(output_dir, "dados_finais_para_modelagem_com_ARCH_e_Coords.csv")
dados_finais_ordenados.to_csv(output_file, index=False, sep=';', decimal=',')

print(f"\n>>> SUCESSO! Planilha final (com ARCH, Latitude e Longitude) integrada salva em: {output_file}")

In [ ]:
##### Codigos complementares abaixo ########

In [ ]:
####Converte abu data para formato longo - full

import os
import pandas as pd
import numpy as np

# =============================================================================
# 1. Definir caminhos dos arquivos e do diretório de saída
# =============================================================================

# Arquivo de cobertura (abundância) – BENTHOS TEMPORAL CLEAN
coverage_file = r"C:\Users\rbfra\OneDrive\########PUBLICACOES\############Menezes et al. Mus his distribution and abundance Abrolhos\######22.04.23\DATA\#################BENTHOS TEMPORAL CLEAN.xlsx"

# Arquivo ambiental – utilizaremos apenas PC1_AMBIENTAL e PC2_AMBIENTAL
env_file = r"C:\Users\rbfra\OneDrive\########PUBLICACOES\############Menezes et al. Mus his distribution and abundance Abrolhos\######22.04.23\DATA\dados_nivel_colonia_integrados32.csv"

# Arquivo de quadrats – usaremos DEPTH, LAT e LONG
quadrat_file = r"C:\Users\rbfra\OneDrive\########PUBLICACOES\############Menezes et al. Mus his distribution and abundance Abrolhos\######22.04.23\DATA\Compiled_quadrats_benthic_MRT_versao2.0.xls"

# Diretório de saída
output_dir = r"C:\Users\rbfra\OneDrive\########PUBLICACOES\############Menezes et al. Mus his distribution and abundance Abrolhos\######22.04.23\DATA\output_data"
os.makedirs(output_dir, exist_ok=True)

# =============================================================================
# 2. Leitura e preparação dos dados de cobertura (abundância)
# =============================================================================

# Lê a planilha 'BRUTO' do arquivo de cobertura
df_bruto = pd.read_excel(coverage_file, sheet_name='BRUTO')

# Padroniza os nomes das colunas: remove espaços, converte para maiúsculo e remove caracteres especiais
df_bruto.columns = (
    df_bruto.columns
    .str.strip()
    .str.upper()
    .str.replace(" ", "_")
    .str.replace("[^A-Z0-9_]", "", regex=True)
)

print("Dados de cobertura (brutos) padronizados:")
print(df_bruto.head())

# =============================================================================
# 2.1. Conversão dos dados de cobertura para formato longo (tidy)
# =============================================================================

# Supondo que as colunas metadados sejam: REEF, SITE, HAB, YEAR
metadata_cols = ["REEF", "SITE", "HAB", "YEAR"]

# Padroniza os nomes das colunas: remove espaços, converte para maiúsculo e remove caracteres especiais
df_bruto.columns = (
    df_bruto.columns
    .str.strip()
    .str.upper()
    .str.replace(" ", "_")
    .str.replace("[^A-Z0-9_]", "", regex=True)
)

# Agora que os nomes estão padronizados, definimos os alvos de interesse
target_cols = [
    "MACROALGAE",
    "TURF",
    "CCA",
    "CYANO",
    "MUSSISMILIA_HISPIDA"  # <- Corrigido com o nome em caixa alta padronizada
]


# Se a coluna de Mussismilia não estiver com o nome esperado, tenta renomeá-la
if "MUSSIMILIA_HISPIDA" not in df_bruto.columns:
    if "M_BRAS_M_ALCIC" in df_bruto.columns or "M_BRAS__M_ALCIC" in df_bruto.columns:
        df_bruto = df_bruto.rename(columns={"M_BRAS_M_ALCIC": "MUSSIMILIA_HISPIDA"})
        if "MUSSIMILIA_HISPIDA" not in target_cols:
            target_cols.append("MUSSIMILIA_HISPIDA")
        print("Renomeado 'M_BRAS_M_ALCIC' para 'MUSSIMILIA_HISPIDA'.")

print("Colunas disponíveis no df_bruto:")
print(df_bruto.columns.tolist())
print("Grupos de interesse disponíveis:",
      [col for col in target_cols if col in df_bruto.columns])

# Converte as colunas de interesse para numéricas (se necessário)
for col in target_cols:
    if col in df_bruto.columns:
        df_bruto[col] = pd.to_numeric(df_bruto[col], errors="coerce")

# Converte o dataframe para formato longo utilizando melt
df_long = pd.melt(df_bruto,
                  id_vars=metadata_cols,
                  value_vars=[col for col in target_cols if col in df_bruto.columns],
                  var_name="ORGANISMO",
                  value_name="COBERTURA")

# Ordena os dados por REEF, SITE, HAB, YEAR e ORGANISMO
df_long = df_long.sort_values(by=["REEF", "SITE", "HAB", "YEAR", "ORGANISMO"]).reset_index(drop=True)
print("\nDados de cobertura em formato longo:")
print(df_long.head())

# =============================================================================
# 3. Leitura e preparação dos dados ambientais (dados_nivel_colonia_integradosALL.csv)
# =============================================================================

env_df = pd.read_csv(env_file)

# Padroniza os nomes das colunas
env_df.columns = (
    env_df.columns
    .str.strip()
    .str.upper()
    .str.replace(" ", "_")
    .str.replace("[^A-Z0-9_]", "", regex=True)
)
print("\nDados ambientais (primeiras linhas):")
print(env_df.head())

# Seleciona apenas as colunas de interesse: SITE, PC1_AMBIENTAL, PC2_AMBIENTAL
required_env_cols = ["SITE", "PC1_AMBIENTAL", "PC2_AMBIENTAL"]
missing_env = [col for col in required_env_cols if col not in env_df.columns]
if missing_env:
    raise ValueError(f"As seguintes colunas ambientais não foram encontradas: {missing_env}")

# Se houver múltiplos registros por SITE, mantém apenas a primeira (ou pode-se computar a média, se desejado)
env_df = env_df[required_env_cols].drop_duplicates(subset=["SITE"])
print("\nDados ambientais filtrados (SITE, PC1_AMBIENTAL e PC2_AMBIENTAL):")
print(env_df.head())

# =============================================================================
# 4. Leitura e preparação dos dados de quadrats (Compiled_quadrats_benthic_MRT_versao2.0.xls)
# =============================================================================

quadrat_df = pd.read_excel(quadrat_file)

# Padroniza os nomes das colunas
quadrat_df.columns = (
    quadrat_df.columns
    .str.strip()
    .str.upper()
    .str.replace(" ", "_")
    .str.replace("[^A-Z0-9_]", "", regex=True)
)
print("\nDados de quadrats (primeiras linhas):")
print(quadrat_df.head())

# Seleciona apenas as colunas de interesse: SITE, DEPTH, LAT, LONG
required_quadrat_cols = ["SITE", "DEPTH", "LAT", "LONG"]
missing_quadrat = [col for col in required_quadrat_cols if col not in quadrat_df.columns]
if missing_quadrat:
    raise ValueError(f"As seguintes colunas essenciais não foram encontradas nos dados de quadrats: {missing_quadrat}")

# Se houver múltiplas entradas por SITE, mantém apenas a primeira
quadrat_df = quadrat_df[required_quadrat_cols].drop_duplicates(subset=["SITE"])
print("\nDados de quadrats filtrados (SITE, DEPTH, LAT, LONG):")
print(quadrat_df.head())

# =============================================================================
# 5. Verificação dos nomes de SITE entre os conjuntos
# =============================================================================

sites_coverage = set(df_long["SITE"].unique())
sites_env = set(env_df["SITE"].unique())
sites_quadrat = set(quadrat_df["SITE"].unique())

if sites_coverage != sites_env:
    missing_in_env = sites_coverage - sites_env
    missing_in_cov = sites_env - sites_coverage
    print("Aviso: Diferenças nos nomes dos SITE entre dados de cobertura e ambientais:")
    if missing_in_env:
        print("Sites presentes na cobertura mas ausentes nos dados ambientais:", missing_in_env)
    if missing_in_cov:
        print("Sites presentes nos dados ambientais mas ausentes na cobertura:", missing_in_cov)

if sites_coverage != sites_quadrat:
    missing_in_quadrat = sites_coverage - sites_quadrat
    missing_in_cov2 = sites_quadrat - sites_coverage
    print("Aviso: Diferenças nos nomes dos SITE entre dados de cobertura e de quadrats:")
    if missing_in_quadrat:
        print("Sites presentes na cobertura mas ausentes nos dados de quadrats:", missing_in_quadrat)
    if missing_in_cov2:
        print("Sites presentes nos dados de quadrats mas ausentes na cobertura:", missing_in_cov2)

# =============================================================================
# 6. Integração dos Dados
# =============================================================================

# Merge 1: Integra os dados de cobertura (df_long) com os dados ambientais (env_df) via SITE.
df_integrated = pd.merge(df_long, env_df, on="SITE", how="left")

# Merge 2: Integra o resultado com os dados de quadrats (quadrat_df) via SITE.
df_integrated = pd.merge(df_integrated, quadrat_df, on="SITE", how="left")

# -----------------------------------------------------------------------------
# (Opcional) Criação de um identificador único para cada amostra (fotoquadrat)
# Isso é útil se você deseja tratar individualmente as amostras (por exemplo, incluir QUADRAT_ID 
# como efeito aleatório nos modelos GLMM)
# -----------------------------------------------------------------------------
df_integrated["QUADRAT_ID"] = (
    df_integrated["SITE"].astype(str) + "_" +
    df_integrated["HAB"].astype(str) + "_" +
    df_integrated.groupby(["SITE", "HAB"]).cumcount().astype(str)
)

print("\nDados integrados (primeiras linhas):")
print(df_integrated.head())

# Exibe um resumo de valores ausentes para as variáveis integradas importantes
missing_summary = df_integrated.isnull().sum()
print("\nResumo de valores ausentes nos dados integrados:")
print(missing_summary)

# =============================================================================
# 7. Salvando o Arquivo Integrado
# =============================================================================

integrated_output_file = os.path.join(output_dir, "dados_integrados_long_format_CV32.csv")
df_integrated.to_csv(integrated_output_file, index=False, encoding="utf-8-sig")
print(f"\nDados integrados salvos em: {integrated_output_file}")



In [ ]:
# integracao_dados_abundancia_v3_final.py
# Script para converter dados de cobertura para formato longo e integrar com 
# os novos preditores ambientais e de PCA a partir do arquivo consolidado.
# VERSÃO FINAL: Corrige a lógica de agregação e junção usando uma chave composta (SITE + HAB).

import os
import pandas as pd
import numpy as np

# =============================================================================
# 1. Definir caminhos dos arquivos e do diretório de saída
# =============================================================================

base_path_input = r"C:\Users\rbfra\OneDrive\########PUBLICACOES\############Menezes et al. Mus his distribution and abundance Abrolhos"
coverage_file = os.path.join(base_path_input, "######22.04.23", "DATA", "#################BENTHOS TEMPORAL CLEAN_v2.xlsx")
predictors_file = os.path.join(base_path_input, "########NEW RESULTS", "#####output_local_PCA_CV_all_FINAL", "dados_consolidados_com_scores_das_duas_PCAs.xlsx")
output_dir = os.path.join(base_path_input, "########NEW RESULTS", "#####output_local_PCA_CV_all_FINAL")
os.makedirs(output_dir, exist_ok=True)

print(f"Arquivo de cobertura: {coverage_file}")
print(f"Arquivo de preditores: {predictors_file}")
print(f"Diretório de saída: {output_dir}")

# =============================================================================
# 2. Leitura e preparação dos dados de cobertura (abundância)
# =============================================================================

df_bruto = pd.read_excel(coverage_file, sheet_name='BRUTO')
df_bruto.columns = (
    df_bruto.columns
    .str.strip()
    .str.upper()
    .str.replace(" ", "_")
    .str.replace(r'[^A-Z0-9_]', "", regex=True)
)

# --- MUDANÇA CRÍTICA: Padronizar o tipo de dados das chaves antes de qualquer coisa ---
for col in ["REEF", "SITE", "HAB"]:
    if col in df_bruto.columns:
        df_bruto[col] = df_bruto[col].astype(str).str.strip().str.upper()

print("\nDados de cobertura (brutos) padronizados:")
print(df_bruto.head(2))

# =============================================================================
# 2.1. Conversão dos dados de cobertura para formato longo (tidy)
# =============================================================================

metadata_cols = ["REEF", "SITE", "HAB", "YEAR"]
target_organisms = ["MACROALGAE", "TURF", "CCA", "CYANO", "MUSSISMILIA_HISPIDA"]

available_organisms = [org for org in target_organisms if org in df_bruto.columns]
print(f"\nOrganismos encontrados para conversão: {available_organisms}")

for col in available_organisms:
    df_bruto[col] = pd.to_numeric(df_bruto[col], errors="coerce")

df_long_coverage = pd.melt(
    df_bruto,
    id_vars=metadata_cols,
    value_vars=available_organisms,
    var_name="ORGANISMO",
    value_name="COBERTURA"
)
df_long_coverage.dropna(subset=['COBERTURA'], inplace=True)
print("\nDados de cobertura em formato longo:")
print(df_long_coverage.head(2))

# =============================================================================
# 3. Leitura e preparação dos dados de preditores
# =============================================================================

df_predictors = pd.read_excel(predictors_file)
df_predictors.columns = (
    df_predictors.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(r'[^a-z0-9_]', "", regex=True)
)

if 'site_name' in df_predictors.columns:
    df_predictors.rename(columns={'site_name': 'site'}, inplace=True)
else:
    raise ValueError("A coluna 'Site_name' não foi encontrada no arquivo de preditores.")

# --- MUDANÇA CRÍTICA: Padronizar as chaves de junção no dataframe de preditores ---
df_predictors['site'] = df_predictors['site'].astype(str).str.strip().str.upper()
df_predictors['hab'] = df_predictors['hab'].astype(str).str.strip().str.upper()


# --- MUDANÇA CRÍTICA: Manter a granularidade por SITE e HAB ---
# Remove duplicatas baseando-se na combinação única de site e hab.
key_cols = ['site', 'hab']
df_predictors = df_predictors.drop_duplicates(subset=key_cols)
print("\nDados de preditores padronizados (granularidade por SITE e HAB):")
print(df_predictors[key_cols + ['pc1_variability']].head())


# =============================================================================
# 4. Integração dos Dados usando CHAVE COMPOSTA
# =============================================================================

# Une os dados usando a chave composta [SITE, HAB]
print("\nRealizando junção (merge) com a chave composta ['SITE', 'HAB']...")
df_integrated = pd.merge(
    df_long_coverage,
    df_predictors,
    # --- MUDANÇA CRÍTICA: Usar listas para a chave de junção composta ---
    left_on=["SITE", "HAB"],
    right_on=["site", "hab"],
    how="left" # 'left' garante que todas as amostras de cobertura sejam mantidas
)

# Remove as colunas de chave redundantes do dataframe da direita
if 'site' in df_integrated.columns:
    df_integrated.drop(columns=['site', 'hab'], inplace=True, errors='ignore')

# Padroniza os nomes finais das colunas para maiúsculas
df_integrated.columns = df_integrated.columns.str.upper()

# =============================================================================
# 5. Verificação e Conversão Final de Tipos
# =============================================================================

print("\nVerificando o resultado da junção...")
# Checa se ainda existem lacunas em uma coluna chave de preditores. Idealmente, o número deve ser 0.
missing_after_merge = df_integrated['PC1_VARIABILITY'].isnull().sum()
if missing_after_merge > 0:
    print(f"ATENÇÃO: {missing_after_merge} linhas não encontraram correspondência na junção.")
    # Para depurar, podemos ver quais combinações falharam
    failed_combinations = df_integrated[df_integrated['PC1_VARIABILITY'].isnull()][['SITE', 'HAB']].drop_duplicates()
    print("Combinações que falharam:")
    print(failed_combinations)
else:
    print("Sucesso! Todas as linhas de cobertura encontraram seus preditores correspondentes.")

# Converte colunas com vírgulas para numéricas
for col in df_integrated.columns:
    if df_integrated[col].dtype == 'object':
        try:
            if df_integrated[col].str.contains(',', na=False).any():
                df_integrated[col] = pd.to_numeric(
                    df_integrated[col].str.replace(',', '.', regex=False),
                    errors='coerce'
                )
        except AttributeError:
            continue

print("\nDados integrados (primeiras linhas após conversão):")
print(df_integrated.head())

# =============================================================================
# 6. Salvando o Arquivo Integrado
# =============================================================================

integrated_output_file = os.path.join(output_dir, "dados_abundancia_integrados_long_format.csv")
df_integrated.to_csv(integrated_output_file, index=False, sep=";", decimal=",", encoding="utf-8-sig")

print(f"\nProcesso concluído!")
print(f"Dados integrados para modelagem salvos em: {integrated_output_file}")

In [ ]:
####Verifica datas faltantes no databse

import os
import glob
import re
import pandas as pd
from datetime import datetime

# ============================================
# CONFIGURAÇÕES
# ============================================
modis_dir = r'D:\remote sensing\MODIS_DATA_FULL'
output_dir = r'C:\Users\rbfra\OneDrive\########CEBIMAR\####PROJETOS\#####Coral trade offs\remote sensing\output_remote_sensing\sst_kdpar_series_maps_Abrolhos_new'

# Padrões de busca para os arquivos
patterns = {
    'SST': 'AQUA_MODIS.*.L3m.DAY.SST.sst.4km.nc',
    'KD490': 'AQUA_MODIS.*.L3m.DAY.KD.Kd_490.4km.nc',
    'PAR': 'AQUA_MODIS.*.L3m.DAY.PAR.par.4km.nc',
    'CHL': 'AQUA_MODIS.*.L3m.DAY.CHL.chlor_a.4km.nc',
    'POC': 'AQUA_MODIS.*.L3m.DAY.POC.poc.4km.nc'
}

# Cria a pasta de saída se não existir
os.makedirs(output_dir, exist_ok=True)

# ============================================
# FUNÇÕES AUXILIARES
# ============================================

def parse_date_from_filename(filename):
    """Extrai a data de um nome de arquivo no formato YYYYMMDD."""
    basename = os.path.basename(filename)
    match = re.search(r'(\d{8})', basename)
    if match:
        try:
            return datetime.strptime(match.group(1), '%Y%m%d').date()
        except ValueError:
            return None
    return None

def get_dates_for_product(product_pattern):
    """Retorna todas as datas disponíveis para um produto específico"""
    files = sorted(glob.glob(os.path.join(modis_dir, product_pattern)))
    dates = set()
    for f in files:
        date = parse_date_from_filename(f)
        if date:
            dates.add(date)
    return sorted(dates)

# ============================================
# PROCESSAMENTO PRINCIPAL
# ============================================

# 1. Obter todas as datas disponíveis para cada produto
print("Analisando arquivos MODIS...")
product_dates = {}
for product, pattern in patterns.items():
    product_dates[product] = get_dates_for_product(pattern)
    print(f"{product}: {len(product_dates[product])} arquivos encontrados")

# 2. Encontrar todas as datas únicas em todos os produtos
all_dates = set()
for dates in product_dates.values():
    all_dates.update(dates)
all_dates = sorted(all_dates)

# 3. Criar um DataFrame para análise
df = pd.DataFrame(index=all_dates, columns=patterns.keys())

for product, dates in product_dates.items():
    df[product] = df.index.isin(dates)

# 4. Identificar datas faltantes para cada produto
missing_data = {}
for product in patterns.keys():
    missing = df[~df[product]].index
    missing_data[product] = missing
    print(f"\nDatas faltantes para {product} ({len(missing)}):")
    print(missing[:10])  # Mostra apenas as primeiras 10 para não poluir a saída
    if len(missing) > 10:
        print(f"...({len(missing)-10} datas adicionais)")

# 5. Gerar relatório completo
report_path = os.path.join(output_dir, 'modis_data_coverage_report_HD8TB.csv')
df.to_csv(report_path)
print(f"\nRelatório completo salvo em: {report_path}")

# 6. Gerar resumo estatístico
summary = pd.DataFrame({
    'Total de dias no período': len(df),
    'Dias com dados': df.sum(),
    'Dias faltantes': (~df).sum(),
    'Percentual de cobertura': (df.sum() / len(df) * 100).round(1)
})

summary_path = os.path.join(output_dir, 'modis_data_coverage_summary.csv')
summary.to_csv(summary_path)
print(f"Resumo estatístico salvo em: {summary_path}")

# 7. Gerar lista de datas faltantes por produto para download
for product, dates in missing_data.items():
    if len(dates) > 0:
        missing_dates_path = os.path.join(output_dir, f'missing_dates_{product}.txt')
        with open(missing_dates_path, 'w') as f:
            for date in dates:
                f.write(f"{date.strftime('%Y%m%d')}\n")
        print(f"Lista de datas faltantes para {product} salva em: {missing_dates_path}")

print("\nAnálise concluída!")